# Math-solver benchmark V2 — generalisation stress suite

This is Version 2 of the original SymPy / SmartSolver / StepSolver Colab benchmark. It preserves the **same 72 core case IDs and reference expectations**, then adds **69 tougher stress cases** (141 total) to check whether fixes generalise beyond C10, K03 and G04. The additions target domain restrictions, identities and contradictions, complex roots, transcendental numerical roots, repeated-pole partial fractions, integration rule families, multivariable derivatives, ODEs, inequalities, matrices, and expression formatting.

V2 also adds direct StepSolver probes for nested-power printer parentheses, native `ln(...)` support, unknown function names, and division-by-zero handling. These diagnostics are reported separately because a correct final mathematical answer alone cannot prove the display/parser behavior is safe.

## Run in Google Colab

1. Open **Runtime → Run all**. The configuration cell lets you pin candidate refs or benchmark current `HEAD`.
2. The notebook clones candidates into isolated environments, validates references before candidate execution, runs the scored benchmark, and generates reports from raw outputs.
3. Download `math_solver_benchmark_v2_results.zip` from the final cell. Start with `results/v2_stress_analysis.md`, `results/summary.md`, and `results/failure_analysis.md`.

## Interpretation safeguards

- The core 72 cases are retained for longitudinal comparison; the 69 stress cases have new IDs and are marked `suite: v2_stress` in `data/cases.json`.
- The main overall score is **not directly comparable** to a V1 score because the case mix is larger and tougher. Compare core-case results separately, then examine stress-case coverage, strict correctness, derivations, domain handling, and failures.
- V2 does not rewrite `ln(...)` to `log(...)` inside the StepSolver adapter. This intentionally measures native parser behavior rather than hiding it with adapter normalisation.
- References are validated before any solver run: the original independent validation plus an operation-specific checker for all 69 additions. Fix any reference-validation failure before interpreting the solver results.
- The direct display checks can be `pass`, `fail`, or `inconclusive`. Inconclusive is **not** a pass; inspect the captured formatted step strings.
- Timings are Colab-machine-dependent. The default five warm repeats and two runs give better reproducibility but take longer than V1. Reduce `REPEATS` or set `RUN_TWICE = False` if runtime limits require it.


In [1]:
#@title 1. Configuration { display-mode: "form" }
# ---- Engine versions ------------------------------------------------------------------------------------------
# Original (baseline) run:  SYMPY_SPEC="sympy==1.14.0"
#   SmartSolver REF f0d46a1089366f8e14f73de9ba811d8e4c1471b0   StepSolver REF bd8ea4ffdbe49277d649017cb6a31bf05b376bd8
SYMPY_SPEC = "sympy"                      #@param {type:"string"}   # "sympy" = latest release, or e.g. "sympy==1.14.0"
PIN_SYMPY_IN_CANDIDATES = True            #@param {type:"boolean"}  # give SmartSolver/StepSolver the same SymPy as the standalone engine (falls back to their own resolution if they conflict)
SMARTSOLVER_URL = "https://github.com/mngadilinda/SmartSolver"      #@param {type:"string"}
SMARTSOLVER_REF = "HEAD"                  #@param {type:"string"}   # branch, tag or commit; HEAD = latest default branch
STEPSOLVER_URL = "https://github.com/Jayfraggs/stepsolver.git"      #@param {type:"string"}
STEPSOLVER_REF = "fix/printer-validation-sympy-fallbacks"                   #@param {type:"string"}

# ---- Judge (kept fixed so the same answers are judged the same way across engine versions) ---------------------
JUDGE_SYMPY_SPEC = "sympy==1.14.0"        #@param {type:"string"}

# ---- Protocol ---------------------------------------------------------------------------------------------------
REPEATS = 5                               #@param {type:"integer"}  # warm, cache-cleared repeats per answered case
TIMEOUT_S = 20                            #@param {type:"integer"}  # per-case timeout (worker is killed and restarted)
RUN_TWICE = True                          #@param {type:"boolean"}  # second full run -> reproducibility check
RUN_STEPSOLVER_DIAGNOSTICS = True       #@param {type:"boolean"}  # direct printer/parser/undefined probes; separate from scored cases
RUN_OWN_TEST_SUITES = False               #@param {type:"boolean"}  # also run each candidate's own pytest suite (slow; feeds the maintainability score)
COMPARE_WITH_BASELINE = True              #@param {type:"boolean"}
PYTHON_VERSION = "3.12"                   #@param {type:"string"}
BENCH_DIR = "/content/math_solver_benchmark_v2"   #@param {type:"string"}

In [2]:
#@title 2. Tooling (uv + git) { display-mode: "form" }
import base64, json, os, pathlib, platform, re, shutil, subprocess, sys, time

ROOT = pathlib.Path(BENCH_DIR)
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

def sh(cmd, check=True, timeout=None, cwd=None):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError("command failed (%d): %s\n%s" % (r.returncode, cmd, (r.stdout + r.stderr)[-1500:]))
    return r

def run_stream(cmd, cwd=None, check=True):
    """Run a command and stream its output into the notebook."""
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError("command failed (%d): %s" % (p.returncode, cmd))
    return p.returncode

def W(rel, text):
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")

if not shutil.which("uv"):
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv") or "uv"
print("uv:", sh([UV, "--version"]).stdout.strip(), "| git:", sh("git --version").stdout.strip(), "| notebook python:", platform.python_version())

uv: uv 0.13.4 (x86_64-unknown-linux-gnu) | git: git version 2.43.0 | notebook python: 3.13.16


## Benchmark source
The next five cells write the V2 generator, the shared benchmark framework, reference validation, diagnostics, scoring, and reports to disk. The original 72 cases remain unchanged; 69 extra stress cases are added. Nothing imports a candidate solver while references are being generated or validated.


In [3]:
W('tools/make_cases.py', '#!/usr/bin/env python3\n"""Generate data/cases.json (72 unchanged core cases + 69 V2 generalisation stress cases).\n\nThe original task-specification expected results are retained for the core set.\nTypographical omissions in specification display strings are restored with explicit \'*\'\nin machine-readable `expected.expr` fields; verbatim text is preserved in `spec_expected`.\nThe extension uses new IDs and explicit mathematical references to probe whether earlier fixes generalise.\n\nAll comparison rules, tolerances and form checks below are declared HERE, before any\nsolver is executed.  Nothing in this file imports a solver.\n"""\nimport json\nimport pathlib\n\nCATS = {\n    "A": "Arithmetic and expression evaluation",\n    "B": "Simplification, expansion and factorisation",\n    "C": "Linear and quadratic equations",\n    "D": "Simultaneous equations and inequalities",\n    "E": "Powers, roots, logarithms and trigonometry",\n    "F": "Differentiation",\n    "G": "Integration",\n    "H": "Limits and continuity",\n    "I": "Matrices and linear algebra",\n    "J": "Differential equations and advanced calculus",\n    "K": "Partial fractions, complex numbers and numerical solving",\n    "L": "Domain restrictions and failure handling",\n}\n\nTOL_EXACT = {"abs": 1e-9, "rel": 1e-9}\nTOL_NUM = {"abs": 1e-8, "rel": 1e-8}   # numeric-root cases (K03/K04)\n\n# Default derivation milestones (regex alternatives, matched case-insensitively against the\n# concatenated rule+description text of the solver\'s steps).  Heuristic -> flagged manual.\nMS = {\n    "eval": [r"order of operations|evaluat|arithmetic|compute|simplif|multipl|add|subtract|power|root|absolute"],\n    "simplify": [r"combine|collect|like terms|distribut|cancel|simplif|factor|identity|rule"],\n    "expand": [r"expand|distribut|multiply|foil"],\n    "factor": [r"factor|common|difference of squares|product"],\n    "lin": [r"inverse operation|both sides|isolate|subtract|add|divide|multiply|collect|rearrang|move|solve"],\n    "quad": [r"coefficient|discriminant|standard form|quadratic equation",\n             r"factor|quadratic formula|complete the square",\n             r"zero.product|each factor|set each|separate",\n             r"verif|check|substitut|extraneous|reject"],\n    "system": [r"augmented|eliminat|substitut|row|rref|scale",\n               r"back.substitut|solve for|solution"],\n    "ineq": [r"isolate|divide|subtract|critical|sign|test|interval|factor|reverse|flip"],\n    "power": [r"power|exponent|product rule|quotient rule|index|law|identity|rule"],\n    "trig": [r"special angle|unit circle|exact value|identity|evaluate|table|known value"],\n    "diff": [r"rule|derivative|differentiat|chain|product|power|sum"],\n    "int": [r"rule|integral|antiderivative|substitut|power|by parts|standard|integrat"],\n    "defint": [r"antiderivative|fundamental theorem|evaluate.*bound|upper.*lower|F\\(b\\)"],\n    "limit": [r"substitut|indeterminate|l.?h.?pital|factor|cancel|divide|dominant|squeeze|standard limit|direct"],\n    "matrix": [r"matrix|row|determinant|cofactor|adjugate|pivot|scale|elimination|formula"],\n    "ode": [r"separat|characteristic|integrating factor|auxiliary|ansatz|integrate|solve"],\n    "pf": [r"factor|decompos|partial fraction|cover.up|equate|solve for|coefficients"],\n    "num": [r"newton|iterat|bisect|numeric|approximat|root|initial"],\n    "dom": [r"domain|restrict|undefined|denominator|extraneous|reject|check|verify|must"],\n}\n\ncases = []\n\n\ndef add(cid, op, display, spec_expected, problem, expected, compare, *, domain="real",\n        dsens=False, restrictions=None, ms=None, tol=None, notes="", sample=None, adapter_inputs=None):\n    cases.append({\n        "id": cid,\n        "category": cid[0],\n        "category_name": CATS[cid[0]],\n        "operation": op,\n        "display_input": display,\n        "problem": problem,\n        "domain": domain,                       # real | complex\n        "domain_sensitive": dsens,\n        "required_restrictions": restrictions or [],   # restriction(s) the answer must preserve/report\n        "spec_expected": spec_expected,         # verbatim from the task specification\n        "expected": expected,                   # machine-readable reference answer\n        "comparison": compare,                  # declared judging method (see bench/judge.py)\n        "tolerance": tol or TOL_EXACT,\n        "reference_milestones": ms or [],\n        "sample_points": sample,\n        "notes": notes,\n        "adapter_inputs": adapter_inputs or {},   # per-solver syntax overrides (counted as adaptation effort)\n    })\n\n\n# --------------------------------------------------------------------------- A\ndef A(cid, expr, exp, spec, form=None, ms="eval"):\n    add(cid, "evaluate", expr, spec, {"expr": expr}, {"expr": exp},\n        {"method": "value", "form": form}, ms=MS[ms])\n\nA("A01", "2 + 3 * 4", "14", "14")\nA("A02", "(2 + 3) * 4", "20", "20")\nA("A03", "7/12 + 5/18", "31/36", "31/36")\nA("A04", "sqrt(72)", "6*sqrt(2)", "6*sqrt(2), if simplified", form="simplified_radical")\nA("A05", "(2^3)^2", "64", "64")\nA("A06", "0.125 * 80", "10", "10")\nA("A07", "(-3)^2 - 3^2", "0", "0")\nA("A08", "abs(-7) + abs(3-9)", "13", "13")\n\n# --------------------------------------------------------------------------- B\nadd("B01", "simplify", "3*x + 5*x - 2*x", "6*x", {"expr": "3*x + 5*x - 2*x"}, {"expr": "6*x"},\n    {"method": "expr_equiv", "form": "collected"}, ms=MS["simplify"])\nadd("B02", "simplify", "2*(x + 3) - x", "x + 6", {"expr": "2*(x + 3) - x"}, {"expr": "x + 6"},\n    {"method": "expr_equiv", "form": "no_unexpanded_product"}, ms=MS["simplify"])\nadd("B03", "expand", "(x + 2)*(x + 3) — expand", "x^2 + 5*x + 6", {"expr": "(x + 2)*(x + 3)"},\n    {"expr": "x^2 + 5*x + 6"}, {"method": "expr_equiv", "form": "expanded"}, ms=MS["expand"])\nadd("B04", "factor", "x^2 + 5*x + 6 — factor", "(x + 2)*(x + 3)", {"expr": "x^2 + 5*x + 6"},\n    {"expr": "(x + 2)*(x + 3)"}, {"method": "expr_equiv", "form": "factored"}, ms=MS["factor"])\nadd("B05", "factor", "x^2 - 9 — factor", "(x - 3)*(x + 3)", {"expr": "x^2 - 9"},\n    {"expr": "(x - 3)*(x + 3)"}, {"method": "expr_equiv", "form": "factored"}, ms=MS["factor"])\nadd("B06", "simplify", "(x^2 - 1)/(x - 1) — simplify", "x + 1, preserving x != 1",\n    {"expr": "(x^2 - 1)/(x - 1)"}, {"expr": "x + 1"}, {"method": "expr_equiv", "form": "no_fraction"},\n    dsens=True, restrictions=["x != 1"], ms=MS["simplify"] + MS["dom"][:1], sample={"x": ["-3", "-1/2", "2", "7/3"]})\nadd("B07", "simplify", "x/x — simplify", "1, preserving x != 0", {"expr": "x/x"}, {"expr": "1"},\n    {"method": "expr_equiv"}, dsens=True, restrictions=["x != 0"], ms=MS["simplify"] + MS["dom"][:1])\nadd("B08", "simplify", "(2*x^2*y)/(4*x*y^2) — simplify", "x/(2*y), preserving original denominator restrictions",\n    {"expr": "(2*x^2*y)/(4*x*y^2)"}, {"expr": "x/(2*y)"}, {"method": "expr_equiv", "form": "reduced_fraction"},\n    dsens=True, restrictions=["x != 0", "y != 0"], ms=MS["simplify"] + MS["dom"][:1],\n    sample={"x": ["-3", "2", "5/3"], "y": ["-2", "3", "1/2"]})\n\n# --------------------------------------------------------------------------- C\ndef C(cid, eq, sol, spec, **kw):\n    add(cid, "solve", eq, spec, {"equation": eq, "var": "x"}, sol,\n        {"method": "solution_set"}, ms=kw.pop("ms", MS["lin"]), **kw)\n\nC("C01", "2*x + 3 = 11", {"kind": "finite", "values": ["4"]}, "x = 4")\nC("C02", "2*(x + 3) = 14", {"kind": "finite", "values": ["4"]}, "x = 4")\nC("C03", "3*x - 7 = x + 5", {"kind": "finite", "values": ["6"]}, "x = 6")\nC("C04", "x/3 + 2 = 5", {"kind": "finite", "values": ["9"]}, "x = 9")\nC("C05", "x^2 - 5*x + 6 = 0", {"kind": "finite", "values": ["2", "3"]}, "x = 2, 3", ms=MS["quad"])\nC("C06", "x^2 - 4 = 0", {"kind": "finite", "values": ["-2", "2"]}, "x = -2, 2", ms=MS["quad"])\nC("C07", "x^2 + 1 = 0", {"kind": "finite", "values": ["I", "-I"]},\n  "Complex solutions x = i, -i; no real solutions", domain="complex", dsens=True,\n  ms=MS["quad"][:2] + [r"complex|imaginary|no real|negative discriminant|\\bi\\b"],\n  notes="Expected over C. A real-only answer (empty set) is recorded as \'real_domain_only\', not correct.")\nC("C08", "x^2 + 2*x + 1 = 0", {"kind": "finite", "values": ["-1"], "multiplicity": {"-1": 2}},\n  "x = -1, repeated root", ms=MS["quad"],\n  notes="Set {-1}. Multiplicity reporting is recorded as a secondary check, not part of correctness.")\nC("C09", "2*x + 1 = 2*x + 3", {"kind": "empty"}, "No solution", dsens=True,\n  ms=[r"no solution|contradiction|inconsistent|false|never|impossible|cancel"])\nC("C10", "2*x + 1 = 2*x + 1", {"kind": "all"}, "All admissible x", dsens=True,\n  ms=[r"identity|all (real )?(numbers|x)|infinitely many|always true|every|any x|all admissible"])\n\n# --------------------------------------------------------------------------- D\ndef Dsys(cid, eqs, sol, spec, **kw):\n    add(cid, "solve_system", "; ".join(eqs), spec, {"equations": eqs, "vars": ["x", "y"], "method": kw.pop("method", None)},\n        sol, {"method": "system_solution"}, ms=MS["system"], **kw)\n\nDsys("D01", ["x + y = 7", "x - y = 1"], {"kind": "unique", "assignments": {"x": "4", "y": "3"}}, "x = 4, y = 3")\nDsys("D02", ["2*x + y = 8", "x - y = 1"], {"kind": "unique", "assignments": {"x": "3", "y": "2"}}, "x = 3, y = 2")\nDsys("D03", ["x + y = 2", "x + y = 5"], {"kind": "empty"}, "No solution", dsens=True)\nfor cid, ineq, sol, spec in [\n    ("D04", "2*x + 3 < 11", ["-oo", "4"], "x < 4"),\n    ("D05", "-2*x > 6", ["-oo", "-3"], "x < -3"),\n]:\n    add(cid, "solve_inequality", ineq, spec, {"inequality": ineq, "var": "x"},\n        {"kind": "intervals", "intervals": [[sol[0], sol[1], True, True]]},   # (lo, hi, open_lo, open_hi)\n        {"method": "inequality_set"}, ms=MS["ineq"], dsens=(cid == "D05"),\n        notes="D05 requires the inequality direction to flip when dividing by a negative number." if cid == "D05" else "")\nadd("D06", "solve_inequality", "x^2 - 4 > 0", "x < -2 or x > 2", {"inequality": "x^2 - 4 > 0", "var": "x"},\n    {"kind": "intervals", "intervals": [["-oo", "-2", True, True], ["2", "oo", True, True]]},\n    {"method": "inequality_set"}, ms=MS["ineq"], dsens=True)\n\n# --------------------------------------------------------------------------- E\ndef E(cid, expr, exp, spec, **kw):\n    op = kw.pop("op", "simplify")\n    add(cid, op, expr, spec, {"expr": expr}, {"expr": exp}, {"method": kw.pop("method", "expr_equiv"),\n        "form": kw.pop("form", None)}, ms=kw.pop("ms", MS["power"]), **kw)\n\nE("E01", "x^3 * x^4", "x^7", "x^7")\nE("E02", "x^6 / x^2", "x^4", "x^4, preserving x != 0 from the original expression", dsens=True,\n  restrictions=["x != 0"], ms=MS["power"] + MS["dom"][:1], sample={"x": ["-3", "-1/2", "2", "7/3"]})\nE("E03", "sqrt(x^2)", "abs(x)", "abs(x) over the reals", dsens=True, sample={"x": ["-3", "-1/2", "2", "7/3"]},\n  ms=[r"abs|absolute|\\|x\\||modulus|principal|real"], notes="Over the reals. Result must equal |x| at negative x.")\nE("E04", "log(1000, 10)", "3", "3", op="evaluate", method="value", ms=MS["power"] + [r"log|power of 10|10\\^3|exponent"])\nE("E05", "ln(e^x)", "x", "x over the real domain", dsens=True, sample={"x": ["-3", "-1/2", "2", "7/3"]})\nE("E06", "sin(pi/6)", "1/2", "1/2", op="evaluate", method="value", ms=MS["trig"])\nE("E07", "cos(pi/3)", "1/2", "1/2", op="evaluate", method="value", ms=MS["trig"])\nE("E08", "tan(pi/4)", "1", "1", op="evaluate", method="value", ms=MS["trig"])\n\n# --------------------------------------------------------------------------- F\ndef F(cid, expr, exp, spec, **kw):\n    add(cid, "diff", f"Differentiate {expr}", spec, {"expr": expr, "var": "x"}, {"expr": exp},\n        {"method": "expr_equiv", "form": None}, ms=kw.pop("ms", MS["diff"]), **kw)\n\nF("F01", "x^3", "3*x^2", "3*x^2")\nF("F02", "5*x^4 - 2*x + 7", "20*x^3 - 2", "20*x^3 - 2")\nF("F03", "x^2 * sin(x)", "2*x*sin(x) + x^2*cos(x)", "2xsin(x) + x^2*cos(x)", ms=MS["diff"] + [r"product"])\nF("F04", "sin(x^2)", "2*x*cos(x^2)", "2xcos(x^2)", ms=MS["diff"] + [r"chain"])\nF("F05", "ln(x)", "1/x", "1/x for x > 0", dsens=True, sample={"x": ["1/2", "13/10", "27/10", "71/10"]})\nF("F06", "x^x", "x^x*(ln(x) + 1)", "x^x*(ln(x) + 1) for x > 0", dsens=True,\n  sample={"x": ["1/2", "13/10", "27/10", "71/10"]}, ms=MS["diff"] + [r"logarithmic|ln|log"])\n\n# --------------------------------------------------------------------------- G\ndef G(cid, expr, exp, spec, **kw):\n    add(cid, "integrate", f"Integrate {expr}", spec, {"expr": expr, "var": "x"}, {"expr": exp},\n        {"method": "antiderivative", "require_real_on": kw.pop("real_on", None)}, ms=kw.pop("ms", MS["int"]), **kw)\n\nG("G01", "3*x^2", "x^3", "x^3 + C")\nG("G02", "2*x + 5", "x^2 + 5*x", "x^2 + 5*x + C")\nG("G03", "cos(x)", "sin(x)", "sin(x) + C")\nG("G04", "1/x", "ln(abs(x))", "ln(abs(x)) + C, on intervals excluding zero", dsens=True,\n  real_on=["-4", "-1/2", "1/2", "4"], ms=MS["int"] + [r"abs|absolute|\\|x\\|"],\n  notes="Declared rule: F\'(x)=1/x AND F real-valued on both x<0 and x>0 (log(x) alone fails for x<0).")\nG("G05", "x*exp(x^2)", "exp(x^2)/2", "exp(x^2)/2 + C", ms=MS["int"] + [r"substitut|u\\s*="])\nadd("G06", "integrate", "Integrate x^2 from 0 to 3", "9", {"expr": "x^2", "var": "x", "lower": "0", "upper": "3"},\n    {"expr": "9"}, {"method": "value"}, ms=MS["defint"])\n\n# --------------------------------------------------------------------------- H\ndef Hh(cid, expr, pt, exp, spec, disp):\n    add(cid, "limit", disp, spec, {"expr": expr, "var": "x", "point": pt, "direction": "both"},\n        {"expr": exp}, {"method": "value"}, ms=MS["limit"])\n\nHh("H01", "sin(x)/x", "0", "1", "1", "Limit as x approaches 0 of sin(x)/x")\nHh("H02", "(x^2 - 4)/(x - 2)", "2", "4", "4", "Limit as x approaches 2 of (x^2 - 4)/(x - 2)")\nHh("H03", "(3*x^2 + 1)/(x^2 - 5)", "oo", "3", "3", "Limit as x approaches infinity of (3*x^2 + 1)/(x^2 - 5)")\nHh("H04", "(1 - cos(x))/x^2", "0", "1/2", "1/2", "Limit as x approaches 0 of (1 - cos(x))/x^2")\n\n# --------------------------------------------------------------------------- I\nadd("I01", "solve_system", "Solve x + y = 5; x - y = 1 by elimination", "x = 3, y = 2",\n    {"equations": ["x + y = 5", "x - y = 1"], "vars": ["x", "y"], "method": "elimination"},\n    {"kind": "unique", "assignments": {"x": "3", "y": "2"}}, {"method": "system_solution"}, ms=MS["system"])\nM = [[1, 2], [3, 4]]\nadd("I02", "det", "Determinant of [[1,2],[3,4]]", "-2", {"matrix": M}, {"expr": "-2"}, {"method": "value"}, ms=MS["matrix"])\nadd("I03", "inverse", "Inverse of [[1,2],[3,4]]", "[[-2,1],[3/2,-1/2]]", {"matrix": M},\n    {"matrix": [["-2", "1"], ["3/2", "-1/2"]]}, {"method": "matrix"}, ms=MS["matrix"])\nadd("I04", "rref", "RREF of [[1,2,3],[2,4,6]]", "[[1,2,3],[0,0,0]]", {"matrix": [[1, 2, 3], [2, 4, 6]]},\n    {"matrix": [["1", "2", "3"], ["0", "0", "0"]]}, {"method": "matrix"}, ms=MS["matrix"])\n\n# --------------------------------------------------------------------------- J\nadd("J01", "dsolve", "Solve y\' = 3*y", "y = Cexp(3x)", {"ode": "Derivative(y(x), x) - 3*y(x)", "func": "y", "var": "x", "order": 1},\n    {"family": "C*exp(3*x)", "n_constants": 1}, {"method": "ode_family"}, ms=MS["ode"],\n    adapter_inputs={"stepsolver": "dsolve(diff(y(x), x) = 3*y(x), y(x))"})\nadd("J02", "dsolve", "Solve y\'\' + y = 0", "y = C1cos(x) + C2sin(x)", {"ode": "Derivative(y(x), (x, 2)) + y(x)", "func": "y", "var": "x", "order": 2},\n    {"family": "C1*cos(x) + C2*sin(x)", "n_constants": 2}, {"method": "ode_family"}, ms=MS["ode"],\n    adapter_inputs={"stepsolver": "dsolve(diff(y(x), x, 2) + y(x) = 0, y(x))"})\nadd("J03", "partial_derivative", "Differentiate exp(x*y) with respect to x, treating y as constant", "yexp(xy)",\n    {"expr": "exp(x*y)", "var": "x", "vars": ["x", "y"]}, {"expr": "y*exp(x*y)"},\n    {"method": "expr_equiv"}, ms=MS["diff"] + [r"constant|chain|partial"], sample={"x": ["-1", "1/2", "2"], "y": ["-3/2", "1", "2"]})\nadd("J04", "gradient", "Gradient of f(x,y) = x^2 + x*y + y^2", "(2x + y, x + 2y)",\n    {"expr": "x^2 + x*y + y^2", "vars": ["x", "y"]}, {"vector": ["2*x + y", "x + 2*y"]},\n    {"method": "gradient"}, ms=MS["diff"] + [r"partial|gradient"], sample={"x": ["-1", "1/2", "2"], "y": ["-3/2", "1", "2"]})\n\n# --------------------------------------------------------------------------- K\nadd("K01", "partial_fractions", "Partial fractions of 1/(x^2 - 1)", "1/(2*(x - 1)) - 1/(2*(x + 1))",\n    {"expr": "1/(x^2 - 1)", "var": "x"}, {"expr": "1/(2*(x - 1)) - 1/(2*(x + 1))"},\n    {"method": "expr_equiv", "form": "partial_fractions"}, ms=MS["pf"], dsens=True,\n    sample={"x": ["-3", "-1/2", "2", "7/3"]},\n    notes="Declared rule: equivalent AND a sum of fractions with linear denominators (must actually be decomposed).")\nadd("K02", "solve", "Solve x^3 - 1 = 0 over complex numbers", "1, (-1 + isqrt(3))/2, (-1 - isqrt(3))/2",\n    {"equation": "x^3 - 1 = 0", "var": "x"},\n    {"kind": "finite", "values": ["1", "(-1 + I*sqrt(3))/2", "(-1 - I*sqrt(3))/2"]},\n    {"method": "solution_set"}, domain="complex", dsens=True, ms=MS["quad"][:3] + [r"complex|cube root|unity|imaginary"])\nadd("K03", "nsolve", "Numerically solve cos(x) = x near 0.7", "x approximately 0.7390851332",\n    {"equation": "cos(x) = x", "var": "x", "guess": "0.7"}, {"kind": "numeric_roots", "values": ["0.7390851332"], "require": "all_expected"},\n    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"],\n    notes="Closest documented API per solver (SymPy: nsolve). Candidates without a numeric root-finder get the generic equation solver.")\nadd("K04", "nsolve", "Numerically solve x^5 - x - 1 = 0", "At least the real root approximately 1.167304015",\n    {"equation": "x^5 - x - 1 = 0", "var": "x", "guess": None}, {"kind": "numeric_roots", "values": ["1.167303978261"], "require": "all_expected", "spec_printed_value": "1.167304015"},\n    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"],\n    notes="SPEC DISCREPANCY (found by reference validation, declared before any solver ran): the specification prints "\n          "\'approximately 1.167304015\' but the real root of x^5-x-1 is 1.1673039782614187 (mpmath 40 digits, bisection, polyroots). "\n          "The printed value is off by 3.7e-8. spec_expected is preserved verbatim; the verified root is the reference; tolerance 1e-8. "\n          "Extra (complex) roots are allowed. An exact CRootOf(...) is accepted if it evaluates to the real root within tolerance.")\n\n# --------------------------------------------------------------------------- L\nadd("L01", "solve", "Solve (x^2 - 1)/(x - 1) = 0", "x = -1; x = 1 remains excluded from the original expression",\n    {"equation": "(x^2 - 1)/(x - 1) = 0", "var": "x"}, {"kind": "finite", "values": ["-1"], "excluded": ["1"]},\n    {"method": "solution_set"}, dsens=True, ms=MS["lin"] + MS["dom"],\n    notes="x = 1 stays excluded: satisfied when the solution set is exactly {-1}; no separate restriction statement required.")\nadd("L02", "solve", "Solve sqrt(x + 1) = x - 1 over the reals", "x = 3 after domain and original-equation checks",\n    {"equation": "sqrt(x + 1) = x - 1", "var": "x"}, {"kind": "finite", "values": ["3"], "extraneous": ["0"]},\n    {"method": "solution_set"}, dsens=True, ms=[r"square|both sides|isolate|rearrang"] + MS["dom"])\nadd("L03", "simplify", "Simplify sqrt(x^2)", "abs(x) over the reals", {"expr": "sqrt(x^2)"}, {"expr": "abs(x)"},\n    {"method": "expr_equiv"}, dsens=True, sample={"x": ["-3", "-1/2", "2", "7/3"]}, ms=[r"abs|absolute|\\|x\\||modulus|principal|real"])\nadd("L04", "solve", "Solve 1/(x - x) = 2", "Undefined expression or domain error; no fabricated solution",\n    {"equation": "1/(x - x) = 2", "var": "x"}, {"kind": "undefined"}, {"method": "undefined"}, dsens=True,\n    ms=[r"undefined|division by zero|denominator|domain"],\n    notes="pass = explicit undefined/domain error; partial = empty set without flagging undefined; fail = any solution.")\n\n# --------------------------------------------------------------------------- V2 STRESS SUITE\n# These cases intentionally extend the original 72-case core set. They probe families,\n# not just the exact failures discovered in the first benchmark.\n\n# A: arithmetic precedence, nested powers, negative bases, exact radicals\nA("A09", "(2^3)^(2^2)", "4096", "4096")\nA("A10", "-2^2 + (-2)^2", "0", "0")\nA("A11", "(-2)^5", "-32", "-32")\nA("A12", "((3/2)^2)^3", "729/64", "729/64")\nA("A13", "sqrt(12)*sqrt(3)", "6", "6")\n\n# B: higher-degree factoring, rational cancellation and form preservation\nadd("B09", "factor", "Factor x^3 - x^2 + 2*x - 2", "(x - 1)*(x^2 + 2)",\n    {"expr": "x^3 - x^2 + 2*x - 2"}, {"expr": "(x - 1)*(x^2 + 2)"},\n    {"method": "expr_equiv", "form": "factored"}, ms=MS["factor"])\nadd("B10", "simplify", "Simplify (x^2 - 1)/(x^2 + 2*x + 1)", "(x - 1)/(x + 1), preserving x != -1",\n    {"expr": "(x^2 - 1)/(x^2 + 2*x + 1)"}, {"expr": "(x - 1)/(x + 1)"},\n    {"method": "expr_equiv", "form": "reduced_fraction"}, dsens=True, restrictions=["x != -1"],\n    sample={"x": ["-3", "-1/2", "0", "2"]}, ms=MS["simplify"] + MS["dom"][:1])\nadd("B11", "expand", "Expand (2*x - 3)*(x^2 + x + 4)", "2*x^3 - x^2 + 5*x - 12",\n    {"expr": "(2*x - 3)*(x^2 + x + 4)"}, {"expr": "2*x^3 - x^2 + 5*x - 12"},\n    {"method": "expr_equiv", "form": "expanded"}, ms=MS["expand"])\nadd("B12", "simplify", "Simplify (x^2 + 2*x + 1)/(x + 1)", "x + 1, preserving x != -1",\n    {"expr": "(x^2 + 2*x + 1)/(x + 1)"}, {"expr": "x + 1"},\n    {"method": "expr_equiv", "form": "no_fraction"}, dsens=True, restrictions=["x != -1"],\n    sample={"x": ["-3", "-1/2", "0", "2"]}, ms=MS["simplify"] + MS["dom"][:1])\nadd("B13", "factor", "Factor x^4 - 5*x^2 + 4", "(x - 2)*(x - 1)*(x + 1)*(x + 2)",\n    {"expr": "x^4 - 5*x^2 + 4"}, {"expr": "(x - 2)*(x - 1)*(x + 1)*(x + 2)"},\n    {"method": "expr_equiv", "form": "factored"}, ms=MS["factor"])\n\n# C: identity/contradiction generalisation, complex roots, and excluded candidates\nC("C11", "3*x + 2 = 2*x - 5", {"kind": "finite", "values": ["-7"]}, "x = -7")\nC("C12", "x^2 - x - 2 = 0", {"kind": "finite", "values": ["-1", "2"]}, "x = -1, 2", ms=MS["quad"])\nC("C13", "x^2 - 2*x + 5 = 0", {"kind": "finite", "values": ["1 - 2*I", "1 + 2*I"]},\n  "x = 1 - 2i, 1 + 2i", domain="complex", dsens=True,\n  ms=MS["quad"][:2] + [r"complex|imaginary|discriminant|\\bi\\b"])\nC("C14", "(x + 1)^2 = x^2 + 2*x + 1", {"kind": "all"}, "All real x (identity)", dsens=True,\n  ms=[r"identity|all (real )?(numbers|x)|infinitely many|always true|every|any x|all admissible"])\nC("C15", "x^2 + 4 = 0", {"kind": "empty"}, "No real solution", domain="real", dsens=True, ms=MS["quad"])\nC("C16", "(x^2 - 4)/(x - 2) = 0", {"kind": "finite", "values": ["-2"], "excluded": ["2"]},\n  "x = -2; x = 2 is excluded by the original denominator", dsens=True, restrictions=["x != 2"],\n  ms=MS["lin"] + MS["dom"])\n\n# D: nontrivial systems and interval boundaries/sign charts\nDsys("D07", ["2*x + 3*y = 7", "x - y = 1"], {"kind": "unique", "assignments": {"x": "2", "y": "1"}}, "x = 2, y = 1")\nDsys("D08", ["x + 2*y = 4", "2*x + 4*y = 10"], {"kind": "empty"}, "No solution", dsens=True)\nadd("D09", "solve_inequality", "Solve x^2 - 5*x + 6 > 0", "x < 2 or x > 3",\n    {"inequality": "x^2 - 5*x + 6 > 0", "var": "x"},\n    {"kind": "intervals", "intervals": [["-oo", "2", True, True], ["3", "oo", True, True]]},\n    {"method": "inequality_set"}, ms=MS["ineq"], dsens=True)\nadd("D10", "solve_inequality", "Solve -(x - 2)*(x + 1) < 0", "x < -1 or x > 2",\n    {"inequality": "-(x - 2)*(x + 1) < 0", "var": "x"},\n    {"kind": "intervals", "intervals": [["-oo", "-1", True, True], ["2", "oo", True, True]]},\n    {"method": "inequality_set"}, ms=MS["ineq"], dsens=True,\n    notes="Tests sign-chart reasoning with a leading negative and two critical points.")\nadd("D11", "solve_inequality", "Solve x^2 - 4 <= 0", "-2 <= x <= 2",\n    {"inequality": "x^2 - 4 <= 0", "var": "x"},\n    {"kind": "intervals", "intervals": [["-2", "2", False, False]]},\n    {"method": "inequality_set"}, ms=MS["ineq"], dsens=True,\n    notes="Closed interval checks inclusive inequality endpoints.")\n\n# E: native ln, composition/domain identities and exact special values\nE("E09", "ln(exp(2*x))", "2*x", "2*x over the real domain", dsens=True,\n  sample={"x": ["-3", "-1/2", "2", "7/3"]}, ms=MS["power"] + MS["dom"][:1])\nE("E10", "sqrt((x - 1)^2)", "abs(x - 1)", "abs(x - 1) over the reals", dsens=True,\n  sample={"x": ["-3", "1/2", "2", "7/3"]}, ms=[r"abs|absolute|modulus|principal|real"])\nE("E11", "sin(2*x)^2 + cos(2*x)^2", "1", "1 for all real x", dsens=True,\n  sample={"x": ["-3", "-1/2", "2", "7/3"]}, ms=MS["trig"] + [r"identity|pythagorean"])\nE("E12", "log(8, 2) + sqrt(81)", "12", "12", op="evaluate", method="value", ms=MS["power"] + MS["eval"])\nE("E13", "tan(pi/3)", "sqrt(3)", "sqrt(3)", op="evaluate", method="value", ms=MS["trig"])\n\n# F: product/chain/logarithmic rules and multivariable calculus\nF("F07", "exp(x)*sin(x)", "exp(x)*(sin(x) + cos(x))", "exp(x)*(sin(x) + cos(x))", ms=MS["diff"] + [r"product"])\nF("F08", "ln(x^2 + 1)", "2*x/(x^2 + 1)", "2x/(x^2 + 1)", ms=MS["diff"] + [r"chain|logarithmic|ln|log"])\nF("F09", "sqrt(x)/(x + 1)", "1/(2*sqrt(x)*(x + 1)) - sqrt(x)/(x + 1)^2",\n  "1/(2sqrt(x)(x + 1)) - sqrt(x)/(x + 1)^2 for x > 0", sample={"x": ["1/2", "3/2", "5/2", "4"]}, ms=MS["diff"] + [r"quotient|chain|power"])\nF("F10", "sin(x^2 + 3*x)", "(2*x + 3)*cos(x^2 + 3*x)", "(2x + 3)cos(x^2 + 3x)", ms=MS["diff"] + [r"chain"])\nadd("F11", "partial_derivative", "Differentiate x^2*y^3 + exp(x*y) with respect to y",\n    "3*x^2*y^2 + x*exp(x*y)", {"expr": "x^2*y^3 + exp(x*y)", "var": "y", "vars": ["x", "y"]},\n    {"expr": "3*x^2*y^2 + x*exp(x*y)"}, {"method": "expr_equiv"},\n    ms=MS["diff"] + [r"partial|chain"], sample={"x": ["-1", "1/2", "2", "3"], "y": ["-3/2", "1", "2", "1/3"]})\nadd("F12", "gradient", "Gradient of f(x,y) = x^2*y + y^3", "(2*x*y, x^2 + 3*y^2)",\n    {"expr": "x^2*y + y^3", "vars": ["x", "y"]}, {"vector": ["2*x*y", "x^2 + 3*y^2"]},\n    {"method": "gradient"}, ms=MS["diff"] + [r"partial|gradient"],\n    sample={"x": ["-1", "1/2", "2", "3"], "y": ["-3/2", "1", "2", "1/3"]})\n\n# G: integration rule families and domain-aware logarithmic antiderivatives\nG("G07", "4*x^3 - 3*x^2 + 2*x - 7", "x^4 - x^3 + x^2 - 7*x", "x^4 - x^3 + x^2 - 7x + C")\nG("G08", "2/(2*x + 1)", "ln(abs(2*x + 1))", "ln|2x + 1| + C, on intervals excluding x = -1/2",\n  dsens=True, restrictions=["x != -1/2"], real_on=["-3", "-1", "0", "2"],\n  ms=MS["int"] + [r"abs|absolute|logarithm|ln|domain"],\n  notes="General u\'/u rule; checks both signs of 2x+1 and excludes its zero.")\nG("G09", "sin(3*x)", "-cos(3*x)/3", "-cos(3x)/3 + C", ms=MS["int"] + [r"chain|substitut"])\nG("G10", "exp(2*x)", "exp(2*x)/2", "exp(2x)/2 + C", ms=MS["int"] + [r"chain|substitut"])\nG("G11", "x*cos(x^2)", "sin(x^2)/2", "sin(x^2)/2 + C", ms=MS["int"] + [r"substitut|u\\s*=|chain"])\nadd("G12", "integrate", "Integrate 2*x + 1 from 0 to 4", "20",\n    {"expr": "2*x + 1", "var": "x", "lower": "0", "upper": "4"}, {"expr": "20"},\n    {"method": "value"}, ms=MS["defint"])\nG("G13", "1/(x^2 + 1)", "atan(x)", "atan(x) + C", ms=MS["int"] + [r"arctan|atan|standard"])\n\n# H: cancellation, standard limits, rationalisation and squeeze-style cases\nHh("H05", "(x^2 - 1)/(x - 1)", "1", "2", "2", "Limit as x approaches 1 of (x^2 - 1)/(x - 1)")\nHh("H06", "(exp(x) - 1)/x", "0", "1", "1", "Limit as x approaches 0 of (exp(x) - 1)/x")\nHh("H07", "sin(5*x)/x", "0", "5", "5", "Limit as x approaches 0 of sin(5x)/x")\nHh("H08", "(sqrt(x + 4) - 2)/x", "0", "1/4", "1/4", "Limit as x approaches 0 of (sqrt(x + 4) - 2)/x")\nHh("H09", "x*sin(1/x)", "0", "0", "0", "Limit as x approaches 0 of x*sin(1/x)")\n\n# I: larger matrices, rank deficiency and rational systems\nM5 = [[1, 2, 3], [0, 1, 4], [5, 6, 0]]\nadd("I05", "det", "Determinant of [[1,2,3],[0,1,4],[5,6,0]]", "1", {"matrix": M5},\n    {"expr": "1"}, {"method": "value"}, ms=MS["matrix"])\nadd("I06", "inverse", "Inverse of [[2,1],[5,3]]", "[[3,-1],[-5,2]]",\n    {"matrix": [[2, 1], [5, 3]]}, {"matrix": [["3", "-1"], ["-5", "2"]]}, {"method": "matrix"}, ms=MS["matrix"])\nadd("I07", "rref", "RREF of [[1,2,3,4],[2,4,6,8],[1,1,1,1]]", "[[1,0,-1,-2],[0,1,2,3],[0,0,0,0]]",\n    {"matrix": [[1, 2, 3, 4], [2, 4, 6, 8], [1, 1, 1, 1]]},\n    {"matrix": [["1", "0", "-1", "-2"], ["0", "1", "2", "3"], ["0", "0", "0", "0"]]},\n    {"method": "matrix"}, ms=MS["matrix"])\nDsys("I08", ["x/2 + y/3 = 2", "x - y = 1"], {"kind": "unique", "assignments": {"x": "14/5", "y": "9/5"}}, "x = 14/5, y = 9/5")\nadd("I09", "rref", "RREF of [[1,2,1],[2,4,2],[1,1,0]]", "[[1,0,-1],[0,1,1],[0,0,0]]",\n    {"matrix": [[1, 2, 1], [2, 4, 2], [1, 1, 0]]},\n    {"matrix": [["1", "0", "-1"], ["0", "1", "1"], ["0", "0", "0"]]},\n    {"method": "matrix"}, ms=MS["matrix"])\n\n# J: additional ODE families and multivariable chain rules. No new adapter overrides are introduced.\nadd("J05", "dsolve", "Solve y\' + 2y = 0", "y = C*exp(-2x)",\n    {"ode": "Derivative(y(x), x) + 2*y(x)", "func": "y", "var": "x", "order": 1},\n    {"family": "C*exp(-2*x)", "n_constants": 1}, {"method": "ode_family"}, ms=MS["ode"])\nadd("J06", "dsolve", "Solve y\'\' - 3y\' + 2y = 0", "y = C1*exp(x) + C2*exp(2x)",\n    {"ode": "Derivative(y(x), (x, 2)) - 3*Derivative(y(x), x) + 2*y(x)", "func": "y", "var": "x", "order": 2},\n    {"family": "C1*exp(x) + C2*exp(2*x)", "n_constants": 2}, {"method": "ode_family"}, ms=MS["ode"])\nadd("J07", "partial_derivative", "Differentiate sin(x*y) with respect to x", "y*cos(x*y)",\n    {"expr": "sin(x*y)", "var": "x", "vars": ["x", "y"]}, {"expr": "y*cos(x*y)"},\n    {"method": "expr_equiv"}, ms=MS["diff"] + [r"partial|chain"],\n    sample={"x": ["-1", "1/2", "2", "3"], "y": ["-3/2", "1", "2", "1/3"]})\nadd("J08", "gradient", "Gradient of f(x,y) = exp(x*y) + x^2*y", "(y*exp(x*y) + 2*x*y, x*exp(x*y) + x^2)",\n    {"expr": "exp(x*y) + x^2*y", "vars": ["x", "y"]},\n    {"vector": ["y*exp(x*y) + 2*x*y", "x*exp(x*y) + x^2"]}, {"method": "gradient"},\n    ms=MS["diff"] + [r"partial|gradient|chain"],\n    sample={"x": ["-1", "1/2", "2", "3"], "y": ["-3/2", "1", "2", "1/3"]})\n\n# K: complex roots, repeated poles and several different transcendental/numeric roots\nadd("K05", "solve", "Solve x^4 + 4 = 0 over the complex numbers", "x = -1-i, -1+i, 1-i, 1+i",\n    {"equation": "x^4 + 4 = 0", "var": "x"},\n    {"kind": "finite", "values": ["-1 - I", "-1 + I", "1 - I", "1 + I"]},\n    {"method": "solution_set"}, domain="complex", dsens=True,\n    ms=MS["quad"][:3] + [r"complex|imaginary|quartic"])\nadd("K06", "solve", "Solve x^4 - 5*x^2 + 4 = 0 over the reals", "x = -2, -1, 1, 2",\n    {"equation": "x^4 - 5*x^2 + 4 = 0", "var": "x"},\n    {"kind": "finite", "values": ["-2", "-1", "1", "2"]},\n    {"method": "solution_set"}, domain="real", dsens=True, ms=MS["quad"])\nadd("K07", "partial_fractions", "Partial fractions of 1/(x^2*(x + 1))", "-1/x + 1/x^2 + 1/(x + 1)",\n    {"expr": "1/(x^2*(x + 1))", "var": "x"}, {"expr": "-1/x + 1/x^2 + 1/(x + 1)"},\n    {"method": "expr_equiv", "form": "partial_fractions"}, ms=MS["pf"], dsens=True,\n    restrictions=["x != 0", "x != -1"], sample={"x": ["-3", "-2", "1/2", "2"]})\nadd("K08", "nsolve", "Numerically solve x^3 - 2*x - 5 = 0 near 2", "x approximately 2.0945514815",\n    {"equation": "x^3 - 2*x - 5 = 0", "var": "x", "guess": "2"},\n    {"kind": "numeric_roots", "values": ["2.0945514815"], "require": "all_expected"},\n    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"])\nadd("K09", "nsolve", "Numerically solve exp(-x) = x near 0.5", "x approximately 0.5671432904",\n    {"equation": "exp(-x) = x", "var": "x", "guess": "0.5"},\n    {"kind": "numeric_roots", "values": ["0.5671432904"], "require": "all_expected"},\n    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"])\nadd("K10", "nsolve", "Numerically solve sin(x) = 1/2 near 0.5", "x approximately pi/6 = 0.5235987756",\n    {"equation": "sin(x) = 1/2", "var": "x", "guess": "0.5"},\n    {"kind": "numeric_roots", "values": ["0.5235987756"], "require": "all_expected"},\n    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"])\nadd("K11", "nsolve", "Numerically solve ln(x) = 1 near 2.5", "x approximately e = 2.7182818285",\n    {"equation": "ln(x) = 1", "var": "x", "guess": "2.5"},\n    {"kind": "numeric_roots", "values": ["2.7182818285"], "require": "all_expected"},\n    {"method": "numeric_roots"}, tol=TOL_NUM, ms=MS["num"])\nadd("K12", "solve", "Solve x^3 + 2*x^2 + 2*x + 1 = 0 over the complex numbers", "x = -1, (-1 ± i*sqrt(3))/2",\n    {"equation": "x^3 + 2*x^2 + 2*x + 1 = 0", "var": "x"},\n    {"kind": "finite", "values": ["-1", "(-1 - I*sqrt(3))/2", "(-1 + I*sqrt(3))/2"]},\n    {"method": "solution_set"}, domain="complex", dsens=True, ms=MS["quad"][:3] + [r"complex|imaginary"])\n\n# L: generalized domain exclusions, extraneous roots and undefined-expression handling\nadd("L05", "simplify", "Simplify ((x - 2)*(x + 2))/(x - 2)", "x + 2, preserving x != 2",\n    {"expr": "((x - 2)*(x + 2))/(x - 2)"}, {"expr": "x + 2"}, {"method": "expr_equiv", "form": "no_fraction"},\n    dsens=True, restrictions=["x != 2"], sample={"x": ["-3", "-1", "0", "3"]}, ms=MS["simplify"] + MS["dom"][:1])\nadd("L06", "solve", "Solve sqrt(x - 3) = x - 3 over the reals", "x = 3, 4",\n    {"equation": "sqrt(x - 3) = x - 3", "var": "x"},\n    {"kind": "finite", "values": ["3", "4"]}, {"method": "solution_set"}, domain="real", dsens=True,\n    ms=[r"square|both sides|isolate|rearrang"] + MS["dom"])\nadd("L07", "solve", "Solve (x^2 - 1)/(x - 1) = 2 with the original domain", "No solution: the only algebraic candidate x=1 is excluded",\n    {"equation": "(x^2 - 1)/(x - 1) = 2", "var": "x"}, {"kind": "empty"}, {"method": "solution_set"},\n    domain="real", dsens=True, restrictions=["x != 1"], ms=MS["lin"] + MS["dom"])\nadd("L08", "solve", "Solve 1/(x - 1) = 1/(x - 1)", "All real x except x=1",\n    {"equation": "1/(x - 1) = 1/(x - 1)", "var": "x"}, {"kind": "all"}, {"method": "solution_set"},\n    domain="real", dsens=True, restrictions=["x != 1"], ms=[r"identity|all (real )?(numbers|x)|domain|denominator|excluded"])\nadd("L09", "solve", "Solve 1/(x^2 - 4) = 0", "No solution; denominator excludes x = -2 and x = 2",\n    {"equation": "1/(x^2 - 4) = 0", "var": "x"}, {"kind": "empty"}, {"method": "solution_set"},\n    domain="real", dsens=True, restrictions=["x != -2", "x != 2"], ms=MS["lin"] + MS["dom"])\nadd("L10", "solve", "Solve 1/(x - x) = x", "Undefined: denominator is identically zero",\n    {"equation": "1/(x - x) = x", "var": "x"}, {"kind": "undefined"}, {"method": "undefined"},\n    domain="real", dsens=True, ms=[r"undefined|division by zero|denominator|domain"])\nadd("L11", "simplify", "Simplify x^3/x^2", "x, preserving x != 0",\n    {"expr": "x^3/x^2"}, {"expr": "x"}, {"method": "expr_equiv", "form": "no_fraction"},\n    dsens=True, restrictions=["x != 0"], sample={"x": ["-3", "-1/2", "2", "7/3"]}, ms=MS["simplify"] + MS["dom"][:1])\nadd("L12", "solve", "Solve sqrt(x + 2) = x over the reals", "x = 2; reject x = -1 after checking the original equation",\n    {"equation": "sqrt(x + 2) = x", "var": "x"},\n    {"kind": "finite", "values": ["2"], "extraneous": ["-1"]}, {"method": "solution_set"},\n    domain="real", dsens=True, ms=[r"square|both sides|isolate|rearrang"] + MS["dom"])\n\nV2_STRESS_IDS = {c["id"] for c in cases if c["id"] in {\n    "A09","A10","A11","A12","A13",\n    "B09","B10","B11","B12","B13",\n    "C11","C12","C13","C14","C15","C16",\n    "D07","D08","D09","D10","D11",\n    "E09","E10","E11","E12","E13",\n    "F07","F08","F09","F10","F11","F12",\n    "G07","G08","G09","G10","G11","G12","G13",\n    "H05","H06","H07","H08","H09",\n    "I05","I06","I07","I08","I09",\n    "J05","J06","J07","J08",\n    "K05","K06","K07","K08","K09","K10","K11","K12",\n    "L05","L06","L07","L08","L09","L10","L11","L12"\n}}\nfor c in cases:\n    c["suite"] = "v2_stress" if c["id"] in V2_STRESS_IDS else "core_72"\n\n\nassert len(cases) == 141, len(cases)\nassert len({c["id"] for c in cases}) == 141\nassert sum(c["suite"] == "core_72" for c in cases) == 72\nassert sum(c["suite"] == "v2_stress" for c in cases) == 69\n\nout = pathlib.Path(__file__).resolve().parent.parent / "data" / "cases.json"\nout.write_text(json.dumps({"schema_version": 1, "count": len(cases), "categories": CATS, "cases": cases}, indent=1, ensure_ascii=False) + "\\n")\nprint("wrote", out, "cases:", len(cases))\n')
W('docs/METHODOLOGY.md', "# Methodology and declared decisions\n\n## Isolation\nThree virtualenvs (`envs/sympy|smartsolver|stepsolver`), one worker process per solver (`bench/workers/*_worker.py`), launched from an empty temp directory with a minimal environment. The controller (`bench/runner.py`) talks JSON lines over pipes, enforces a 20 s per-case timeout (kills and restarts the worker), and preserves every raw reply in `results/raw/<solver>.json`. No application/production code exists in or is referenced by this workspace. SymPy runs through `bench/workers/sympy_worker.py`, which imports only `sympy`.\n\n## Adapters (what each solver was asked)\nDocumented at the top of each worker file. Summary: SymPy bare API (simplify/expand/factor, `solveset` with explicit real/complex domain, `solve` for systems, `solve_univariate_inequality`, `diff`, `integrate`, `limit(dir='+-')`, Matrix ops, `dsolve`, `apart`, `nsolve`/`Poly.nroots`); SmartSolver one documented public method per operation family, **unsupported** where none exists (public-method list recorded in each reply); StepSolver documented ASCII operations, direct `nsolve(equation, variable, guess)` calls for numerical cases that supply an initial guess, the original high-degree polynomial K04 fallback case (which has no guess), the two original per-case ODE syntax overrides in `cases.json:adapter_inputs`, and gradient as two native `diff` calls. V2 sends `ln(...)` exactly as written (no adapter normalisation), so native alias support is measured rather than hidden. Counts of adaptations are measured (`adapter_overrides`).\n\n## Judge (`bench/judge.py`)\nNo string-equality-only comparison. Value: symbolic zero or 50-digit numeric agreement. Expression equivalence: symbolic zero and/or agreement at exact rational sample points (a complex/NaN value at a real sample point is a mismatch). Solution sets: exact set equality over the declared domain. Systems, inequality sets (via `as_set`), matrices, gradients, ODE general solutions (substitution residual + constant count + independent basis/Wronskian), numeric roots (tolerance 1e-8), antiderivatives (F'=f, plus real-valuedness on both sides of 0 for G04), undefined-expression handling (L04). Form checks and restriction-reporting rules are declared in `cases.json` and the module docstring. Judge parsing of solver text uses an allow-list and no builtins.\n\n## Reference validation (`bench/reference_validation.py`)\nThe original 72 core reference answers are re-checked with stdlib `Fraction`/`cmath` and mpmath numerics (numerical differentiation/quadrature/root finding, exhaustive rational scans, truth tables for inequalities), without SymPy symbolic solvers. The 69 V2 stress references are independently checked before candidate execution by `bench/v2_reference_validation.py` using operation-specific SymPy identities and residual checks. Both validation reports are preserved under `results/`. The core validation exposed the K04 specification typo (printed 1.167304015 vs true 1.1673039782614187); the verbatim spec text is kept in `spec_expected`, the verified root is the reference.\n\n## Declared benchmark decisions inherited from V1\n\nThe original core suite retains its previously declared rules: `L01` can pass by excluding the invalid value from the solution set without repeating a restriction statement; generic simplify/evaluate form checks reject answers that echo the original input; ODE input overrides are listed in the case data; raw rule-tree dumps are not counted as prose; separate answer/workings code paths are capped at correctness 2; opaque-only derivations get correctness<=2, consistency 0 and presentation 0; and boilerplate labels such as “Apply algebraic identities” do not count as specific rules. The original derivation overlays remain fingerprinted and stale overlays are ignored.\n\n## V2 protocol changes declared before the run\n\n- The StepSolver adapter sends `ln(...)` exactly as written; no `ln->log` rewrite is applied to the scored path. This specifically tests native parser support and prevents adapter normalisation from hiding a regression.\n- The original 72 core case IDs and their expectations are unchanged. Sixty-nine additional cases were appended under fresh IDs, before any V2 solver run; `bench/v2_reference_validation.py` validates their references before the candidate workers run.\n- Direct printer, unknown-function and zero-denominator probes are reported separately from the scored mathematics cases because they inspect presentation/parser-safety properties rather than a conventional final-answer comparison.\n- V2 supports inclusive inequality operators in the standalone SymPy adapter (`<=`, `>=`) so that the reference engine can fairly answer the harder boundary tests.\n\n## Manual review\nIn the baseline run, 39 of 113 scored derivations were reviewed by a language model (Claude) using the printed raw workings; each entry has adjustments, confidence and justification in `data/manual_reviews.json`; `raw_results.json` keeps automated and final scores. It is **not** human review and must be audited. Unreviewed derivations use automated scores. Mathematical correctness is not decided by this review.\n\n## Reproducibility\n`./run_all.sh` (clean machine with git, uv, Python 3.12). Candidate commits and dependency pins: `pins/`. When enabled, two full runs compare all 423 solver/case rows (141 cases × 3 solvers) in `results/reproducibility_check.json`; only latencies vary.\n\n## Re-running on new engine versions (Colab notebook)\nThe V2 notebook rebuilds isolated environments for the engine versions you configure, runs the same judge and scoring framework against the original 72 core cases plus 69 tougher stress cases, adds direct printer/parser probes, and regenerates reports from data (no run-specific outcome claims). The manual derivation overlay carries a fingerprint (hash of the reviewed steps+answer): it is applied only when an engine's output is unchanged; otherwise the automated score is used and the report states how many reviews were not applied. The judge runs in a separate environment with a fixed SymPy version so the same answers are judged identically across engine versions. `data/baseline.json` stores the original run for `changes_vs_baseline.md`.\n")
W('docs/SCORING_RULES.md', '# Scoring rules (declared before the overall score was computed)\n\n`Overall = 0.30*Correctness + 0.30*Derivations + 0.20*Coverage + 0.10*Reliability + 0.05*Integration + 0.05*Performance`\n\nApplicable cases = all 141 for every solver (no case is excluded as "not applicable"; unsupported cases count against the solver).\n\n| Component | Normalisation to 0-100 |\n|---|---|\n| Correctness | strictly-correct cases / 141 x 100. Strict = correct value (symbolic equivalence, set equality, substitution/ODE residual, numeric tolerance) AND required form AND (if the case requires it) the domain restriction reported. Unsupported, runtime errors and invalid answers all count as not correct but are reported separately. Value-only correctness is reported as a sensitivity. |\n| Derivations | (mean final rubric score /10 over derivations that exist and whose answer was produced) x 10, as specified. Availability reported separately; an availability-penalised variant (unavailable = 0 over all 141) is in the sensitivity table. |\n| Coverage | cases where the solver produced a result in the expected form / 141 x 100 (correctness not required). |\n| Reliability | 100 x mean(R1..R5): R1 share of repeated identical runs (5 cache-cleared repeats + main pass) with identical canonical output; R2 mean engineering-probe score (1 / 0.5 / 0, n/a excluded); R3 = 1 - (runtime errors+timeouts+infra)/attempted (attempted = cases where an API exists); R4 strict-correct rate on the declared domain-sensitive cases; R5 = 1 if the benign canary payload is rejected by the solver\'s own string interface, else 0. |\n| Integration & maintainability | 14-point rubric computed from evidence: env compat 2 (<=4 third-party deps; 1 if <=30), install 2 (single command; 1 if it needed an unpinned-SymPy fallback), adaptation effort 2 (<=2% of cases need overrides; 1 if <=10%), testability/extensibility 2 (>=100 static test functions; 1 if >=1; -1 if the own suite was run and fails; SymPy fixed 0: no derivation framework), licence 2 (permissive; 1 unknown; 0 copyleft), maintenance 2 (>=2 years and >=5 authors; 1 if >=5 commits and active within a year; SymPy fixed 2), UI independence 1 and no-production-rewrite 1 (fixed assumptions) -> x 100/14. Evidence strings are in `scores.json`. |\n| Performance | latency score S(m) = 100 if m<=10 ms; 0 if m>=2000 ms; else 100*log10(2000/m)/log10(200). Performance = 0.4*S(median) + 0.4*S(p95) + 0.2*100*(1-failure rate). Median/p95 are warm-process, SymPy-cache-cleared repeats over the cases ALL three solvers answered. Cold start and memory are reported, not scored. |\n\n## Derivation rubric (10 points) - automated implementation\nSee the docstring of `bench/derivations.py`. Criteria: correctness 0-3, completeness 0-2, explanation 0-2, consistency 0-1, domain awareness 0-1, presentation 0-1. A manual overlay (`data/manual_reviews.json`) records per-criterion adjustments with confidence and justification; the overlay author is a language model and is flagged as requiring human audit. Mathematical correctness is never decided by a language model: it comes from `bench/judge.py` and `bench/reference_validation.py`.\n\n## Hard gates\n- G1 critical correctness defect: a confidently wrong answer (false "no solution", wrong value/set, fabricated solution). FAIL if >=1 confirmed.\n- G2 invalid or self-contradicting derivation (automated invalid-step detection, `direct_evaluation` override, or manual finding of contradiction/unchanged-input final step). FAIL if >=2 cases, CONDITIONAL if 1.\n- G3 benign canary `__import__(\'math\').factorial(4)` evaluated by the solver\'s own string interface. FAIL if evaluated and no safe alternative interface; CONDITIONAL for SymPy (programmatic API + allow-list adapter demonstrated).\n- G4 licence incompatible with commercial use: none found (BSD-3, MIT, MIT).\n')
W('data/manual_reviews.json', '{\n "stepsolver|A04": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "high",\n  "justification": "Single generic \'Apply algebraic identities\' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown. Essential factoring of 72 skipped.",\n  "fingerprint": "d2c5a1815a14a5f4"\n },\n "stepsolver|A08": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Single generic \'Apply algebraic identities\' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",\n  "fingerprint": "88d43fea6b54fe76"\n },\n "stepsolver|E04": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Single generic \'Apply algebraic identities\' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",\n  "fingerprint": "fc5c446851e8ede3"\n },\n "stepsolver|E06": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Single generic \'Apply algebraic identities\' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",\n  "fingerprint": "4a001801febd6348"\n },\n "stepsolver|E07": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Single generic \'Apply algebraic identities\' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",\n  "fingerprint": "7e4e2bf5616a7fa5"\n },\n "stepsolver|E08": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Single generic \'Apply algebraic identities\' step: the specific identity (e.g. 72=36*2 / special-angle value / log definition) is never named or shown.",\n  "fingerprint": "fc32ccf45b3afce8"\n },\n "stepsolver|A03": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Mentions a common denominator but never shows the LCD (36) or the rewritten numerators; result appears in one jump.",\n  "fingerprint": "873b1d8cbf095f42"\n },\n "stepsolver|A07": {\n  "adjust": {\n   "correctness": 1,\n   "consistency": 0,\n   "presentation": 0\n  },\n  "confidence": "high",\n  "justification": "Displayed first step is \'-3 ^ 2 - 3 ^ 2\' (LaTeX: -3^{2} - 3^{2}) i.e. parentheses around the negative base are lost; as displayed it equals -18, not the problem\'s value. Final answer 0 is right, but the shown expression misstates the problem. Reproduce: Solver().solve(\'simplify((-3)^2 - 3^2)\').",\n  "fingerprint": "15c441dccd74cb75"\n },\n "stepsolver|B06": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Explicit restriction \'x - 1 != 0\' reported (good) but the factorisation (x-1)(x+1) that justifies the cancellation is described, not shown.",\n  "fingerprint": "34a560a177eece67"\n },\n "stepsolver|C01": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Clear, justified inverse-operation steps. Steps 2 and 5 display identical before/after text (cancellation only implied in ASCII) - minor. No change.",\n  "fingerprint": "ba795e5b0bf49d85"\n },\n "stepsolver|C07": {\n  "adjust": {\n   "correctness": 1,\n   "domain": 1\n  },\n  "confidence": "medium",\n  "justification": "Derivation is valid and explicitly bounded to the real number system (\'no roots in the real number system\'), but stops there; the benchmark expects complex roots i,-i. Partial credit for valid workings; the final answer itself stays \'invalid_answer (real_domain_only)\'.",\n  "fingerprint": "a23a89b48f84ebbd"\n },\n "stepsolver|F03": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Product rule named and described, but u, v, u\', v\' and the u\'v+uv\' assembly are not displayed; result appears in one jump.",\n  "fingerprint": "5dedf8f3621706a1"\n },\n "stepsolver|H03": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Key idea (dominant leading powers) stated clearly, but the division by x^2 step is not shown.",\n  "fingerprint": "028f84a47983df3a"\n },\n "stepsolver|G05": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "Clear, ordered, justified steps with correct rule names; domain restrictions handled where relevant (L01 discards x=1). No change.",\n  "fingerprint": "91e8344a6ece8f9d"\n },\n "stepsolver|G06": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "Clear, ordered, justified steps with correct rule names; domain restrictions handled where relevant (L01 discards x=1). No change.",\n  "fingerprint": "25e50c9738d6c60e"\n },\n "stepsolver|L01": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "Clear, ordered, justified steps with correct rule names; domain restrictions handled where relevant (L01 discards x=1). No change.",\n  "fingerprint": "610c97357f665056"\n },\n "stepsolver|H01": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Valid L\'Hopital derivation with correct explanations. Pedagogical caveat: L\'Hopital for sin(x)/x is circular in many curricula; not penalised.",\n  "fingerprint": "a5b3abe1616a506c"\n },\n "stepsolver|D04": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "Only the opaque \'Compute exact result ... with the symbolic backend\' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",\n  "fingerprint": "f1a9ac3c3647ccc5"\n },\n "stepsolver|I03": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "Only the opaque \'Compute exact result ... with the symbolic backend\' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",\n  "fingerprint": "43692c4717f1260d"\n },\n "stepsolver|J01": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "Only the opaque \'Compute exact result ... with the symbolic backend\' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",\n  "fingerprint": "95bd57b17e94af84"\n },\n "stepsolver|L02": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "Only the opaque \'Compute exact result ... with the symbolic backend\' step: no workings at all (for L02 no squaring/domain/extraneous-root check although the spec requires them). Rubric rule 5/9: score stays at the minimum automated level.",\n  "fingerprint": "508aa821c2a87214"\n },\n "smartsolver|C07": {\n  "adjust": {\n   "completeness": 1,\n   "domain": 1\n  },\n  "confidence": "medium",\n  "justification": "Correctly derives complex roots (discriminant -4) and names the complex-domain result, but never lists both roots in the steps (only \'-I\' shown) and has a pointless \'complete the square\' display.",\n  "fingerprint": "ccfd830bd1abdecc"\n },\n "smartsolver|F03": {\n  "adjust": {\n   "explanation": 2\n  },\n  "confidence": "medium",\n  "justification": "Product rule named with u and v given and u\'v+uv\' assembled explicitly.",\n  "fingerprint": "2ca7dccae09337e6"\n },\n "smartsolver|G05": {\n  "adjust": {\n   "explanation": 2,\n   "presentation": 0\n  },\n  "confidence": "high",\n  "justification": "Clear u-substitution steps, but an internal variable name leaks into the displayed workings (\'exp(_u)/2\').",\n  "fingerprint": "371571bc499021b7"\n },\n "smartsolver|G06": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Power rule and FTC named, but F(3)-F(0) arithmetic (27/3 - 0) is skipped.",\n  "fingerprint": "26488e1e5383791a"\n },\n "smartsolver|G04": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "CONTRADICTION: step 2 states the antiderivative is ln|x|, then a \'Direct evaluation\' step replaces it with log(x), which is not real-valued for x<0. steps_mode=direct_evaluation. Counts toward the fabricated/inconsistent-derivation hard gate.",\n  "fingerprint": "e5af563d33f2f413"\n },\n "smartsolver|K01": {\n  "adjust": {\n   "correctness": 1,\n   "completeness": 1\n  },\n  "confidence": "high",\n  "justification": "Factorisation and cover-up coefficients (1/2, -1/2) are correct, but the final \'Partial fraction decomposition\' step shows the ORIGINAL expression 1/(x**2-1) - the assembled decomposition is never produced.",\n  "fingerprint": "b396ac1897bed41c"\n },\n "smartsolver|K02": {\n  "adjust": {\n   "completeness": 1\n  },\n  "confidence": "medium",\n  "justification": "Factors x^3-1 correctly, then \'Solve factor x**2+x+1 = 0\' without ever showing the complex roots.",\n  "fingerprint": "cb9be6b4b35849ba"\n },\n "smartsolver|C01": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",\n  "fingerprint": "7e11e87b61dc90ad"\n },\n "smartsolver|D01": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",\n  "fingerprint": "5820ea7817edf2cb"\n },\n "smartsolver|I04": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",\n  "fingerprint": "e6be2c5610c8d597"\n },\n "smartsolver|J03": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",\n  "fingerprint": "84743dea511a1466"\n },\n "smartsolver|J04": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Ordered, correctly labelled steps.",\n  "fingerprint": "ae2407ffb19ef063"\n },\n "smartsolver|F06": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Names logarithmic differentiation but shows none of the method.",\n  "fingerprint": "f28642b8244cb20a"\n },\n "smartsolver|H01": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Only \'Setup / Limit setup / Final result\' - no justification for the limit value.",\n  "fingerprint": "d086deaaede64b49"\n },\n "smartsolver|L01": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Only \'Solved: x = -1\'; no domain reasoning shown for the x=1 exclusion.",\n  "fingerprint": "384d49a09d4c9ebc"\n },\n "smartsolver|L02": {\n  "adjust": {},\n  "confidence": "medium",\n  "justification": "Reviewed: automated score accepted. Only \'Solved: x = 3\'; no squaring/extraneous-root check shown.",\n  "fingerprint": "45fb3c6eb5d03267"\n },\n "sympy-direct|G01": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "SymPy\'s integral_steps returns a raw rule-tree repr (ConstantTimesRule(...)): rules are identifiable by name but unexplained and not student-readable; it comes from a separate code path from integrate(), so it does not prove how the answer was obtained.",\n  "fingerprint": "2fbaf191d7867bf6"\n },\n "sympy-direct|G05": {\n  "adjust": {},\n  "confidence": "high",\n  "justification": "SymPy\'s integral_steps returns a raw rule-tree repr (ConstantTimesRule(...)): rules are identifiable by name but unexplained and not student-readable; it comes from a separate code path from integrate(), so it does not prove how the answer was obtained.",\n  "fingerprint": "60c0ed88db2fecc6"\n }\n}')
W('data/baseline.json', '{"label": "original benchmark run: SymPy 1.14.0, SmartSolver 0.4.1 @ f0d46a1, StepSolver 0.1.0 @ bd8ea4f (2026-10-23, 1-core sandbox)", "engines": {"sympy-direct": {"version": "1.14.0", "commit": null}, "smartsolver": {"version": "0.4.1", "commit": "f0d46a1089366f8e14f73de9ba811d8e4c1471b0"}, "stepsolver": {"version": "0.1.0", "commit": "bd8ea4ffdbe49277d649017cb6a31bf05b376bd8"}}, "overall": {"sympy-direct": 75.54963105541215, "smartsolver": 59.34776249462321, "stepsolver": 87.36096356205924}, "components": {"sympy-direct": {"correctness": 91.66666666666667, "derivations": 43.33333333333333, "coverage": 98.61111111111111, "reliability": 64.28571428571429, "integration": 85.71428571428571, "performance": 92.26246237808425}, "smartsolver": {"correctness": 48.611111111111114, "derivations": 67.94871794871794, "coverage": 51.388888888888886, "reliability": 57.11904761904761, "integration": 85.71428571428571, "performance": 82.08833902555335}, "stepsolver": {"correctness": 93.05555555555556, "derivations": 80.29411764705883, "coverage": 95.83333333333333, "reliability": 90.19047619047619, "integration": 78.57142857142857, "performance": 64.83551773978422}}, "cases": {"sympy-direct|A01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|A08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|B06": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|B07": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|B08": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|C01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C09": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|C10": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|D06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E02": {"status": "valid", "strict_correct": false, "derivation_final": null}, "sympy-direct|E03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E07": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|E08": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F05": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|F06": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|G01": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G02": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G03": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 2}, "sympy-direct|G05": {"status": "valid", "strict_correct": true, "derivation_final": 5}, "sympy-direct|G06": {"status": "valid", "strict_correct": true, "derivation_final": 4}, "sympy-direct|H01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|H02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|H03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|H04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|I04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|J04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|K04": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L01": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L02": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L03": {"status": "valid", "strict_correct": true, "derivation_final": null}, "sympy-direct|L04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": null}, "smartsolver|A01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|A08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|B08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|C01": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C02": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|C06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|C07": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|C08": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|C09": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|C10": {"status": "runtime_error", "strict_correct": false, "derivation_final": null}, "smartsolver|D01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|D02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|D03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|D04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|D05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|D06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E05": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E06": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E07": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|E08": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|F01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|F02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|F03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "smartsolver|F04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|F05": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|F06": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "smartsolver|G01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|G02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|G03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|G04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 3}, "smartsolver|G05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|G06": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|H01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|H02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|H03": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|H04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "smartsolver|I01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|I02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|I03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|I04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|J01": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|J02": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|J03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "smartsolver|J04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "smartsolver|K01": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 4}, "smartsolver|K02": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "smartsolver|K03": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 1}, "smartsolver|K04": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "smartsolver|L01": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|L02": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "smartsolver|L03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "smartsolver|L04": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 0}, "stepsolver|A01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|A02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|A03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|A04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|A05": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|A06": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|A07": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "stepsolver|A08": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|B01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|B05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|B06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|B07": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|B08": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C03": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C04": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|C05": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|C06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|C07": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 6}, "stepsolver|C08": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|C09": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|C10": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|D01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|D02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|D03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|D04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|D05": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|D06": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|E01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|E02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|E03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E04": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E05": {"status": "valid", "strict_correct": true, "derivation_final": 6}, "stepsolver|E06": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E07": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|E08": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|F01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F02": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|F05": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|F06": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|G01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|G02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|G03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|G04": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|G05": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|G06": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|H01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|H02": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|H03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|H04": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|I01": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|I02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|I03": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|I04": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|J01": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|J02": {"status": "valid", "strict_correct": true, "derivation_final": 3}, "stepsolver|J03": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|J04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|K01": {"status": "valid", "strict_correct": true, "derivation_final": 7}, "stepsolver|K02": {"status": "invalid_answer", "strict_correct": false, "derivation_final": 4}, "stepsolver|K03": {"status": "unsupported", "strict_correct": false, "derivation_final": null}, "stepsolver|K04": {"status": "valid", "strict_correct": true, "derivation_final": 9}, "stepsolver|L01": {"status": "valid", "strict_correct": true, "derivation_final": 10}, "stepsolver|L02": {"status": "valid", "strict_correct": true, "derivation_final": 2}, "stepsolver|L03": {"status": "valid", "strict_correct": true, "derivation_final": 8}, "stepsolver|L04": {"status": "valid", "strict_correct": true, "derivation_final": null}}}')
print("ok: wrote 5 V2 benchmark source files")


ok: wrote 5 V2 benchmark source files


In [4]:
W('bench/__init__.py', '')
W('bench/workers/__init__.py', '')
W('bench/workers/common.py', '"""Shared worker protocol (stdlib only). One JSON request per line on stdin, one JSON reply per line on stdout.\n\nA worker module defines:  NAME, info() -> dict, solve_case(case: dict) -> dict (see schema in docs/METHODOLOGY.md)\nThe worker measures ONLY the solver call (perf_counter around it); adapter (de)serialisation is excluded.\nSolver stdout/stderr noise is redirected so it cannot corrupt the protocol.\n"""\nimport contextlib\nimport io\nimport json\nimport os\nimport resource\nimport sys\nimport time\nimport traceback\n\nPROTO_OUT = sys.stdout   # keep the real stdout for protocol messages\n\n\ndef timed(fn, *a, **kw):\n    t = time.perf_counter()\n    try:\n        r = fn(*a, **kw)\n        return r, (time.perf_counter() - t) * 1000.0, None\n    except BaseException as e:   # noqa: BLE001  (RecursionError etc. must be captured, not crash the worker)\n        if isinstance(e, (KeyboardInterrupt, SystemExit)):\n            raise\n        return None, (time.perf_counter() - t) * 1000.0, e\n\n\ndef err_dict(e):\n    tb = traceback.format_exception(type(e), e, e.__traceback__)\n    return {"type": type(e).__name__, "message": str(e)[:500], "traceback_tail": "".join(tb[-4:])[-1200:]}\n\n\ndef serve(mod):\n    t_boot = time.perf_counter()\n    for line in sys.stdin:\n        line = line.strip()\n        if not line:\n            continue\n        req = json.loads(line)\n        cmd = req.get("cmd")\n        if cmd == "shutdown":\n            break\n        sink = io.StringIO()\n        try:\n            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):\n                if cmd == "info":\n                    out = mod.info()\n                elif cmd == "ping":\n                    out = {"pong": True}\n                elif cmd == "clear_cache":\n                    try:\n                        from sympy.core.cache import clear_cache\n                        clear_cache()\n                        out = {"cleared": True}\n                    except Exception as ce:   # noqa: BLE001\n                        out = {"cleared": False, "error": str(ce)}\n                elif cmd == "solve":\n                    out = mod.solve_case(req["case"])\n                elif cmd == "raw":     # engineering tests: solver-specific probe\n                    out = mod.raw_probe(req["probe"])\n                else:\n                    out = {"error": {"type": "BadCommand", "message": str(cmd)}}\n        except BaseException as e:   # noqa: BLE001\n            out = {"outcome": "error", "error": err_dict(e), "worker_level_failure": True}\n        out["req_id"] = req.get("req_id")\n        out["maxrss_kb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n        if sink.getvalue():\n            out["captured_stdout_stderr"] = sink.getvalue()[:2000]\n        PROTO_OUT.write(json.dumps(out, default=str, ensure_ascii=True) + "\\n")\n        PROTO_OUT.flush()\n')
W('bench/workers/safeinput.py', '"""Allow-list validation for benchmark-controlled math strings (never executes anything)."""\nimport re\n\n_OK = re.compile(r"^[A-Za-z0-9_+\\-*/^().,=<>!&| \\[\\]]*$")\n_ATTR = re.compile(r"[A-Za-z_\\)\\]]\\s*\\.\\s*[A-Za-z_]")\n\n\ndef check(s):\n    if not isinstance(s, str) or len(s) > 400:\n        raise ValueError("input rejected: not a short string")\n    if "__" in s or not _OK.match(s) or _ATTR.search(s):\n        raise ValueError("input rejected by allow-list: %r" % s[:80])\n    return s\n')
W('bench/workers/sympy_worker.py', '"""Standalone SymPy adapter (configuration \'sympy-direct\').\n\nDesign: SymPy is evaluated as a BARE ENGINE through its public API, exactly as an application developer\nwould call it, with NO added step generator and NO added domain-tracking layer.  Consequences (documented,\nnot hidden): SymPy provides no student-facing derivation (except sympy.integrals.manualintegrate.integral_steps\nfor integrals, captured as-is), and reports no domain restrictions unless the API returns them.\n\nAPI choices (declared before execution):\n  evaluate/simplify -> simplify      expand -> expand      factor -> factor\n  solve             -> solveset(Eq, x, S.Reals | S.Complexes per case.domain)   [explicit domain]\n  solve_system      -> solve(..., dict=True)          solve_inequality -> solve_univariate_inequality\n  diff/partial/grad -> diff                           integrate -> integrate (indefinite & definite)\n  limit             -> limit(dir=\'+-\')                det/inverse/rref -> Matrix.det/.inv/.rref\n  dsolve            -> dsolve                         partial_fractions -> apart\n  nsolve            -> nsolve (K03, guess given) / Poly.nroots (K04)\nSymbols are created real=True when case.domain == \'real\' (explicit domain assumption), plain otherwise.\n"""\nimport sys\n\nimport sympy as sp\nfrom sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations\n\ntry:\n    from . import common, safeinput\nexcept ImportError:  # executed as a script\n    import common\n    import safeinput\n\nNAME = "sympy-direct"\n_TR = standard_transformations + (convert_xor,)\n_ALLOWED = ["Integer", "Float", "Rational", "Symbol", "Function", "Mul", "Add", "Pow", "Derivative", "Eq",\n            "sin", "cos", "tan", "asin", "acos", "atan", "exp", "log", "sqrt", "Abs", "pi", "E", "I", "oo"]\n\n\ndef _ns(extra=None):\n    g = {k: getattr(sp, k) for k in _ALLOWED}\n    g["__builtins__"] = {}\n    g["abs"] = sp.Abs\n    g["ln"] = sp.log\n    g["e"] = sp.E      # Euler\'s number: SymPy\'s parser would otherwise read \'e\' as a plain symbol\n    g["i"] = sp.I\n    if extra:\n        g.update(extra)\n    return g\n\n\n_PARSE_MS = [0.0]\n\n\ndef parse(s, syms, extra=None):\n    import time as _t\n    _t0 = _t.perf_counter()\n    try:\n        return _parse(s, syms, extra)\n    finally:\n        _PARSE_MS[0] += (_t.perf_counter() - _t0) * 1000.0\n\n\ndef _parse(s, syms, extra=None):\n    safeinput.check(s)\n    loc = {k: v for k, v in syms.items() if not (extra and k in extra)}   # function names (y) must not be shadowed by Symbols\n    return parse_expr(s, local_dict=loc, global_dict=_ns(extra), transformations=_TR, evaluate=True)\n\n\ndef info():\n    return {"solver": NAME, "version": sp.__version__, "python": sys.version.split()[0],\n            "commit": None, "package": "sympy", "api_methods": ["simplify", "expand", "factor", "solveset", "solve",\n            "solve_univariate_inequality", "diff", "integrate", "limit", "Matrix.det/inv/rref", "dsolve", "apart",\n            "nsolve", "Poly.nroots"]}\n\n\ndef _syms(case, names):\n    real = case["domain"] == "real"\n    return {n: sp.Symbol(n, real=True) if real else sp.Symbol(n) for n in names}\n\n\ndef _expr_answer(e):\n    return {"kind": "expr", "text": str(e)}\n\n\ndef _fmt_set(s):\n    """Convert a SymPy set to the canonical, string-free structure."""\n    if s == sp.S.EmptySet:\n        return {"kind": "empty"}\n    if s in (sp.S.Reals, sp.S.Complexes):\n        return {"kind": "all", "domain": str(s)}\n    if isinstance(s, sp.FiniteSet):\n        return {"kind": "solutions", "values": [str(v) for v in s]}\n    if isinstance(s, sp.Interval):\n        return {"kind": "intervals", "intervals": [[str(s.start), str(s.end), bool(s.left_open), bool(s.right_open)]]}\n    if isinstance(s, sp.Union) and all(isinstance(a, sp.Interval) for a in s.args):\n        return {"kind": "intervals", "intervals": [[str(a.start), str(a.end), bool(a.left_open), bool(a.right_open)] for a in s.args]}\n    if isinstance(s, sp.ConditionSet):\n        return {"kind": "condition_set", "text": str(s)}\n    return {"kind": "other_set", "text": str(s)}\n\n\ndef solve_case(case):\n    op, pb = case["operation"], case["problem"]\n    _PARSE_MS[0] = 0.0\n    res = {"solver": NAME, "restrictions_reported": [], "steps": [], "native_flags": {},\n           "domain_assumption_passed": True, "input_overrides_used": False}\n    symnames = ["x", "y"]\n    S = _syms(case, symnames)\n    x = S["x"]\n    out = lambda **k: {**res, **k}\n\n    def run(call_desc, fn):\n        r, ms, e = common.timed(fn)\n        res["api_call"] = call_desc\n        res["time_ms"] = ms + _PARSE_MS[0]     # parse time included, as for the other two solvers\n        return r, e\n\n    if op in ("evaluate", "simplify"):\n        expr = parse(pb["expr"], S)\n        r, e = run("sympy.simplify(parse(expr))", lambda: sp.simplify(expr))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")\n    if op in ("expand", "factor"):\n        expr = parse(pb["expr"], S)\n        fn = sp.expand if op == "expand" else sp.factor\n        r, e = run("sympy.%s(parse(expr))" % op, lambda: fn(expr))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")\n    if op == "solve":\n        lhs, rhs = [parse(t, S) for t in pb["equation"].split("=", 1)]\n        dom = sp.S.Reals if case["domain"] == "real" else sp.S.Complexes\n        r, e = run("sympy.solveset(Eq(lhs, rhs), x, %s)" % dom, lambda: sp.solveset(sp.Eq(lhs, rhs), x, dom))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        ans = _fmt_set(r)\n        if ans["kind"] == "condition_set":\n            return out(outcome="unsupported", unsupported_reason="solveset returned an unevaluated ConditionSet (SymPy\'s declared \'cannot solve\')",\n                       raw_output=str(r), answer=ans, input_sent=pb["equation"], native_status="ConditionSet")\n        return out(outcome="answer", raw_output=str(r), answer=ans, input_sent=pb["equation"], native_status="ok")\n    if op == "solve_system":\n        eqs = []\n        for t in pb["equations"]:\n            l, rr = [parse(s, S) for s in t.split("=", 1)]\n            eqs.append(sp.Eq(l, rr))\n        r, e = run("sympy.solve(eqs, [x, y], dict=True)", lambda: sp.solve(eqs, [S["x"], S["y"]], dict=True))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        if not r:\n            ans = {"kind": "empty"}\n        else:\n            ans = {"kind": "system", "solutions": [{str(k): str(v) for k, v in d.items()} for d in r]}\n        return out(outcome="answer", raw_output=str(r), answer=ans, input_sent=pb["equations"], native_status="ok")\n    if op == "solve_inequality":\n        s = pb["inequality"]\n        import re\n        m = re.search(r"(<=|>=|<|>)", s)\n        if not m:\n            return out(outcome="input_incompatible", error={"type": "InequalityParseError", "message": "no supported inequality operator found"})\n        op_text = m.group(1)\n        ltxt, rtxt = s[:m.start()], s[m.end():]\n        l, rr = parse(ltxt, S), parse(rtxt, S)\n        rel = {"<": lambda: l < rr, "<=": lambda: l <= rr,\n               ">": lambda: l > rr, ">=": lambda: l >= rr}[op_text]()\n        r, e = run("sympy.solve_univariate_inequality(rel, x, relational=False)",\n                   lambda: sp.solve_univariate_inequality(rel, x, relational=False))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer=_fmt_set(r), input_sent=s, native_status="ok")\n    if op in ("diff", "partial_derivative"):\n        expr = parse(pb["expr"], S)\n        v = S[pb["var"]]\n        r, e = run("sympy.diff(expr, %s)" % pb["var"], lambda: sp.diff(expr, v))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")\n    if op == "gradient":\n        expr = parse(pb["expr"], S)\n        r, e = run("[sympy.diff(expr, v) for v in vars]", lambda: [sp.diff(expr, S[v]) for v in pb["vars"]])\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer={"kind": "vector", "items": [str(i) for i in r]},\n                   input_sent=pb["expr"], native_status="ok")\n    if op == "integrate":\n        expr = parse(pb["expr"], S)\n        definite = "lower" in pb\n        if definite:\n            lo, hi = parse(pb["lower"], S), parse(pb["upper"], S)\n            r, e = run("sympy.integrate(expr, (x, lo, hi))", lambda: sp.integrate(expr, (x, lo, hi)))\n        else:\n            r, e = run("sympy.integrate(expr, x)", lambda: sp.integrate(expr, x))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        # SymPy\'s own documented step API (rule tree); captured as-is, NOT paraphrased or enriched.\n        try:\n            from sympy.integrals.manualintegrate import integral_steps\n            tree = repr(integral_steps(expr, x))\n            res["steps"] = [{"rule": "sympy.integral_steps (raw rule tree)", "description": tree[:4000], "before": str(expr),\n                             "after": "", "notes": [], "verification": "", "constraints": []}]\n            res["native_flags"]["workings_source"] = "sympy.integrals.manualintegrate.integral_steps"\n            res["native_flags"]["workings_tied_to_answer"] = False   # integrate() and integral_steps() are separate code paths\n        except Exception as ex:   # noqa: BLE001\n            res["native_flags"]["workings_error"] = type(ex).__name__\n        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")\n    if op == "limit":\n        expr = parse(pb["expr"], S)\n        pt = parse(pb["point"], S)\n        r, e = run("sympy.limit(expr, x, point, dir=\'+-\')", lambda: sp.limit(expr, x, pt, dir="+-"))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")\n    if op in ("det", "inverse", "rref"):\n        M = sp.Matrix(pb["matrix"])\n        fn = {"det": lambda: M.det(), "inverse": lambda: M.inv(), "rref": lambda: M.rref()[0]}[op]\n        r, e = run("sympy.Matrix(...).%s" % {"det": "det()", "inverse": "inv()", "rref": "rref()[0]"}[op], fn)\n        if e: return out(outcome="error", error=common.err_dict(e))\n        if op == "det":\n            return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["matrix"], native_status="ok")\n        return out(outcome="answer", raw_output=str(r), answer={"kind": "matrix", "rows": [[str(c) for c in r.row(i)] for i in range(r.rows)]},\n                   input_sent=pb["matrix"], native_status="ok")\n    if op == "dsolve":\n        f = sp.Function(pb["func"])\n        ode = parse(pb["ode"], S, extra={pb["func"]: f})\n        r, e = run("sympy.dsolve(ode, y(x))", lambda: sp.dsolve(ode, f(x)))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer={"kind": "equation", "text": str(r)}, input_sent=pb["ode"], native_status="ok")\n    if op == "partial_fractions":\n        expr = parse(pb["expr"], S)\n        r, e = run("sympy.apart(expr, x)", lambda: sp.apart(expr, x))\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer=_expr_answer(r), input_sent=pb["expr"], native_status="ok")\n    if op == "nsolve":\n        l, rr = [parse(t, S) for t in pb["equation"].split("=", 1)]\n        f = l - rr\n        if pb.get("guess") is not None:\n            r, e = run("sympy.nsolve(lhs-rhs, x, guess, prec=30)", lambda: sp.nsolve(f, x, sp.Float(pb["guess"]), prec=30))\n            vals = [str(r)] if not e else []\n        else:\n            r, e = run("sympy.Poly(lhs-rhs, x).nroots(n=30)", lambda: sp.Poly(f, x).nroots(n=30))\n            vals = [str(v) for v in r] if not e else []\n        if e: return out(outcome="error", error=common.err_dict(e))\n        return out(outcome="answer", raw_output=str(r), answer={"kind": "numeric", "values": vals}, input_sent=pb["equation"], native_status="ok")\n    return out(outcome="unsupported", unsupported_reason="adapter has no mapping for operation %r" % op)\n\n\ndef raw_probe(p):\n    """Engineering probes (see bench/engineering.py). Probe inputs are benchmark-authored strings."""\n    kind = p["kind"]\n    x = sp.Symbol("x")\n    if kind == "solve_vs_solveset":\n        eq = sp.sympify(p["lhs"], locals={"x": x}) - sp.sympify(p["rhs"], locals={"x": x})\n        return {"outcome": "answer", "solve": str(sp.solve(eq, x)), "solveset": str(sp.solveset(eq, x, sp.S.Reals))}\n    if kind == "parse":\n        safeinput.check(p["text"])\n        r, ms, e = common.timed(lambda: parse_expr(p["text"], local_dict={"x": x}, global_dict=_ns(), transformations=_TR))\n        if e:\n            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}\n        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}\n    if kind == "eval":      # evaluate through the allow-listed adapter parse\n        r, ms, e = common.timed(lambda: sp.simplify(parse(p["text"], {"x": x})))\n        if e:\n            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}\n        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}\n    if kind == "solve_for":\n        syms = {"x": x, "y": sp.Symbol("y"), "z": sp.Symbol("z")}\n        l, rr = [parse(s, syms) for s in p["equation"].split("=", 1)]\n        v = syms[p["var"]]\n        r, ms, e = common.timed(lambda: sp.solveset(sp.Eq(l, rr), v, sp.S.Reals))\n        if e:\n            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}\n        return {"outcome": "answer", "raw_output": str(r), "answer_values": [str(i) for i in r] if isinstance(r, sp.FiniteSet) else None, "time_ms": ms}\n    if kind == "canary":    # engine\'s OWN string interface, as shipped (sympy.sympify); benign payload: math.factorial(4)\n        r, ms, e = common.timed(lambda: sp.sympify(p["text"]))\n        if e:\n            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}\n        return {"outcome": "answer", "raw_output": str(r), "time_ms": ms}\n    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}\n\n\nif __name__ == "__main__":\n    common.serve(sys.modules[__name__])\n')
W('bench/workers/smartsolver_worker.py', '"""SmartSolver adapter (package \'ed-master-smartsolver\', import name \'smartsolver\').\n\nMapping (declared before execution; one documented public method per operation family):\n  solve (C, K02, K03, K04, L01, L02, L04) -> SmartSolver.solve_equation(equation, \'x\')\n  solve_system (D01-D03, I01)             -> SmartSolver.solve_linear_system(equations, variables)\n  diff (F01-F06)                          -> SmartSolver.differentiate(expr, \'x\')\n  partial_derivative (J03)                -> SmartSolver.partial_derivative(expr, \'x\', [\'x\',\'y\'])\n  gradient (J04)                          -> SmartSolver.gradient(expr, vars)\n  integrate (G01-G06)                     -> SmartSolver.integrate(expr, \'x\'[, lower, upper])\n  limit (H01-H04)                         -> SmartSolver.limit(expr, \'x\', point)      (documented default direction \'+\')\n  rref (I04)                              -> SmartSolver.matrix_rref(matrix)\n  partial_fractions (K01)                 -> SmartSolver.partial_fractions(expr)\nOperations with NO public method (evaluate, simplify, expand, factor, solve_inequality, det, inverse, dsolve,\nnsolve) are reported as UNSUPPORTED with the introspected method list as evidence; nothing is manufactured.\nK03/K04 (numeric root finding) have no numeric API, so the generic equation solver is used (documented in cases.json).\nSmartSolver result objects are converted with str(); steps with the library\'s own step_to_dict / StepRenderer.\nNo answer is modified.\n"""\nimport re\nimport sys\n\nimport smartsolver\nfrom smartsolver import SmartSolver, StepRenderer, step_to_dict\n\ntry:\n    from . import common\nexcept ImportError:\n    import common\n\nNAME = "smartsolver"\nS = SmartSolver()\nPUBLIC = sorted(m for m in dir(SmartSolver) if not m.startswith("_") and callable(getattr(SmartSolver, m)))\n\n\ndef info():\n    import importlib.metadata as md\n    dist = (md.packages_distributions().get("smartsolver") or ["ed-master-smartsolver"])[0]\n    return {"solver": NAME, "version": md.version(dist), "package": dist,\n            "module_version": getattr(smartsolver, "__version__", None), "python": sys.version.split()[0], "commit": None,\n            "sympy_version": md.version("sympy"), "public_methods": PUBLIC}\n\n\ndef _steps(r):\n    out = []\n    for s in r.get("steps", []):\n        d = step_to_dict(s)\n        out.append({"rule": d.get("rule_applied", ""), "description": d.get("description", ""), "before": "",\n                    "after": d.get("expression", ""), "notes": [], "verification": "", "constraints": [],\n                    "substeps": d.get("substeps", [])})\n    return out\n\n\ndef _restrictions(steps):\n    out = []\n    for s in steps:\n        if s["rule"] in ("Domain", "Extraneous root") or re.search(r"reject|undefined|denominator|must be positive", s["description"], re.I):\n            out.append(s["description"])\n    return out\n\n\nUNSUPPORTED = {"evaluate", "simplify", "expand", "factor", "solve_inequality", "det", "inverse", "dsolve", "nsolve_api"}\n\n\ndef solve_case(case):\n    op, pb = case["operation"], case["problem"]\n    base = {"solver": NAME, "domain_assumption_passed": False, "input_overrides_used": False}   # SmartSolver has no domain parameter\n\n    def unsupported(why):\n        return {**base, "outcome": "unsupported", "unsupported_reason": why, "api_call": None, "steps": [], "restrictions_reported": [],\n                "native_flags": {"public_methods_checked": PUBLIC}}\n\n    call = None\n    if op == "solve":\n        call = ("solve_equation(%r, \'x\')" % pb["equation"], lambda: S.solve_equation(pb["equation"], "x"), "solutions")\n    elif op == "nsolve":\n        call = ("solve_equation(%r, \'x\')  [no numeric API; generic equation solver]" % pb["equation"],\n                lambda: S.solve_equation(pb["equation"], "x"), "solutions")\n    elif op == "solve_system":\n        call = ("solve_linear_system(%r, %r)" % (pb["equations"], pb["vars"]),\n                lambda: S.solve_linear_system(pb["equations"], pb["vars"]), "system")\n    elif op == "diff":\n        call = ("differentiate(%r, %r)" % (pb["expr"], pb["var"]), lambda: S.differentiate(pb["expr"], pb["var"]), "derivative")\n    elif op == "partial_derivative":\n        call = ("partial_derivative(%r, %r, %r)" % (pb["expr"], pb["var"], pb["vars"]),\n                lambda: S.partial_derivative(pb["expr"], pb["var"], pb["vars"]), "derivative")\n    elif op == "gradient":\n        call = ("gradient(%r, %r)" % (pb["expr"], pb["vars"]), lambda: S.gradient(pb["expr"], pb["vars"]), "gradient")\n    elif op == "integrate":\n        if "lower" in pb:\n            call = ("integrate(%r, \'x\', lower=%r, upper=%r)" % (pb["expr"], pb["lower"], pb["upper"]),\n                    lambda: S.integrate(pb["expr"], "x", lower=pb["lower"], upper=pb["upper"]), "integral")\n        else:\n            call = ("integrate(%r, \'x\')" % pb["expr"], lambda: S.integrate(pb["expr"], "x"), "integral")\n    elif op == "limit":\n        call = ("limit(%r, \'x\', point=%r)  [documented default direction \'+\']" % (pb["expr"], pb["point"]),\n                lambda: S.limit(pb["expr"], "x", point=pb["point"]), "limit")\n    elif op == "rref":\n        call = ("matrix_rref(%r)" % (pb["matrix"],), lambda: S.matrix_rref(pb["matrix"]), "matrix")\n    elif op == "partial_fractions":\n        call = ("partial_fractions(%r)" % pb["expr"], lambda: S.partial_fractions(pb["expr"]), "decomposition")\n    else:\n        return unsupported("SmartSolver has no public method for operation %r (public methods: %s)" % (op, ", ".join(PUBLIC)))\n\n    desc, fn, kind = call\n    method_name = desc.split("(")[0]\n    if not hasattr(S, method_name):          # a newer/older version may have removed or renamed the method\n        return unsupported("SmartSolver has no public method %r in this version (public methods: %s)" % (method_name, ", ".join(PUBLIC)))\n    r, ms, e = common.timed(fn)\n    res = {**base, "api_call": desc, "time_ms": ms, "input_sent": pb}\n    if e is not None:\n        return {**res, "outcome": "error", "error": common.err_dict(e), "steps": [], "restrictions_reported": []}\n    steps = _steps(r)\n    flags = {"steps_verified": r.get("steps_verified"), "steps_mode": r.get("steps_mode"),\n             "consistent": r.get("consistent"), "methods": [str(m) for m in r.get("methods", [])],\n             "result_keys": sorted(k for k in r if k != "steps")}\n    raw = {k: str(v) for k, v in r.items() if k != "steps"}\n    raw["steps_text"] = StepRenderer.to_text(r.get("steps", []))\n    res.update(outcome="answer", steps=steps, restrictions_reported=_restrictions(steps), native_flags=flags, raw_output=raw, native_status="ok")\n    if kind == "solutions":\n        vals = [str(v) for v in r.get("solutions", [])]\n        ans = {"kind": "solutions", "values": vals} if vals else {"kind": "empty"}\n        if r.get("general_solutions") is not None:\n            ans["general_solutions"] = str(r["general_solutions"])\n    elif kind == "system":\n        by = r.get("solutions_by_variable") or {}\n        if r.get("consistent") is False:\n            ans = {"kind": "empty"}\n        elif by:\n            ans = {"kind": "system", "solutions": [{k: str(v) for k, v in by.items()}]}\n        else:\n            ans = {"kind": "empty"} if not r.get("solutions") else {"kind": "other", "text": str(r.get("solutions"))}\n    elif kind == "derivative":\n        ans = {"kind": "expr", "text": str(r["derivative"])}\n    elif kind == "gradient":\n        g = r["gradient"]\n        ans = {"kind": "vector", "items": [str(i) for i in list(g)]}\n    elif kind == "integral":\n        ans = {"kind": "expr", "text": str(r["integral"])}\n    elif kind == "limit":\n        ans = {"kind": "expr", "text": str(r["limit"])}\n    elif kind == "matrix":\n        m = r["matrix"]\n        ans = {"kind": "matrix", "rows": [[str(m[i, j]) for j in range(m.cols)] for i in range(m.rows)]}\n    elif kind == "decomposition":\n        ans = {"kind": "expr", "text": str(r["decomposition"])}\n    return {**res, "answer": ans}\n\n\ndef raw_probe(p):\n    kind = p["kind"]\n    if kind == "call":\n        fn = getattr(S, p["method"])\n        r, ms, e = common.timed(lambda: fn(*p.get("args", []), **p.get("kwargs", {})))\n        if e:\n            return {"outcome": "error", "error": common.err_dict(e), "time_ms": ms}\n        return {"outcome": "answer", "raw_output": {k: str(v)[:300] for k, v in r.items() if k != "steps"}, "time_ms": ms,\n                "nsteps": len(r.get("steps", []))}\n    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}\n\n\nif __name__ == "__main__":\n    common.serve(sys.modules[__name__])\n')
W('bench/workers/stepsolver_worker.py', '"""StepSolver adapter (package \'stepsolver\', Solver().solve(<ASCII query>)).\n\nV2 mapping (declared before execution) -- StepSolver\'s documented ASCII operations:\n  evaluate/simplify -> simplify(e)    expand -> expand(e)     factor -> factor(e)\n  solve             -> solve(eq, x)\n  nsolve with guess -> nsolve(eq, x, guess) (direct API probe; no adapter-side symbolic fallback)\n  nsolve without guess (K04) -> solve(eq, x) to retain the original high-degree-polynomial fallback test\n  solve_system      -> solve([eq1, eq2], [x, y])    solve_inequality -> solve_inequality(ineq, x)\n  diff/partial      -> diff(e, x)     gradient -> TWO native diff(e, v) calls (no gradient op; composition is\n                       definitional: gradient == vector of partials; flagged composed_from)\n  integrate         -> integrate(e, x) / integrate(e, x, a, b)      limit -> limit(e, x, p)\n  det/inverse/rref  -> det(matrix(..)) / inverse(matrix(..)) / rref(matrix(..))\n  dsolve            -> dsolve(<ode in StepSolver syntax from cases.json adapter_inputs>, y(x))\n  partial_fractions -> apart(e, x)\nSerialisation: answers are read from StepSolver\'s typed MathValue tree with its own format_expression(); steps from\nSolutionStep fields (rule, explanation, before, after, notes, verification, introduced_constraints).  Nothing is\nrewritten.  Parse errors are \'input_incompatible\'; UnsolvedResult is \'unsupported\' (the solver\'s own declaration).\n"""\nimport sys\n\nimport stepsolver\nfrom stepsolver import (DivergentResult, ExactResult, MappingValue, MatrixValue, NoSolutionValue, ScalarValue,\n                        SequenceValue, BooleanValue, Solver, UndefinedResult, UnsolvedResult, format_ascii,\n                        format_expression, ParseError, StepSolverError)\n\ntry:\n    from . import common\nexcept ImportError:\n    import common\n\nNAME = "stepsolver"\nSOLVER = Solver()\n\n\ndef info():\n    import importlib.metadata as md\n    dist = (md.packages_distributions().get("stepsolver") or ["stepsolver"])[0]\n    return {"solver": NAME, "version": md.version(dist), "package": dist, "python": sys.version.split()[0],\n            "commit": None, "sympy_version": md.version("sympy"), "operations": [o.value for o in stepsolver.Operation]}\n\n\ndef _mat(rows):\n    return "matrix([" + ", ".join("[" + ", ".join(str(c) for c in r) + "]" for r in rows) + "])"\n\n\ndef _ln(q, on):\n    """Optional ln-to-log rewrite for experiments only. V2 deliberately disables it in the scored path so native parsing is tested."""\n    return q.replace("ln(", "log(") if on else q\n\n\ndef build_queries(case, normalise_ln=False):\n    qs, ov = _build(case)\n    return [_ln(q, normalise_ln) for q in qs] if qs else qs, ov\n\n\ndef _build(case):\n    op, pb = case["operation"], case["problem"]\n    ov = case.get("adapter_inputs", {}).get("stepsolver")\n    if ov:\n        return [ov], True\n    if op in ("evaluate", "simplify"):\n        return ["simplify(%s)" % pb["expr"]], False\n    if op in ("expand", "factor"):\n        return ["%s(%s)" % (op, pb["expr"])], False\n    if op == "solve":\n        return ["solve(%s, %s)" % (pb["equation"], pb.get("var", "x"))], False\n    if op == "nsolve":\n        var = pb.get("var", "x")\n        guess = pb.get("guess")\n        if guess is not None:\n            # This is a native numerical API test, not a symbolic-solve fallback.\n            return ["nsolve(%s, %s, %s)" % (pb["equation"], var, guess)], False\n        # K04 intentionally has no initial guess: preserve it as a polynomial solver/fallback probe.\n        return ["solve(%s, %s)" % (pb["equation"], var)], False\n    if op == "solve_system":\n        return ["solve([%s], [%s])" % (", ".join(pb["equations"]), ", ".join(pb["vars"]))], False\n    if op == "solve_inequality":\n        return ["solve_inequality(%s, x)" % pb["inequality"]], False\n    if op in ("diff", "partial_derivative"):\n        return ["diff(%s, %s)" % (pb["expr"], pb["var"])], False\n    if op == "gradient":\n        return ["diff(%s, %s)" % (pb["expr"], v) for v in pb["vars"]], False\n    if op == "integrate":\n        if "lower" in pb:\n            return ["integrate(%s, x, %s, %s)" % (pb["expr"], pb["lower"], pb["upper"])], False\n        return ["integrate(%s, x)" % pb["expr"]], False\n    if op == "limit":\n        return ["limit(%s, x, %s)" % (pb["expr"], pb["point"])], False\n    if op in ("det", "inverse", "rref"):\n        return ["%s(%s)" % (op, _mat(pb["matrix"]))], False\n    if op == "partial_fractions":\n        return ["apart(%s, x)" % pb["expr"]], False\n    return None, False\n\n\ndef _val(v, op):\n    """Typed MathValue -> canonical answer dict (no string re-interpretation)."""\n    if isinstance(v, NoSolutionValue):\n        return {"kind": "empty"}\n    if isinstance(v, BooleanValue):\n        return {"kind": "bool", "value": v.value}\n    if isinstance(v, MatrixValue):\n        return {"kind": "matrix", "rows": [[format_expression(c) for c in r] for r in v.rows]}\n    if isinstance(v, ScalarValue):\n        return {"kind": "expr", "text": format_expression(v.expression)}\n    if isinstance(v, MappingValue):\n        return {"kind": "mapping", "entries": {format_expression(e.key): _val(e.value, op) for e in v.entries}}\n    if isinstance(v, SequenceValue):\n        return {"kind": "sequence", "items": [_val(i, op) for i in v.items]}\n    return {"kind": "unknown", "text": repr(v)[:300]}\n\n\ndef _canon(av, op):\n    """Flatten typed values into the benchmark\'s canonical kinds without inventing a numerical result."""\n    if op == "nsolve" and av["kind"] == "expr":\n        return {"kind": "numeric", "values": [av["text"]]}\n    if op == "nsolve" and av["kind"] == "sequence":\n        vals = []\n        for item in av["items"]:\n            if item["kind"] == "expr":\n                vals.append(item["text"])\n            elif item["kind"] == "mapping" and len(item["entries"]) == 1:\n                value = list(item["entries"].values())[0]\n                if value.get("kind") == "expr":\n                    vals.append(value["text"])\n                else:\n                    return {"kind": "other", "detail": av}\n            else:\n                return {"kind": "other", "detail": av}\n        return {"kind": "numeric", "values": vals} if vals else {"kind": "empty"}\n    if op == "solve" and av["kind"] == "sequence":\n        vals = []\n        for it in av["items"]:\n            if it["kind"] == "mapping" and len(it["entries"]) == 1:\n                e = list(it["entries"].values())[0]\n                vals.append(e.get("text", str(e)))\n            else:\n                return {"kind": "other", "detail": av}\n        return {"kind": "solutions", "values": vals} if vals else {"kind": "empty"}\n    if op == "solve_system" and av["kind"] == "sequence":\n        sols = []\n        for it in av["items"]:\n            if it["kind"] == "mapping":\n                sols.append({k: v.get("text", str(v)) for k, v in it["entries"].items()})\n            else:\n                return {"kind": "other", "detail": av}\n        return {"kind": "system", "solutions": sols} if sols else {"kind": "empty"}\n    if op == "rref" and av["kind"] == "sequence" and av["items"] and av["items"][0]["kind"] == "matrix":\n        return av["items"][0]      # (matrix, pivots) -> matrix ; pivots dropped (documented)\n    if op == "solve_inequality" and av["kind"] == "expr":\n        return {"kind": "relation", "text": av["text"]}\n    if op == "dsolve" and av["kind"] == "expr":\n        return {"kind": "equation", "text": av["text"]}\n    return av\n\n\ndef _steps(res):\n    out = []\n    for s in res.steps:\n        out.append({"rule": s.rule, "description": s.explanation, "before": format_expression(s.before),\n                    "after": format_expression(s.after),\n                    "notes": [[n.label, format_expression(n.expression)] for n in s.notes],\n                    "verification": "%s: %s" % (s.verification.method.value, s.verification.detail),\n                    "constraints": [[c.explanation, format_expression(c.expression)] for c in s.introduced_constraints]})\n    return out\n\n\ndef solve_case(case):\n    queries, overridden = build_queries(case, normalise_ln=False)\n    # V2 deliberately sends ln(...) exactly as written. Native parser support is under test;\n    # adapter-side normalisation would hide a parser regression and inflate/obscure adaptation metrics.\n    base = {"solver": NAME, "domain_assumption_passed": False, "input_overrides_used": bool(overridden)}\n    if queries is None:\n        return {**base, "outcome": "unsupported", "unsupported_reason": "adapter has no mapping for operation %r" % case["operation"],\n                "steps": [], "restrictions_reported": []}\n    op = case["operation"]\n    results, total_ms, raws = [], 0.0, []\n    for q in queries:\n        r, ms, e = common.timed(lambda q=q: SOLVER.solve(q))\n        total_ms += ms\n        if e is not None:\n            kind = "input_incompatible" if isinstance(e, ParseError) else "error"\n            err = common.err_dict(e)\n            err["contract_exception"] = isinstance(e, StepSolverError)\n            return {**base, "outcome": kind, "error": err, "api_call": "Solver().solve(%r)" % q, "input_sent": queries, "time_ms": total_ms,\n                    "steps": [], "restrictions_reported": [], "raw_output": raws or None}\n        results.append(r)\n        raws.append(format_ascii(r))\n    steps, restr, answers = [], [], []\n    native = {"result_types": [type(r).__name__ for r in results], "composed_from": queries if len(queries) > 1 else None}\n    for q, r in zip(queries, results):\n        st = _steps(r)\n        steps += st\n        for s in st:\n            restr += [c[0] + " | " + c[1] for c in s["constraints"]]\n        if isinstance(r, UndefinedResult):\n            restr.append("undefined: " + r.reason)\n        native.setdefault("opaque_steps", 0)\n        native["opaque_steps"] += sum(1 for s in st if s["rule"].strip().lower() == "compute exact result")\n    res = {**base, "api_call": " ; ".join("Solver().solve(%r)" % q for q in queries), "input_sent": queries, "time_ms": total_ms,\n           "steps": steps, "restrictions_reported": restr, "native_flags": native, "raw_output": raws if len(raws) > 1 else raws[0]}\n    r0 = results[0]\n    for r in results:\n        if isinstance(r, UnsolvedResult):\n            return {**res, "outcome": "unsupported", "unsupported_reason": "UnsolvedResult: " + r.reason[:300], "native_status": "UnsolvedResult"}\n    if len(results) > 1 and all(isinstance(r, ExactResult) for r in results):   # gradient composition\n        items = [_val(r.value, op) for r in results]\n        return {**res, "outcome": "answer", "native_status": "ExactResult", "answer": {"kind": "vector", "items": [i.get("text", str(i)) for i in items]}}\n    if isinstance(r0, ExactResult):\n        return {**res, "outcome": "answer", "native_status": "ExactResult", "answer": _canon(_val(r0.value, op), op)}\n    if isinstance(r0, UndefinedResult):\n        return {**res, "outcome": "answer", "native_status": "UndefinedResult", "answer": {"kind": "undefined", "reason": r0.reason}}\n    if isinstance(r0, DivergentResult):\n        return {**res, "outcome": "answer", "native_status": "DivergentResult", "answer": {"kind": "divergent", "reason": r0.reason, "divergence": r0.kind.value}}\n    return {**res, "outcome": "error", "error": {"type": "UnexpectedResultType", "message": type(r0).__name__}}\n\n\ndef raw_probe(p):\n    kind = p["kind"]\n    if kind == "query":\n        r, ms, e = common.timed(lambda: SOLVER.solve(p["query"]))\n        if e:\n            return {"outcome": "error", "error": {**common.err_dict(e), "contract_exception": isinstance(e, StepSolverError)}, "time_ms": ms}\n        return {"outcome": "answer", "native_status": type(r).__name__, "raw_output": format_ascii(r), "time_ms": ms,\n                "nsteps": len(r.steps)}\n    return {"outcome": "error", "error": {"type": "UnknownProbe", "message": kind}}\n\n\nif __name__ == "__main__":\n    common.serve(sys.modules[__name__])\n')
W('bench/v2_stepsolver_diagnostics.py', '#!/usr/bin/env python3\n"""Direct StepSolver probes for printer structure, native ln, unknown functions and undefined divisions.\n\nThese are deliberately outside the cross-engine scored dataset because some are negative tests\nand others check exact display structure, not just mathematical equivalence. They still produce\nmachine-readable evidence and a concise pass/fail report.\n"""\nimport argparse\nimport json\nimport pathlib\nimport re\nimport sys\nimport traceback\n\nROOT = pathlib.Path(__file__).resolve().parent.parent\n\ndef safe_format(fn, value):\n    try:\n        return str(fn(value))\n    except Exception as e:\n        return "<format-error %s: %s>" % (type(e).__name__, str(e)[:120])\n\ndef normalized(s):\n    s = str(s).replace("**", "^")\n    s = s.replace("\\\\left", "").replace("\\\\right", "")\n    s = re.sub(r"\\\\(?:,|;|!|quad|qquad)", "", s)\n    s = re.sub(r"\\s+", "", s)\n    return s\n\ndef expr_from_result(result, format_expression):\n    value = getattr(result, "value", None)\n    if value is None:\n        return None\n    expr = getattr(value, "expression", value)\n    return safe_format(format_expression, expr)\n\ndef call_result(solver, query):\n    try:\n        value = solver.solve(query)\n        return value, None\n    except BaseException as e:  # diagnostics must record an exception instead of stopping all probes\n        if isinstance(e, (KeyboardInterrupt, SystemExit)):\n            raise\n        return None, {"type": type(e).__name__, "message": str(e)[:400]}\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", default="results/stepsolver_v2_diagnostics.json")\n    args = ap.parse_args()\n    out_path = pathlib.Path(args.out)\n    if not out_path.is_absolute():\n        out_path = ROOT / out_path\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    report = {"schema_version": 1, "purpose": "generalisation diagnostics outside the scored case set",\n              "printer_probes": [], "parser_probes": [], "undefined_probe": {}, "import_error": None}\n    try:\n        from stepsolver import Solver, ExactResult, UndefinedResult, UnsolvedResult, ParseError, format_expression, format_ascii\n        solver = Solver()\n    except Exception as e:\n        report["import_error"] = {"type": type(e).__name__, "message": str(e), "traceback": traceback.format_exc()[-1800:]}\n        out_path.write_text(json.dumps(report, indent=2) + "\\n")\n        print("StepSolver diagnostics skipped: import failed (%s: %s)" % (type(e).__name__, str(e)[:180]))\n        return\n\n    printer_cases = [\n        {"id": "printer_nested_numeric_power", "query": "simplify((2^3)^2)",\n         "good": [r"\\(2\\^3\\)\\^2", r"\\(\\(?2\\^\\{?3\\}?\\)?\\)\\^\\{?2\\}?"],\n         "bad": [r"2\\^3\\^2"]},\n        {"id": "printer_negative_numeric_base", "query": "simplify((-3)^2)",\n         "good": [r"\\(-3\\)\\^\\{?2\\}?"], "bad": [r"-3\\^\\{?2\\}?"]},\n        {"id": "printer_nested_symbolic_power", "query": "simplify((x^2)^3)",\n         "good": [r"\\(x\\^\\{?2\\}?\\)\\^\\{?3\\}?"], "bad": [r"x\\^2\\^3"]},\n        {"id": "printer_negative_symbolic_base", "query": "simplify((-x)^2)",\n         "good": [r"\\(-x\\)\\^\\{?2\\}?"], "bad": [r"-x\\^\\{?2\\}?"]},\n        {"id": "printer_parenthesized_sum_base", "query": "simplify((x + 1)^2)",\n         "good": [r"\\(x\\+1\\)\\^\\{?2\\}?", r"\\(1\\+x\\)\\^\\{?2\\}?"], "bad": [r"x\\+1\\^\\{?2\\}?"]},\n    ]\n    for spec in printer_cases:\n        result, error = call_result(solver, spec["query"])\n        steps = []\n        texts = []\n        if result is not None:\n            for st in getattr(result, "steps", []) or []:\n                before = safe_format(format_expression, getattr(st, "before", ""))\n                after = safe_format(format_expression, getattr(st, "after", ""))\n                desc = str(getattr(st, "explanation", ""))\n                rule = str(getattr(st, "rule", ""))\n                steps.append({"rule": rule, "description": desc[:250], "before": before, "after": after})\n                texts.extend([before, after])\n            texts.append(safe_format(format_ascii, result))\n        norm_text = " | ".join(normalized(t) for t in texts)\n        good = any(re.search(p, norm_text) for p in spec["good"])\n        bad = any(re.search(p, norm_text) for p in spec["bad"])\n        # A fully rendered bad pattern is a failure. When no relevant nested expression is\n        # preserved in any step, call the probe inconclusive instead of claiming a pass.\n        status = "fail" if bad else ("pass" if good else "inconclusive")\n        report["printer_probes"].append({"id": spec["id"], "query": spec["query"], "status": status,\n            "result_type": type(result).__name__ if result is not None else None, "error": error,\n            "formatted_evidence": texts, "steps": steps})\n\n    # `ln` is a required native alias. Differentiate it so an opaque echoed function cannot pass.\n    q = "diff(ln(x), x)"\n    result, error = call_result(solver, q)\n    answer_text = expr_from_result(result, format_expression) if result is not None else None\n    if answer_text is None and result is not None:\n        answer_text = safe_format(format_ascii, result)\n    try:\n        import sympy as sp\n        x = sp.Symbol("x", real=True)\n        got = sp.sympify(answer_text.replace("^", "**"), locals={"x": x, "ln": sp.log, "log": sp.log})\n        ln_ok = isinstance(result, ExactResult) and sp.simplify(got - 1/x) == 0\n        ln_detail = "derivative simplifies to 1/x" if ln_ok else "expected derivative 1/x, got %s" % answer_text\n    except Exception as e:\n        ln_ok = False\n        ln_detail = "could not independently parse derivative output: %s: %s" % (type(e).__name__, str(e)[:160])\n    report["parser_probes"].append({"id": "native_ln_alias", "query": q, "status": "pass" if ln_ok else "fail",\n        "result_type": type(result).__name__ if result is not None else None, "answer": answer_text, "error": error, "detail": ln_detail})\n\n    # Unknown names must be rejected/declined; never returned as a successful exact/verified result.\n    for name, q in [("frobnicate", "simplify(frobnicate(x))"),\n                    ("mystery_function", "simplify(mystery_function(x) + 1)"),\n                    ("unknown_function_with_derivative", "diff(unknown_function_927(x), x)")]:\n        result, error = call_result(solver, q)\n        if error is not None:\n            ok = error["type"] in ("ParseError", "StepSolverError", "ValueError", "TypeError", "NotImplementedError")\n            outcome = "controlled_exception" if ok else "exception_review"\n        elif isinstance(result, (UnsolvedResult, ParseError)):\n            ok, outcome = True, "rejected_or_declined"\n        elif isinstance(result, UndefinedResult):\n            ok, outcome = True, "explicit_non_answer"\n        elif isinstance(result, ExactResult):\n            ok, outcome = False, "dangerous_exact_result_for_unknown_function"\n        else:\n            ok, outcome = False, "unexpected_result_type"\n        report["parser_probes"].append({"id": "unknown_function_" + name, "query": q,\n            "status": "pass" if ok else "fail", "result_type": type(result).__name__ if result is not None else None,\n            "error": error, "outcome": outcome,\n            "formatted_result": safe_format(format_ascii, result) if result is not None else None})\n\n    q = "simplify(1/(x-x))"\n    result, error = call_result(solver, q)\n    if error is not None:\n        explicit = error["type"] in ("ZeroDivisionError", "DivisionByZero", "ValueError", "StepSolverError")\n        status = "pass" if explicit else "review"\n        kind = "controlled_exception"\n        reason = error["type"] + ": " + error["message"]\n    elif isinstance(result, UndefinedResult):\n        status, kind, reason = "pass", "UndefinedResult", str(getattr(result, "reason", ""))\n    elif isinstance(result, UnsolvedResult):\n        status, kind, reason = "review", "UnsolvedResult", str(getattr(result, "reason", ""))\n    elif isinstance(result, ExactResult):\n        txt = safe_format(format_ascii, result).lower()\n        status = "fail" if any(t in txt for t in ("zoo", "complex infinity", "verified")) else "review"\n        kind, reason = "ExactResult", txt[:300]\n    else:\n        status, kind, reason = "review", type(result).__name__ if result is not None else "exception", "No clear undefined signal"\n    report["undefined_probe"] = {"id": "undefined_zero_denominator", "query": q, "status": status,\n        "result_type": type(result).__name__ if result is not None else None, "error": error,\n        "outcome": kind, "detail": reason, "formatted_result": safe_format(format_ascii, result) if result is not None else None}\n\n    report["summary"] = {\n        "printer_pass": sum(p["status"] == "pass" for p in report["printer_probes"]),\n        "printer_fail": sum(p["status"] == "fail" for p in report["printer_probes"]),\n        "printer_inconclusive": sum(p["status"] == "inconclusive" for p in report["printer_probes"]),\n        "parser_pass": sum(p["status"] == "pass" for p in report["parser_probes"]),\n        "parser_fail": sum(p["status"] == "fail" for p in report["parser_probes"]),\n        "undefined_status": report["undefined_probe"]["status"],\n        "status_note": "An inconclusive printer probe is not a pass; inspect its captured step strings."\n    }\n    out_path.write_text(json.dumps(report, indent=2, default=str) + "\\n")\n    print("StepSolver V2 diagnostics:", json.dumps(report["summary"], sort_keys=True))\n    print("Detailed results:", out_path)\n\nif __name__ == "__main__":\n    main()\n')
print("ok: wrote 8 V2 benchmark source files")


ok: wrote 8 V2 benchmark source files


In [5]:
W('bench/runner.py', '#!/usr/bin/env python3\n"""Run the dataset against each solver (each in its own venv + worker process). Raw replies are preserved verbatim.\n\n  python -m bench.runner [--solver sympy-direct|smartsolver|stepsolver ...] [--category A] [--case C05]\n                         [--repeats 5] [--timeout 20] [--no-timing] [--out results/raw]\n\nProtocol per solver:\n  1. cold-start trials  : N fresh processes; time to ready (spawn->first reply to \'ping\') and first solve of C01.\n  2. functional pass    : each selected case once, in dataset order, per-case timeout (process killed + restarted on timeout).\n  3. warm repeats       : for each case that produced an answer: `repeats` extra runs, each preceded by a SymPy cache clear\n                          (untimed), same process -> warm-process / cold-cache latency; canonical outputs hashed (repeatability).\nSame cases, same timeout, same machine, same protocol for every solver. No solver gets special pre-processing.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport pathlib\nimport select\nimport statistics\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nROOT = pathlib.Path(__file__).resolve().parent.parent\nWORKERS = {\n    "sympy-direct": ("sympy", ROOT / "bench/workers/sympy_worker.py"),\n    "smartsolver": ("smartsolver", ROOT / "bench/workers/smartsolver_worker.py"),\n    "stepsolver": ("stepsolver", ROOT / "bench/workers/stepsolver_worker.py"),\n}\n\n\nclass Worker:\n    def __init__(self, name):\n        env_name, script = WORKERS[name]\n        self.name = name\n        self.py = str(ROOT / "envs" / env_name / "bin" / "python")\n        self.script = str(script)\n        self.proc = None\n        self.spawn_to_ready_ms = None\n        self.start()\n\n    def start(self):\n        self.cwd = tempfile.mkdtemp(prefix="bench_wd_")       # neutral working directory (no repo files visible)\n        env = {"PATH": "/usr/bin:/bin", "PYTHONHASHSEED": "0", "PYTHONNOUSERSITE": "1", "PYTHONDONTWRITEBYTECODE": "1",\n               "HOME": self.cwd, "LC_ALL": "C.UTF-8"}\n        t = time.perf_counter()\n        self.proc = subprocess.Popen([self.py, "-s", self.script], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\n                                     stderr=subprocess.PIPE, text=True, bufsize=1, cwd=self.cwd, env=env)\n        r = self.call({"cmd": "ping"}, timeout=120)\n        self.spawn_to_ready_ms = (time.perf_counter() - t) * 1000.0\n        return r\n\n    def call(self, req, timeout):\n        """Returns (reply|None, status) where status in ok|timeout|crash."""\n        try:\n            self.proc.stdin.write(json.dumps(req) + "\\n")\n            self.proc.stdin.flush()\n        except BrokenPipeError:\n            return None, "crash"\n        end = time.time() + timeout\n        buf = ""\n        while time.time() < end:\n            rem = max(0.0, end - time.time())\n            r, _, _ = select.select([self.proc.stdout], [], [], min(rem, 0.5))\n            if r:\n                line = self.proc.stdout.readline()\n                if not line:\n                    return None, "crash"\n                try:\n                    return json.loads(line), "ok"\n                except json.JSONDecodeError:\n                    buf += line\n            elif self.proc.poll() is not None:\n                return None, "crash"\n        return None, "timeout"\n\n    def kill(self):\n        if self.proc and self.proc.poll() is None:\n            self.proc.kill()\n            try:\n                self.proc.wait(5)\n            except Exception:   # noqa: BLE001\n                pass\n\n    def stderr_tail(self):\n        try:\n            if self.proc.poll() is not None:\n                return self.proc.stderr.read()[-1500:]\n        except Exception:   # noqa: BLE001\n            pass\n        return ""\n\n    def close(self):\n        try:\n            self.proc.stdin.write(json.dumps({"cmd": "shutdown"}) + "\\n")\n            self.proc.stdin.flush()\n            self.proc.wait(5)\n        except Exception:   # noqa: BLE001\n            self.kill()\n\n\ndef canon_hash(reply):\n    """Hash of the semantically relevant output (answer, steps, outcome) -- excludes timings/memory."""\n    keep = {k: reply.get(k) for k in ("outcome", "answer", "steps", "restrictions_reported", "unsupported_reason", "raw_output")}\n    err = reply.get("error")\n    if err:\n        keep["error"] = {k: err.get(k) for k in ("type", "message")}\n    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]\n\n\ndef run_solver(name, cases, repeats, timeout, timing, cold_trials=5):\n    out = {"solver": name, "cold_start": [], "rows": {}, "info": None, "maxrss_kb": 0, "timeouts": 0, "crashes": 0}\n    w = Worker(name)\n    reply, _ = w.call({"cmd": "info"}, 60)\n    out["info"] = reply\n    c01 = next(c for c in json.load(open(ROOT / "data/cases.json"))["cases"] if c["id"] == "C01")\n    if timing:\n        for _ in range(cold_trials):                                  # cold-start trials: fresh process each time\n            cw = Worker(name)\n            t = time.perf_counter()\n            r, st = cw.call({"cmd": "solve", "case": c01, "req_id": "cold"}, timeout)\n            out["cold_start"].append({"spawn_to_ready_ms": cw.spawn_to_ready_ms, "first_solve_wall_ms": (time.perf_counter() - t) * 1000.0,\n                                      "first_solve_inner_ms": (r or {}).get("time_ms"), "status": st})\n            cw.close()\n    for c in cases:\n        t = time.perf_counter()\n        r, st = w.call({"cmd": "solve", "case": c, "req_id": c["id"]}, timeout)\n        wall = (time.perf_counter() - t) * 1000.0\n        row = {"case_id": c["id"], "wall_ms": wall, "transport": st}\n        if st == "ok":\n            row["reply"] = r\n            row["hash"] = canon_hash(r)\n            out["maxrss_kb"] = max(out["maxrss_kb"], r.get("maxrss_kb", 0))\n        else:\n            row["reply"] = {"outcome": "timeout" if st == "timeout" else "infra_error", "solver": name,\n                            "error": {"type": "WorkerTimeout" if st == "timeout" else "WorkerCrash",\n                                      "message": "no reply within %ss" % timeout if st == "timeout" else w.stderr_tail()}}\n            out["timeouts" if st == "timeout" else "crashes"] += 1\n            w.kill()\n            w = Worker(name)      # restart for the remaining cases (state of a killed process is not trusted)\n        out["rows"][c["id"]] = row\n    if timing and repeats:\n        for c in cases:\n            row = out["rows"][c["id"]]\n            if row["transport"] != "ok" or row["reply"].get("outcome") != "answer":\n                continue\n            times, hashes = [], []\n            for _ in range(repeats):\n                w.call({"cmd": "clear_cache"}, 30)\n                r, st = w.call({"cmd": "solve", "case": c, "req_id": c["id"] + "-rep"}, timeout)\n                if st != "ok":\n                    row.setdefault("repeat_failures", []).append(st)\n                    if st != "ok":\n                        w.kill(); w = Worker(name)\n                    continue\n                times.append(r.get("time_ms"))\n                hashes.append(canon_hash(r))\n            row["warm_times_ms"] = times\n            row["repeat_hashes"] = hashes\n            row["repeatable"] = len(set(hashes + [row["hash"]])) == 1 if hashes else None\n    w.close()\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--solver", nargs="*", default=list(WORKERS))\n    ap.add_argument("--category")\n    ap.add_argument("--case", nargs="*")\n    ap.add_argument("--repeats", type=int, default=5)\n    ap.add_argument("--timeout", type=float, default=20.0)\n    ap.add_argument("--no-timing", action="store_true")\n    ap.add_argument("--out", default=str(ROOT / "results/raw"))\n    a = ap.parse_args()\n    data = json.load(open(ROOT / "data/cases.json"))\n    cases = data["cases"]\n    if a.category:\n        cases = [c for c in cases if c["category"] == a.category.upper()]\n    if a.case:\n        cases = [c for c in cases if c["id"] in set(a.case)]\n    pathlib.Path(a.out).mkdir(parents=True, exist_ok=True)\n    print("cases loaded: %d (dataset total %d)" % (len(cases), data["count"]))\n    for s in a.solver:\n        t = time.time()\n        res = run_solver(s, cases, a.repeats, a.timeout, not a.no_timing)\n        res["protocol"] = {"repeats": a.repeats, "timeout_s": a.timeout, "cases_run": [c["id"] for c in cases],\n                           "cpu_count": os.cpu_count(), "started_unix": t}\n        (pathlib.Path(a.out) / ("%s.json" % s)).write_text(json.dumps(res, indent=1, default=str))\n        oc = {}\n        for r in res["rows"].values():\n            o = r["reply"].get("outcome")\n            oc[o] = oc.get(o, 0) + 1\n        print("%-13s %5.1fs  outcomes=%s  maxrss=%.0fMB timeouts=%d crashes=%d" % (s, time.time() - t, oc, res["maxrss_kb"] / 1024, res["timeouts"], res["crashes"]))\n\n\nif __name__ == "__main__":\n    main()\n')
W('bench/judge.py', '"""Mathematical judge (runs in the controller env: SymPy used ONLY as a checker, never to produce a solver\'s answer).\n\nRules (declared in cases.json \'comparison\' + here, applied identically to every solver):\n  value        scalar equality: symbolic simplify(a-b)==0 OR high-precision numeric |a-b|<=tol  (numeric alone is flagged)\n  expr_equiv   equivalence as functions: symbolic zero OR agreement at >=3 exact sample points (50-digit) [flagged if numeric only]\n               + form check (below).  Complex/NaN at a real sample point is a MISMATCH (catches log(x) vs ln|x|).\n  solution_set exact set equality over the case\'s declared domain (real|complex); multiplicity not part of correctness\n  system_solution / inequality_set / matrix / gradient / ode_family / numeric_roots / antiderivative / undefined: see functions.\nForm rules (uniform): simplify/evaluate -> structure of the RAW answer text must not be more than 1 operation more complex than the\n  reference\'s raw structure (rejects echoing the input); named checks: expanded, factored, simplified_radical, no_fraction,\n  reduced_fraction, partial_fractions.  The generic simplification-form rule was added after the first functional pass showed one solver\n  echoing its input for E05; it is applied to all solvers, and results with/without form rules are both reported (sensitivity).\nDomain rule: cases with required_restrictions need the solver to REPORT each restriction (constraint list or step text);\n  strict_correct = value_correct AND form_ok AND domain_ok.  value-only correctness is reported alongside.\n"""\nimport json\nimport re\n\nimport sympy as sp\nfrom sympy.parsing.sympy_parser import convert_xor, parse_expr, standard_transformations\n\ntry:\n    from .workers import safeinput\nexcept ImportError:\n    from workers import safeinput\n\nTR = standard_transformations + (convert_xor,)\nX, Y = sp.Symbol("x"), sp.Symbol("y")\nXR, YR = sp.Symbol("x", real=True), sp.Symbol("y", real=True)\n_ALLOWED = ["Integer", "Float", "Rational", "Symbol", "Function", "Mul", "Add", "Pow", "Derivative", "Eq", "sin", "cos", "tan", "exp", "log",\n            "sqrt", "Abs", "atan", "asin", "acos", "pi", "E", "I", "oo", "zoo", "CRootOf", "Matrix", "Or", "And", "Not"]\n\n\ndef ns(extra=None):\n    g = {k: getattr(sp, k) for k in _ALLOWED}\n    g.update({"__builtins__": {}, "abs": sp.Abs, "ln": sp.log, "e": sp.E, "i": sp.I, "C": sp.Symbol("C")})\n    if extra:\n        g.update(extra)\n    return g\n\n\ndef P(text, evaluate=True, extra=None, syms=None):\n    """Restricted parse of solver-produced text (allow-list; no builtins; no attribute access)."""\n    safeinput.check(text.replace("\\n", " "))\n    loc = dict(syms or {})\n    return parse_expr(text, local_dict=loc, global_dict=ns(extra), transformations=TR, evaluate=evaluate)\n\n\ndef N(e, n=50):\n    return sp.N(e, n)\n\n\ndef ops_raw(text):\n    try:\n        return int(sp.count_ops(P(text, evaluate=False)))\n    except Exception:   # noqa: BLE001\n        return 10 ** 6\n\n\ndef strip_constant(e):\n    """Remove an additive integration constant C/C1 (recorded)."""\n    consts = [s for s in e.free_symbols if re.fullmatch(r"C\\d*", s.name)]\n    if consts:\n        return e.subs({c: 0 for c in consts}), True\n    return e, False\n\n\ndef sample_pts(case, var_names):\n    sp_ = case.get("sample_points")\n    if sp_:\n        cols = [[sp.Rational(v) for v in sp_[n]] for n in var_names if n in sp_]\n        if len(cols) == len(var_names):\n            k = min(len(c) for c in cols)\n            return [dict(zip(var_names, [c[i] for c in cols])) for i in range(k)]\n    base = [sp.Rational(-37, 10), sp.Rational(-3, 5), sp.Rational(9, 10), sp.Rational(23, 10), sp.Rational(51, 10)]\n    ycol = [sp.Rational(21, 10), sp.Rational(-7, 5), sp.Rational(1, 3), sp.Rational(17, 10), sp.Rational(-5, 2)]\n    return [dict(zip(var_names, [base[i], ycol[i]][: len(var_names)])) for i in range(5)]\n\n\ndef equiv(a, b, case, tol=1e-9):\n    """Returns (ok, proof) with proof in {\'symbolic\',\'numeric\',\'mismatch\'}."""\n    names = sorted({s.name for s in (a.free_symbols | b.free_symbols) if s.name in ("x", "y")})\n    sub = lambda e, pt: e.subs({sp.Symbol(k): v for k, v in pt.items()}).subs({sp.Symbol(k, real=True): v for k, v in pt.items()})\n    pts = sample_pts(case, names) if names else [{}]\n    numeric_ok = True\n    for pt in pts:\n        try:\n            av, bv = N(sub(a, pt)), N(sub(b, pt))\n        except Exception:   # noqa: BLE001\n            return False, "mismatch"\n        if not (av.is_number and bv.is_number) or av.has(sp.nan, sp.zoo, sp.oo, -sp.oo) or bv.has(sp.nan, sp.zoo, sp.oo, -sp.oo):\n            return False, "mismatch"\n        if abs(sp.im(av)) > tol or abs(sp.im(bv)) > tol:          # real input must give real output (cases are declared real)\n            if case["domain"] == "real":\n                return False, "mismatch"\n        if abs(av - bv) > tol * max(1, abs(bv)):\n            return False, "mismatch"\n    try:\n        if sp.simplify(a - b) == 0:\n            return True, "symbolic"\n    except Exception:   # noqa: BLE001\n        pass\n    return numeric_ok, "numeric"\n\n\ndef numeq(a, b, tol):\n    try:\n        return abs(N(a) - N(b)) <= max(tol["abs"], tol["rel"] * abs(N(b)))\n    except Exception:   # noqa: BLE001\n        return False\n\n\n# --------------------------------------------------------------------------- form checks\ndef form_check(name, text, expected_text, case):\n    """Returns (ok|None, note)."""\n    try:\n        if name is None and case["operation"] in ("simplify", "evaluate"):\n            name = "generic_ops"\n        if name is None:\n            return None, ""\n        if name == "generic_ops" or name in ("collected", "no_unexpanded_product"):\n            a, b = ops_raw(text), ops_raw(expected_text)\n            return a <= b + 1, "ops(raw)=%d vs reference %d (allowed +1)" % (a, b)\n        e = P(text, syms={"x": X, "y": Y})\n        if name == "simplified_radical":\n            raw = P(text, evaluate=False)\n            for pw in raw.atoms(sp.Pow):\n                if pw.exp == sp.Rational(1, 2) and pw.base.is_Integer and any(k >= 2 for k in sp.factorint(pw.base).values()):\n                    return False, "radical %s has a square factor" % pw\n            return True, ""\n        if name == "expanded":\n            return e == sp.expand(e), "structurally expanded" if e == sp.expand(e) else "not expanded"\n        if name == "factored":\n            f = sp.factor(e)\n            ok = isinstance(e, sp.Mul) and len(e.args) == len(f.args) and sp.simplify(f - e) == 0\n            return ok, "product of irreducible factors" if ok else "not a fully factored product"\n        if name == "no_fraction":\n            n, d = sp.fraction(sp.together(e))\n            return not d.has(X), "denominator has no x" if not d.has(X) else "still a fraction in x"\n        if name == "reduced_fraction":\n            n, d = sp.fraction(sp.together(e))\n            return sp.gcd(n, d) == 1, "numerator/denominator coprime"\n        if name == "partial_fractions":\n            def denominator_has_only_linear_factors(den):\n                den = sp.factor(den)\n                if not den.has(X):\n                    return True\n                try:\n                    _, factors = sp.factor_list(den)\n                    return all(sp.degree(base, X) <= 1 for base, _power in factors)\n                except Exception:  # noqa: BLE001\n                    return sp.degree(den, X) <= 1\n            ok = isinstance(e, sp.Add) and len(e.args) >= 2 and all(\n                denominator_has_only_linear_factors(sp.fraction(sp.together(t))[1]) for t in e.args)\n            return ok, "sum of partial fractions over linear factors (repeated poles allowed)" if ok else "not decomposed (must be a sum of terms with linear denominator factors)"\n    except Exception as ex:   # noqa: BLE001\n        return None, "form check could not run: %s" % type(ex).__name__\n    return None, ""\n\n\n# --------------------------------------------------------------------------- restrictions\ndef restriction_ok(required, reported_texts, step_texts):\n    """required like \'x != 1\'.  Satisfied if a reported constraint/step expresses the same exclusion."""\n    m = re.fullmatch(r"(\\w+)\\s*!=\\s*(.+)", required)\n    v, c = m.group(1), sp.Rational(m.group(2))\n    var = sp.Symbol(v)\n    for t in reported_texts + step_texts:\n        for part in re.split(r"[;,\\n]| and ", t):\n            mm = re.search(r"(.+?)\\s*(?:!=|≠)\\s*(.+)", part)\n            if mm:\n                try:\n                    l, r = P(mm.group(1).strip().split("|")[-1].strip(), syms={v: var}), P(mm.group(2).strip().split("(")[0].strip() if "(" in mm.group(2) and not re.match(r"^[\\w\\s\\+\\-\\*/\\^\\.]+$", mm.group(2).split("(")[0]) else mm.group(2).strip(), syms={v: var})\n                    sols = sp.solve(l - r, var)\n                    if sols and all(s == c for s in sols):\n                        return True\n                except Exception:   # noqa: BLE001\n                    pass\n            if re.search(r"\\b%s\\s*=\\s*%s\\b" % (re.escape(v), re.escape(str(c))), part) and re.search(r"denominator|undefined|excluded|reject|domain|zero", t, re.I):\n                return True\n    return False\n\n\n# --------------------------------------------------------------------------- answer comparers\ndef _finite_set_equal(actual_vals, exp_vals, tol=1e-9):\n    A, B = list(actual_vals), list(exp_vals)\n    if len(A) != len(B):\n        return False\n    used = [False] * len(B)\n    for a in A:\n        hit = False\n        for i, b in enumerate(B):\n            if not used[i]:\n                try:\n                    if sp.simplify(a - b) == 0 or abs(N(a - b)) < tol:\n                        used[i] = hit = True\n                        break\n                except Exception:   # noqa: BLE001\n                    pass\n        if not hit:\n            return False\n    return True\n\n\ndef judge(case, reply):\n    """-> verdict dict.  \'reply\' is the preserved raw worker reply."""\n    v = {"status": None, "sub": "", "value_correct": None, "form_ok": None, "domain_ok": None, "answer_produced": False,\n         "covered": False, "strict_correct": False, "value_only_correct": False, "expected": case["spec_expected"], "actual": "",\n         "proof": None, "manual_review": False, "manual_reasons": [], "form_note": "", "constant_shown": None, "detail": "",\n         "domain_check": {"required": case["required_restrictions"], "reported": reply.get("restrictions_reported", []), "satisfied": None}}\n    oc = reply.get("outcome")\n    err = reply.get("error") or {}\n    exp, cmp_ = case["expected"], case["comparison"]\n    method = cmp_["method"]\n    tol = case["tolerance"]\n\n    def done(status, sub="", **kw):\n        v.update(status=status, sub=sub, **kw)\n        v["value_only_correct"] = bool(v["value_correct"]) and v["form_ok"] is not False\n        v["strict_correct"] = bool(v["value_correct"]) and v["form_ok"] is not False and v["domain_ok"] is not False\n        v["covered"] = v["answer_produced"] and v["form_ok"] is not False and not v.get("_not_in_form")\n        v.pop("_not_in_form", None)\n        return v\n\n    if oc == "timeout":\n        return done("timeout", "no reply within the per-case timeout")\n    if oc == "infra_error":\n        return done("infra_error", err.get("message", "")[:200])\n    if oc == "unsupported":\n        r = reply.get("unsupported_reason", "")\n        sub = "no_api" if ("no public method" in r or "no mapping" in r) else "unsolved_by_solver"\n        v["actual"] = r[:200]\n        return done("unsupported", sub)\n    if oc == "input_incompatible":\n        v["actual"] = err.get("message", "")\n        return done("input_incompatible", err.get("type", ""))\n    if oc == "error":\n        v["actual"] = "%s: %s" % (err.get("type"), err.get("message", "")[:120])\n        if method == "undefined" and re.search(r"zerodivision|division|undefined|zoo|domain", (err.get("type", "") + err.get("message", "")), re.I):\n            v.update(answer_produced=True, value_correct=True, form_ok=True)\n            v["manual_review"] = True\n            v["manual_reasons"].append("L04 passed via raised division-by-zero error (no structured undefined result)")\n            v["detail"] = "error signalled instead of a fabricated solution; contract_exception=%s" % err.get("contract_exception")\n            return done("valid", "error_signalled_domain_error")\n        return done("runtime_error", err.get("type", ""))\n\n    # ------------- outcome == answer\n    ans = reply.get("answer") or {}\n    kind = ans.get("kind")\n    v["answer_produced"] = True\n    steps_text = " ".join((s.get("rule", "") + " " + s.get("description", "")) for s in reply.get("steps", []))\n    try:\n        # ---------------- scalar / expression families\n        if method in ("value", "expr_equiv", "antiderivative") and kind in ("expr", "numeric", "solutions", "equation"):\n            text = ans.get("text") or (ans.get("values") or [""])[0]\n            v["actual"] = text\n            e = P(text, syms={"x": X, "y": Y})\n            ref = P(exp["expr"], syms={"x": X, "y": Y})\n            if method == "value":\n                ok = numeq(e, ref, tol)\n                try:\n                    proof = "symbolic" if sp.simplify(e - ref) == 0 else "numeric"\n                except Exception:   # noqa: BLE001\n                    proof = "numeric"\n                v.update(value_correct=bool(ok), proof=proof if ok else "mismatch")\n            elif method == "expr_equiv":\n                ok, proof = equiv(e, ref, case)\n                v.update(value_correct=ok, proof=proof)\n                if ok and proof == "numeric":\n                    v["manual_review"] = True; v["manual_reasons"].append("equivalence established numerically only (no symbolic zero)")\n            else:   # antiderivative\n                e0, shown = strip_constant(e)\n                v["constant_shown"] = shown\n                real_points = cmp_.get("require_real_on")\n                real_ok = True\n                if real_points:\n                    # Evaluate the derivative in a real-symbol context. Generic complex symbols can leave\n                    # Derivative(re(x), x) / Derivative(Abs(x), x) unevaluated for log(Abs(u)) rules.\n                    xr, yr = sp.Symbol("x", real=True), sp.Symbol("y", real=True)\n                    eraw = P(text, syms={"x": xr, "y": yr})\n                    er0, _ = strip_constant(eraw)\n                    fr = P(case["problem"]["expr"], syms={"x": xr, "y": yr})\n                    deriv_residual = sp.diff(er0, xr) - fr\n                    sample_ok = True\n                    for pt in real_points:\n                        val = N(deriv_residual.subs(xr, sp.Rational(pt)), 40)\n                        if (not val.is_number or val.has(sp.zoo, sp.nan, sp.oo, -sp.oo) or abs(sp.im(val)) > 1e-12 or abs(sp.re(val)) > 1e-8):\n                            sample_ok = False\n                    ok, proof = sample_ok, "real-domain derivative samples" if sample_ok else "real-domain derivative mismatch"\n                    for pt in real_points:\n                        val = N(er0.subs(xr, sp.Rational(pt)))\n                        if not val.is_number or val.has(sp.zoo, sp.nan, sp.oo, -sp.oo) or abs(sp.im(val)) > 1e-12:\n                            real_ok = False\n                    v["domain_ok"] = real_ok\n                    v["detail"] = "F\'==f at declared real-domain sample points: %s ; antiderivative real-valued at those points: %s" % (ok, real_ok)\n                else:\n                    d = sp.diff(e0, X)\n                    f = P(case["problem"]["expr"], syms={"x": X, "y": Y})\n                    ok, proof = equiv(d, f, {**case, "domain": "complex"})\n                v.update(value_correct=bool(ok and real_ok), proof=proof)\n            fo, note = form_check(cmp_.get("form"), text, exp["expr"], case)\n            v["form_ok"], v["form_note"] = fo, note\n            if case["required_restrictions"]:\n                texts = list(reply.get("restrictions_reported", []))\n                oks = [restriction_ok(r, texts, [steps_text]) for r in case["required_restrictions"]]\n                v["domain_ok"] = all(oks)\n                v["domain_check"]["satisfied"] = all(oks)\n                v["manual_review"] = True; v["manual_reasons"].append("restriction-reporting decided by text matching")\n            return done("valid" if v["value_correct"] and v["form_ok"] is not False else "invalid_answer",\n                        "" if v["value_correct"] else ("domain_error" if v["domain_ok"] is False else "wrong_value"))\n\n        # ---------------- equation solving\n        if method == "solution_set":\n            domain_cx = case["domain"] == "complex"\n            ek = exp["kind"]\n            if kind == "empty":\n                v["actual"] = "no solution (empty set)"\n                if ek == "empty":\n                    v.update(value_correct=True, form_ok=True)\n                elif ek == "finite" and domain_cx and re.search(r"no real|real number system|over the real", steps_text, re.I):\n                    v.update(value_correct=False, form_ok=True)\n                    return done("invalid_answer", "real_domain_only (states no real solutions; complex solutions missing)")\n                else:\n                    v.update(value_correct=False, form_ok=True)\n                    return done("invalid_answer", "claims no solution but solutions exist" if ek != "empty" else "")\n                return done("valid")\n            if kind == "all":\n                v["actual"] = "all %s" % ans.get("domain", "x")\n                v.update(value_correct=(ek == "all"), form_ok=True)\n                return done("valid" if ek == "all" else "invalid_answer", "" if ek == "all" else "claims every x is a solution")\n            if kind == "solutions":\n                vals = [P(t, syms={"x": X}) for t in ans["values"]]\n                v["actual"] = "{" + ", ".join(ans["values"]) + "}"\n                if ek == "finite":\n                    ref = [P(t, syms={"x": X}) for t in exp["values"]]\n                    ok = _finite_set_equal(vals, ref)\n                    sub = ""\n                    if not ok and domain_cx and len(vals) < len(ref) and all(any(_finite_set_equal([a], [b]) for b in ref) for a in vals):\n                        sub = "real_domain_only (complex roots missing)"\n                    v.update(value_correct=ok, form_ok=True)\n                    return done("valid" if ok else "invalid_answer", sub or ("" if ok else "wrong_solution_set"))\n                v.update(value_correct=False, form_ok=True)\n                return done("invalid_answer", "returned solutions where %s expected" % ek)\n            v["actual"] = json.dumps(ans)[:200]\n            v["_not_in_form"] = True\n            return done("invalid_answer", "unexpected answer form %r" % kind)\n\n        if method == "system_solution":\n            ek = exp["kind"]\n            if kind == "empty":\n                v.update(value_correct=(ek == "empty"), form_ok=True, actual="no solution")\n                return done("valid" if ek == "empty" else "invalid_answer", "" if ek == "empty" else "claims no solution")\n            if kind == "system":\n                sols = ans["solutions"]\n                v["actual"] = json.dumps(sols)\n                if ek == "unique" and len(sols) == 1:\n                    ok = all(k in sols[0] and sp.simplify(P(sols[0][k]) - sp.Rational(val)) == 0 for k, val in exp["assignments"].items()) and set(sols[0]) == set(exp["assignments"])\n                    v.update(value_correct=ok, form_ok=True)\n                    return done("valid" if ok else "invalid_answer", "" if ok else "wrong_solution")\n                v.update(value_correct=False, form_ok=True)\n                return done("invalid_answer", "returned %d solution(s) where %s expected" % (len(sols), ek))\n            v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True\n            return done("invalid_answer", "unexpected answer form %r" % kind)\n\n        if method == "inequality_set":\n            exp_set = sp.Union(*[sp.Interval(sp.sympify(a.replace("oo", "oo")), sp.sympify(b), l, r) for a, b, l, r in exp["intervals"]])\n            if kind == "intervals":\n                act = sp.Union(*[sp.Interval(sp.sympify(a), sp.sympify(b), l, r) for a, b, l, r in ans["intervals"]])\n                v["actual"] = str(act)\n            elif kind == "relation":\n                rel = P(ans["text"], syms={"x": XR})\n                act = rel.as_set()\n                v["actual"] = ans["text"]\n            else:\n                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True\n                return done("invalid_answer", "unexpected answer form %r" % kind)\n            ok = act == exp_set or act.symmetric_difference(exp_set) == sp.S.EmptySet\n            v.update(value_correct=bool(ok), form_ok=True)\n            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_interval_set")\n\n        if method == "matrix":\n            if kind != "matrix":\n                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True\n                return done("invalid_answer", "unexpected answer form %r" % kind)\n            A = sp.Matrix([[P(c) for c in r] for r in ans["rows"]])\n            B = sp.Matrix([[sp.Rational(c) for c in r] for r in exp["matrix"]])\n            v["actual"] = str(A.tolist())\n            ok = A.shape == B.shape and (A - B).applyfunc(sp.simplify) == sp.zeros(*B.shape)\n            v.update(value_correct=bool(ok), form_ok=True)\n            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_matrix")\n\n        if method == "gradient":\n            if kind != "vector":\n                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True\n                return done("invalid_answer", "unexpected answer form %r" % kind)\n            v["actual"] = "(" + ", ".join(ans["items"]) + ")"\n            ok = len(ans["items"]) == len(exp["vector"]) and all(equiv(P(a, syms={"x": X, "y": Y}), P(b, syms={"x": X, "y": Y}), case)[0] for a, b in zip(ans["items"], exp["vector"]))\n            v.update(value_correct=bool(ok), form_ok=True)\n            return done("valid" if ok else "invalid_answer", "" if ok else "wrong_gradient")\n\n        if method == "ode_family":\n            if kind != "equation":\n                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True\n                return done("invalid_answer", "unexpected answer form %r" % kind)\n            f = sp.Function("y")\n            eq = P(ans["text"], syms={"x": X}, extra={"y": f})\n            v["actual"] = ans["text"]\n            rhs = eq.rhs\n            ode = P(case["problem"]["ode"], syms={"x": X}, extra={"y": f})\n            consts = sorted([s for s in rhs.free_symbols if re.fullmatch(r"C\\d*", s.name)], key=lambda s: s.name)\n            resid = sp.simplify(ode.subs(f(X), rhs).doit())\n            ok_ode = resid == 0\n            ok_n = len(consts) == exp["n_constants"]\n            basis = [sp.diff(rhs, c) for c in consts]\n            indep = (basis[0] != 0) if len(basis) == 1 else (sp.simplify(sp.wronskian(basis, X)) != 0 if len(basis) >= 2 else False)\n            ok = ok_ode and ok_n and indep\n            v["detail"] = "ODE residual after substitution: %s ; constants=%d ; independent basis: %s" % (resid, len(consts), indep)\n            v.update(value_correct=bool(ok), form_ok=True)\n            return done("valid" if ok else "invalid_answer", "" if ok else "not a general solution")\n\n        if method == "numeric_roots":\n            vals = ans.get("values", [])\n            if kind == "empty":\n                v.update(value_correct=False, form_ok=True, actual="no solution")\n                return done("invalid_answer", "claims no solution but a real root exists")\n            if kind not in ("numeric", "solutions"):\n                v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True\n                return done("invalid_answer", "unexpected answer form %r" % kind)\n            nums = []\n            for t in vals:\n                try:\n                    nums.append(N(P(t, syms={"x": X}), 30))\n                except Exception:   # noqa: BLE001\n                    nums.append(None)\n            v["actual"] = "[" + ", ".join(vals) + "]"\n            ref = [sp.Float(t, 30) for t in exp["values"]]\n            ok = all(any(n is not None and n.is_number and abs(sp.im(n)) < 1e-9 and abs(sp.re(n) - r) <= tol["abs"] for n in nums) for r in ref)\n            spec_val = exp.get("spec_printed_value")\n            if spec_val:\n                v["detail"] = "matches spec-printed value %s within 1e-6: %s" % (spec_val, any(n is not None and abs(sp.im(n)) < 1e-9 and abs(sp.re(n) - sp.Float(spec_val, 30)) <= 1e-6 for n in nums))\n            v.update(value_correct=bool(ok), form_ok=True)\n            if any("CRootOf" in t for t in vals):\n                v["manual_review"] = True; v["manual_reasons"].append("exact CRootOf accepted via numeric evaluation (declared rule)")\n            return done("valid" if ok else "invalid_answer", "" if ok else "no root within tolerance")\n\n        if method == "undefined":\n            if kind == "undefined":\n                v.update(value_correct=True, form_ok=True, actual="undefined: " + str(ans.get("reason", ""))[:120])\n                return done("valid", "explicit_undefined_result")\n            if kind == "empty":\n                v.update(value_correct=False, form_ok=False, actual="no solution (empty set) - undefined not flagged")\n                v["manual_review"] = True; v["manual_reasons"].append("L04 partial: empty set without undefined flag")\n                return done("invalid_answer", "partial: empty solution set, expression not flagged undefined; no fabricated solution")\n            v.update(value_correct=False, form_ok=False, actual=json.dumps(ans)[:150])\n            return done("invalid_answer", "fabricated_solution")\n    except Exception as ex:   # noqa: BLE001\n        v["manual_review"] = True\n        v["manual_reasons"].append("judge could not evaluate the output: %s: %s" % (type(ex).__name__, str(ex)[:100]))\n        v["actual"] = v["actual"] or json.dumps(ans)[:200]\n        v.update(value_correct=None, _not_in_form=True)\n        return done("invalid_answer", "unjudgeable_output")\n    v["manual_review"] = True; v["manual_reasons"].append("no comparer matched answer kind %r for method %r" % (kind, method))\n    v["actual"] = json.dumps(ans)[:200]; v["_not_in_form"] = True\n    return done("invalid_answer", "unexpected answer form %r" % kind)\n')
W('bench/derivations.py', '"""Derivation-quality scoring (rubric: correctness 0-3, completeness 0-2, explanation 0-2, consistency 0-1, domain 0-1, presentation 0-1).\n\nAutomated layer (deterministic, auditable; thresholds below are declared before results were inspected):\n  correctness   0 final answer not (strictly) correct | 1 final correct but an automated check found an invalid step or the\n                conclusion contradicts the final answer | 2 final correct, no invalid step, but the worked steps do not demonstrably\n                produce the answer (solver flagged \'direct_evaluation\' override, or final answer absent from every step) | 3 clean.\n  completeness  share of the case\'s reference milestones (regex over step text) found: >=0.75 -> 2, >=0.34 -> 1, else 0; a derivation\n                that is only opaque/structural steps scores 0; fewer than MIN_STEPS[op] substantive steps caps the score at 1.\n  explanation   per substantive step: 2 = specific rule label + >=8 prose words ; 1 = specific rule label OR >=4 prose words ; 0 = opaque\n                (\'Compute exact result\', \'Direct evaluation\', ...).  mean>=1.5 -> 2 ; >=0.6 -> 1 ; else 0.\n  consistency   1 if no invalid transition, no override, and the final answer appears in the workings; else 0.\n  domain        domain-sensitive case: 1 iff the answer\'s domain handling is correct AND the workings mention the restriction/rejection\n                (constraint list or domain vocabulary); else 0.  Non-sensitive case: 1 unless a domain error was detected (N/A -> full;\n                reported separately as \'domain-sensitive-only mean\').\n  presentation  1 iff >=80% of steps have readable expression text free of Python-object artefacts (<...>, Symbol(, ...Rule( trees).\nThese are PROXIES.  A manual-review overlay (data/manual_reviews.json) records per-criterion adjustments with confidence and\njustification; every row keeps both the automated and the final score.  The reviewer is a language model (Claude), so the overlay is\nflagged \'model_review\' and is NOT the authority on mathematical correctness (that comes from judge.py\'s symbolic/numeric checks).\n"""\nimport json\nimport math\nimport re\n\nimport sympy as sp\n\ntry:\n    from . import judge as J\nexcept ImportError:\n    import judge as J\n\nMIN_STEPS = {"evaluate": 1, "simplify": 1, "expand": 1, "factor": 1, "solve": 2, "nsolve": 2, "solve_system": 3, "solve_inequality": 2,\n             "diff": 2, "partial_derivative": 2, "gradient": 2, "integrate": 2, "limit": 2, "det": 2, "inverse": 2, "rref": 2,\n             "dsolve": 2, "partial_fractions": 3}\nOPAQUE = {"compute exact result", "direct evaluation", "general solving"}\nGENERIC_RULES = {"apply algebraic identities"}      # boilerplate label reused across unrelated problems: not a specific rule identification\nSTRUCTURAL = {"setup", "final result", "limit setup", "partial derivative setup", "gradient setup", "augmented matrix",\n              "identify linear equation", "identify quadratic equation", "identify polynomial", "identify rational form",\n              "reduced row echelon form"}\nDOMAIN_WORDS = re.compile(r"domain|restrict|undefined|denominator|extraneous|reject|!=|≠|no real|real number|nonzero|non-zero|positive|"\n                          r"excluded|must not|cannot be zero|inconsistent|identity|contradict|absolute value|\\|x\\|", re.I)\nARTEFACT = re.compile(r"<[^>]*object|Symbol\\(|Rule\\(|\\bRule\\b.*integrand=|_sympy|\\bzoo\\b")\n\n\ndef prose_words(text):\n    if ARTEFACT.search(text) or "integrand=" in text:      # raw object/rule-tree dumps are not prose\n        return 0\n    t = re.sub(r"[-+*/^=()\\[\\]{}<>|,.:;→←∫∂Δ∇]", " ", text)\n    return len([w for w in t.split() if re.fullmatch(r"[A-Za-z\']{3,}", w) and w.lower() not in ("sin", "cos", "tan", "log", "exp", "sqrt", "abs", "dx", "matrix")])\n\n\ndef _flat(steps):\n    out = []\n    for s in steps:\n        out.append(" ".join([s.get("rule", ""), s.get("description", ""), s.get("before", ""), s.get("after", "")] + [" ".join(n) for n in s.get("notes", [])]))\n        for sub in s.get("substeps", []) or []:\n            out.append(json.dumps(sub))\n    return " \\n".join(out)\n\n\ndef _expr_or_none(t):\n    try:\n        if not t or any(tok in t for tok in ("Eq(", "[", "{", "<", ">", "diff(", "integrate(", "limit(", "matrix", "solve", "|")):\n            return None\n        return J.P(t, syms={"x": J.X, "y": J.Y})\n    except Exception:   # noqa: BLE001\n        return None\n\n\ndef check_steps(case, steps, final_ok):\n    """-> (n_checked, n_invalid, notes).  Only algebra families are checked transition-by-transition."""\n    op = case["operation"]\n    checked = invalid = 0\n    notes = []\n    if op in ("evaluate", "simplify", "expand", "factor"):\n        prev = _expr_or_none(case["problem"]["expr"].replace("^", "**"))\n        for i, s in enumerate(steps):\n            for key in ("before", "after"):\n                e = _expr_or_none(s.get(key, "").replace("^", "**"))\n                if e is None or prev is None:\n                    continue\n                if key == "before":\n                    # StepSolver gives before/after pairs; \'before\' must equal the previous \'after\' (or input)\n                    ok, _ = J.equiv(e, prev, {**case, "domain": "complex"})\n                else:\n                    ok, _ = J.equiv(e, prev, {**case, "domain": "complex"})\n                    prev = e\n                checked += 1\n                if not ok:\n                    invalid += 1\n                    notes.append("step %d (%s): \'%s\' not equivalent to previous expression" % (i + 1, key, s.get(key)))\n    elif op in ("solve", "nsolve"):\n        truth = [J.P(t) for t in case["expected"].get("values", [])] if case["expected"].get("kind") == "finite" else None\n        for i, s in enumerate(steps):\n            m = re.match(r"^\\s*Eq\\(\\s*x\\s*,\\s*(.+)\\)\\s*$", s.get("after", ""))\n            if m and truth is not None:\n                try:\n                    val = J.P(m.group(1).replace("^", "**"), syms={"x": J.X})\n                except Exception:   # noqa: BLE001\n                    continue\n                checked += 1\n                if not any(sp.simplify(val - t) == 0 for t in truth):\n                    # a step may legitimately show a REJECTED candidate (extraneous root); only flag if the text does not say so\n                    if not re.search(r"reject|extraneous|does not satisfy|not a solution", s.get("description", "") + s.get("rule", ""), re.I):\n                        invalid += 1\n                        notes.append("step %d claims x = %s which is not a true solution" % (i + 1, m.group(1)))\n    return checked, invalid, notes\n\n\ndef automated_score(case, reply, verdict):\n    steps = [s for s in reply.get("steps", []) if s]\n    rec = {"available": bool(steps), "n_steps": len(steps), "scored": False}\n    if not steps or reply.get("outcome") != "answer":\n        return rec\n    flat = _flat(steps)\n    nf = reply.get("native_flags") or {}\n    rules = [s.get("rule", "").strip().lower() for s in steps]\n    substantive = [s for s, r in zip(steps, rules) if r not in OPAQUE and r not in STRUCTURAL and not r.startswith("identify")]\n    final_ok = bool(verdict.get("value_correct")) and verdict.get("form_ok") is not False\n    n_chk, n_inv, inv_notes = check_steps(case, steps, final_ok)\n\n    # --- correctness\n    override = nf.get("steps_mode") == "direct_evaluation" or nf.get("workings_tied_to_answer") is False\n    if not final_ok:\n        c = 0\n        why_c = "final answer not correct"\n    elif n_inv:\n        c, why_c = 1, "invalid step(s): " + "; ".join(inv_notes[:2])\n    else:\n        ans_text = json.dumps(reply.get("answer"))\n        present = True\n        # final answer present in workings? (value tokens of the answer must occur in some step text)\n        toks = re.findall(r"[A-Za-z0-9_./^*+-]+", re.sub(r"[\\[\\]{}\\"\']", " ", ans_text))\n        toks = [t for t in toks if len(t) > 0 and t not in ("kind", "expr", "text", "values", "solutions", "empty", "rows", "items", "system", "relation", "equation", "mapping", "entries")]\n        if toks:\n            norm = lambda t: re.sub(r"\\s+", "", t.replace("**", "^"))\n            present = any(norm(t) in norm(flat) for t in toks)\n        if override:\n            c, why_c = 2, "workings do not demonstrably produce the answer (steps_mode=direct_evaluation override, or workings from a code path separate from the answer, e.g. SymPy integral_steps vs integrate)"\n        elif not present:\n            c, why_c = 2, "final answer does not appear in the workings"\n        else:\n            c, why_c = 3, "no invalid step found (%d transitions auto-checked)" % n_chk\n\n    # --- contradiction detection for integrals G04-like (derivation says ln|x| but answer is log(x))\n    if case["id"] == "G04" and final_ok is False and re.search(r"ln\\|x\\||log\\(abs", flat):\n        why_c += "; workings state ln|x| but the returned antiderivative is not real-valued for x<0 (contradiction)"\n\n    # --- completeness\n    ms = case.get("reference_milestones") or []\n    hits = sum(1 for m in ms if re.search(m, flat, re.I))\n    frac = (hits / len(ms)) if ms else 1.0\n    comp = 2 if frac >= 0.75 else 1 if frac >= 0.34 else 0\n    if not substantive:\n        comp = 0\n    elif len(substantive) < MIN_STEPS.get(case["operation"], 1):\n        comp = min(comp, 1)\n\n    # --- explanation\n    per = []\n    for s in substantive:\n        spec = s.get("rule", "").strip().lower() not in OPAQUE | GENERIC_RULES and bool(s.get("rule", "").strip())\n        w = prose_words(s.get("description", ""))\n        per.append(2 if (spec and w >= 8) else 1 if (spec or w >= 4) else 0)\n    m_exp = (sum(per) / len(per)) if per else 0.0\n    expl = 2 if m_exp >= 1.5 else 1 if m_exp >= 0.6 else 0\n\n    # --- consistency\n    cons = 1 if (n_inv == 0 and not override and c >= 2) else 0\n\n    # --- domain\n    if case["domain_sensitive"]:\n        mentions = bool(reply.get("restrictions_reported")) or bool(DOMAIN_WORDS.search(flat))\n        dom = 1 if (verdict.get("domain_ok") is not False and final_ok and mentions) else 0\n    else:\n        dom = 1 if verdict.get("domain_ok") is not False else 0\n\n    # --- presentation\n    readable = [s for s in steps if (s.get("after") or s.get("description")) and not ARTEFACT.search(s.get("after", "") + s.get("description", ""))]\n    pres = 1 if len(readable) >= 0.8 * len(steps) else 0\n\n    if not substantive:\n        # Rubric rule 5/9 refinement (declared after the first inspection, applied uniformly to every solver):\n        # a derivation with no substantive step shows no transformations -> correctness<=2, nothing to follow (consistency 0), nothing readable (presentation 0).\n        c, cons, pres = min(c, 2), 0, 0\n        why_c += "; opaque/structural steps only (no transformation shown)"\n    rec.update(scored=True, automated={"correctness": c, "completeness": comp, "explanation": expl, "consistency": cons, "domain": dom,\n                                       "presentation": pres}, automated_total=c + comp + expl + cons + dom + pres,\n               evidence={"correctness": why_c, "milestones_hit": "%d/%d" % (hits, len(ms)), "mean_explanation_per_step": round(m_exp, 2),\n                         "substantive_steps": len(substantive), "steps_auto_checked": n_chk, "invalid_steps": n_inv,\n                         "opaque_steps": sum(1 for r in rules if r in OPAQUE), "native_flags": {k: nf.get(k) for k in ("steps_mode", "steps_verified", "workings_tied_to_answer")}})\n    return rec\n\n\nCRITERIA = ["correctness", "completeness", "explanation", "consistency", "domain", "presentation"]\nCAPS = dict(correctness=3, completeness=2, explanation=2, consistency=1, domain=1, presentation=1)\n\n\ndef apply_manual(rec, review):\n    """review: {\'adjust\': {criterion: new_value}, \'confidence\': \'low|medium|high\', \'justification\': str}"""\n    if not rec.get("scored"):\n        return rec\n    final = dict(rec["automated"])\n    if review:\n        for k, v in review.get("adjust", {}).items():\n            assert k in CAPS and 0 <= v <= CAPS[k], (k, v)\n            final[k] = v\n        rec["manual"] = {"reviewer": "model_review (Claude) - requires human audit", "confidence": review["confidence"],\n                         "justification": review["justification"], "adjusted": review.get("adjust", {})}\n    rec["final"] = final\n    rec["final_total"] = sum(final.values())\n    return rec\n')
W('bench/reference_validation.py', '#!/usr/bin/env python3\n"""Independently validate every reference answer in data/cases.json.\n\nMethod: exact rational arithmetic (fractions.Fraction), cmath/math, and mpmath numerics\n(numerical differentiation/quadrature/root polishing).  No SymPy symbolic engine is used,\nso these checks do not share code with any solver under test or with the judge.\nEach case gets >=1 check; most get two independent methods (listed in \'methods\').\n"""\nimport cmath\nimport json\nimport math\nimport pathlib\nimport random\nimport sys\nfrom fractions import Fraction as Fr\n\nimport mpmath as mp\n\nmp.mp.dps = 40\nROOT = pathlib.Path(__file__).resolve().parent.parent\nRNG = random.Random(20240601)   # fixed seed -> deterministic\nR = {}\n\n\ndef rec(cid, ok, methods, detail=""):\n    R[cid] = {"ok": bool(ok), "methods": methods, "detail": detail}\n\n\ndef rat_points(n=8, avoid=(), lo=-9, hi=9):\n    pts = []\n    while len(pts) < n:\n        p = Fr(RNG.randint(lo * 7, hi * 7), RNG.randint(2, 7))\n        if p not in avoid and p not in pts:\n            pts.append(p)\n    return pts\n\n\ndef identity(cid, f, g, avoid=(), n=10, extra=""):\n    """f(x)==g(x) at n random rationals (exact)."""\n    ok = all(f(p) == g(p) for p in rat_points(n, avoid))\n    rec(cid, ok, ["exact rational identity test at %d random points" % n], extra)\n\n\ndef mpnum_deriv(f, x):\n    return mp.diff(f, x)\n\n\n# ---------------------------------------------------------------- A\nrec("A01", Fr(2) + 3 * 4 == 14, ["Fraction arithmetic"])\nrec("A02", (Fr(2) + 3) * 4 == 20, ["Fraction arithmetic"])\nrec("A03", Fr(7, 12) + Fr(5, 18) == Fr(31, 36), ["Fraction arithmetic"])\nrec("A04", 72 == 36 * 2 and abs(math.sqrt(72) - 6 * math.sqrt(2)) < 1e-12 and math.isqrt(36) ** 2 == 36,\n    ["integer factorisation 72=6^2*2", "float check"], "largest square factor of 72 is 36")\nrec("A05", (2 ** 3) ** 2 == 64, ["integer arithmetic"])\nrec("A06", Fr("0.125") * 80 == 10, ["Fraction arithmetic (exact decimal)"])\nrec("A07", (-3) ** 2 - 3 ** 2 == 0, ["integer arithmetic"])\nrec("A08", abs(-7) + abs(3 - 9) == 13, ["integer arithmetic"])\n\n# ---------------------------------------------------------------- B\nidentity("B01", lambda x: 3 * x + 5 * x - 2 * x, lambda x: 6 * x)\nidentity("B02", lambda x: 2 * (x + 3) - x, lambda x: x + 6)\nidentity("B03", lambda x: (x + 2) * (x + 3), lambda x: x * x + 5 * x + 6)\nidentity("B04", lambda x: x * x + 5 * x + 6, lambda x: (x + 2) * (x + 3))\nidentity("B05", lambda x: x * x - 9, lambda x: (x - 3) * (x + 3))\nidentity("B06", lambda x: (x * x - 1) / (x - 1), lambda x: x + 1, avoid=(Fr(1),),\n         extra="original undefined at x=1 (ZeroDivisionError confirmed below)")\ntry:\n    (Fr(1) * Fr(1) - 1) / (Fr(1) - 1)\n    z = False\nexcept ZeroDivisionError:\n    z = True\nR["B06"]["ok"] = R["B06"]["ok"] and z\nidentity("B07", lambda x: x / x, lambda x: Fr(1), avoid=(Fr(0),))\nidentity("B08", lambda x: None, lambda x: None)  # replaced below (two variables)\npts = [(a, b) for a in rat_points(5, (Fr(0),)) for b in rat_points(3, (Fr(0),))]\nrec("B08", all((2 * a * a * b) / (4 * a * b * b) == a / (2 * b) for a, b in pts),\n    ["exact rational identity test at 15 (x,y) points"], "original denominator 4*x*y^2 zero iff x=0 or y=0")\n\n# ---------------------------------------------------------------- C\ndef lin(a, b):  # a*x + b = 0\n    return -Fr(b) / Fr(a)\n\nrec("C01", 2 * Fr(4) + 3 == 11 and lin(2, -8) == 4, ["substitution", "closed form"])\nrec("C02", 2 * (Fr(4) + 3) == 14, ["substitution", "closed form"])\nrec("C03", 3 * Fr(6) - 7 == Fr(6) + 5, ["substitution"])\nrec("C04", Fr(9, 3) + 2 == 5, ["substitution"])\nrec("C05", all(x * x - 5 * x + 6 == 0 for x in (2, 3)) and (25 - 24) == 1 and\n    sorted(mp.polyroots([1, -5, 6])) == sorted([mp.mpf(2), mp.mpf(3)]) or\n    all(abs(r - t) < 1e-25 for r, t in zip(sorted(mp.polyroots([1, -5, 6]), key=lambda z: z.real), [2, 3])),\n    ["substitution", "mpmath.polyroots", "discriminant 1>0"])\nrec("C06", all(x * x - 4 == 0 for x in (-2, 2)), ["substitution", "discriminant 16>0 (two real roots)"])\nrts = mp.polyroots([1, 0, 1])\nrec("C07", all(abs(r * r + 1) < 1e-30 for r in (1j, -1j)) and\n    sorted([complex(r).imag for r in rts]) == [-1.0, 1.0] and all(abs(complex(r).real) < 1e-20 for r in rts) and\n    (0 - 4) < 0, ["complex substitution", "mpmath.polyroots", "discriminant -4<0 -> no real roots"])\nrec("C08", (-1) ** 2 + 2 * (-1) + 1 == 0 and (4 - 4) == 0 and\n    mp.diff(lambda x: x * x + 2 * x + 1, -1) == 0,\n    ["substitution", "discriminant 0", "p\'(-1)=0 -> double root"])\nrec("C09", 2 * 1 + 1 != 2 * 1 + 3 and all(2 * p + 1 != 2 * p + 3 for p in rat_points(8)),\n    ["LHS-RHS == -2 identically (point test)"])\nrec("C10", all(2 * p + 1 == 2 * p + 1 for p in rat_points(8)), ["LHS-RHS == 0 identically (point test)"])\n\n# ---------------------------------------------------------------- D\ndef cramer(a1, b1, c1, a2, b2, c2):\n    det = Fr(a1) * b2 - Fr(a2) * b1\n    if det == 0:\n        return None\n    return ((Fr(c1) * b2 - Fr(c2) * b1) / det, (Fr(a1) * c2 - Fr(a2) * c1) / det)\n\nfor cid, args, sol in [("D01", (1, 1, 7, 1, -1, 1), (4, 3)), ("D02", (2, 1, 8, 1, -1, 1), (3, 2)), ("I01", (1, 1, 5, 1, -1, 1), (3, 2))]:\n    r = cramer(*args)\n    a1, b1, c1, a2, b2, c2 = args\n    sub = a1 * sol[0] + b1 * sol[1] == c1 and a2 * sol[0] + b2 * sol[1] == c2\n    rec(cid, r == tuple(Fr(v) for v in sol) and sub, ["Cramer\'s rule (Fractions)", "substitution"])\n# D03 inconsistent: det = 0 and rank test (augmented rank 2 vs coefficient rank 1)\nrec("D03", Fr(1) * 1 - Fr(1) * 1 == 0 and 2 != 5, ["det=0 with inconsistent RHS (coefficients proportional, constants not)"])\n\ndef sample_ineq(pred, intervals, grid):\n    """pred(x) truth must equal membership in the expected intervals for every grid point (incl. boundaries)."""\n    def member(x):\n        for lo, hi in intervals:\n            if (lo is None or x > lo) and (hi is None or x < hi):\n                return True\n        return False\n    return all(pred(g) == member(g) for g in grid)\n\ngrid = [Fr(k, 4) for k in range(-80, 81)]\nrec("D04", sample_ineq(lambda x: 2 * x + 3 < 11, [(None, Fr(4))], grid), ["exhaustive grid truth table (161 pts incl. boundary x=4)"])\nrec("D05", sample_ineq(lambda x: -2 * x > 6, [(None, Fr(-3))], grid), ["grid truth table incl. boundary x=-3 and direction flip"])\nrec("D06", sample_ineq(lambda x: x * x - 4 > 0, [(None, Fr(-2)), (Fr(2), None)], grid), ["grid truth table incl. boundaries x=±2"])\n\n# ---------------------------------------------------------------- E\nidentity("E01", lambda x: x ** 3 * x ** 4, lambda x: x ** 7)\nidentity("E02", lambda x: x ** 6 / x ** 2, lambda x: x ** 4, avoid=(Fr(0),))\nrec("E03", all(math.sqrt(x * x) == abs(x) for x in [-3.0, -0.5, 2.0, 7 / 3]) and math.sqrt((-3.0) ** 2) != -3.0,\n    ["float check at negative/positive x", "sqrt((-3)^2)=3 != -3"])\nrec("E04", math.isclose(math.log(1000, 10), 3, rel_tol=1e-12) and 10 ** 3 == 1000 and mp.log(1000, 10) == 3,\n    ["math.log", "10^3==1000", "mpmath.log"])\nrec("E05", all(math.isclose(math.log(math.exp(x)), x, abs_tol=1e-12) for x in [-3.0, -0.5, 2.0, 7 / 3]),\n    ["float check over reals incl. negatives"])\nrec("E06", abs(mp.sin(mp.pi / 6) - mp.mpf(1) / 2) < mp.mpf(10) ** -35, ["mpmath 40-digit"])\nrec("E07", abs(mp.cos(mp.pi / 3) - mp.mpf(1) / 2) < mp.mpf(10) ** -35, ["mpmath 40-digit"])\nrec("E08", abs(mp.tan(mp.pi / 4) - 1) < mp.mpf(10) ** -35, ["mpmath 40-digit"])\n\n# ---------------------------------------------------------------- F  (mpmath numerical differentiation vs expected closed form)\ndef deriv_check(cid, f, d, pts):\n    err = max(abs(mp.diff(f, p) - d(p)) for p in pts)\n    rec(cid, err < mp.mpf(10) ** -25, ["mpmath.diff numerical differentiation (40 digits) vs closed form at %d pts" % len(pts)],\n        "max abs err %s" % mp.nstr(err, 3))\n\nP = [mp.mpf(v) for v in ("-2.7", "-0.4", "0.5", "1.3", "3.9")]\nPpos = [mp.mpf(v) for v in ("0.5", "1.3", "2.7", "7.1")]\nderiv_check("F01", lambda x: x ** 3, lambda x: 3 * x ** 2, P)\nderiv_check("F02", lambda x: 5 * x ** 4 - 2 * x + 7, lambda x: 20 * x ** 3 - 2, P)\nderiv_check("F03", lambda x: x ** 2 * mp.sin(x), lambda x: 2 * x * mp.sin(x) + x ** 2 * mp.cos(x), P)\nderiv_check("F04", lambda x: mp.sin(x ** 2), lambda x: 2 * x * mp.cos(x ** 2), P)\nderiv_check("F05", lambda x: mp.log(x), lambda x: 1 / x, Ppos)\nderiv_check("F06", lambda x: x ** x, lambda x: x ** x * (mp.log(x) + 1), Ppos)\n\n# ---------------------------------------------------------------- G\ndef anti_check(cid, f, F, pts):\n    err = max(abs(mp.diff(F, p) - f(p)) for p in pts)\n    rec(cid, err < mp.mpf(10) ** -25, ["mpmath.diff of expected F equals integrand at %d pts" % len(pts),\n                                       "second check: F(b)-F(a) == mpmath.quad"],\n        "max abs err %s" % mp.nstr(err, 3))\n    a, b = mp.mpf("0.5"), mp.mpf("2.5")\n    q = mp.quad(f, [a, b])\n    R[cid]["ok"] = R[cid]["ok"] and abs((F(b) - F(a)) - q) < mp.mpf(10) ** -25\n\nanti_check("G01", lambda x: 3 * x ** 2, lambda x: x ** 3, P)\nanti_check("G02", lambda x: 2 * x + 5, lambda x: x ** 2 + 5 * x, P)\nanti_check("G03", lambda x: mp.cos(x), lambda x: mp.sin(x), P)\nanti_check("G05", lambda x: x * mp.exp(x ** 2), lambda x: mp.exp(x ** 2) / 2, P)\n# G04: ln|x| differentiates to 1/x on both sides of 0; ln(x) alone is not real for x<0\nneg, pos = [mp.mpf("-4"), mp.mpf("-0.5")], [mp.mpf("0.5"), mp.mpf("4")]\nerr = max(abs(mp.diff(lambda x: mp.log(abs(x)), p) - 1 / p) for p in neg + pos)\nrec("G04", err < mp.mpf(10) ** -25 and all(abs(mp.log(abs(p)).imag) == 0 for p in neg) and\n    all(abs(mp.log(p).imag) > 1 for p in neg),\n    ["mpmath.diff of ln|x| at x<0 and x>0", "realness: ln|x| real, ln(x) complex for x<0"], "err %s" % mp.nstr(err, 3))\nq = mp.quad(lambda x: x ** 2, [0, 3])\nrec("G06", abs(q - 9) < mp.mpf(10) ** -30 and Fr(27, 3) == 9, ["mpmath.quad", "exact (b^3-a^3)/3"])\n\n# ---------------------------------------------------------------- H\ndef limit_check(cid, f, x0, target, side_eps=("1e-12", "-1e-12")):\n    vals = [f(x0 + mp.mpf(e)) for e in side_eps] if x0 is not None else [f(mp.mpf(10) ** 15), f(mp.mpf(10) ** 20)]\n    ok = all(abs(v - target) < mp.mpf(10) ** -9 for v in vals)\n    rec(cid, ok, ["mpmath 60-digit evaluation approaching the point from both sides"], "values %s" % [mp.nstr(v, 12) for v in vals])\n\nmp.mp.dps = 60\nlimit_check("H01", lambda x: mp.sin(x) / x, mp.mpf(0), mp.mpf(1))\nlimit_check("H02", lambda x: (x ** 2 - 4) / (x - 2), mp.mpf(2), mp.mpf(4))\nlimit_check("H03", lambda x: (3 * x ** 2 + 1) / (x ** 2 - 5), None, mp.mpf(3))\nlimit_check("H04", lambda x: (1 - mp.cos(x)) / x ** 2, mp.mpf(0), mp.mpf(1) / 2)\nlim = mp.limit(lambda x: (1 - mp.cos(x)) / x ** 2, 0)\nR["H04"]["ok"] = R["H04"]["ok"] and abs(lim - mp.mpf(1) / 2) < mp.mpf(10) ** -15\nR["H04"]["methods"].append("mpmath.limit")\nmp.mp.dps = 40\n\n# ---------------------------------------------------------------- I\nA = [[Fr(1), Fr(2)], [Fr(3), Fr(4)]]\ndet = A[0][0] * A[1][1] - A[0][1] * A[1][0]\ninv = [[Fr(-2), Fr(1)], [Fr(3, 2), Fr(-1, 2)]]\nprod = [[sum(A[i][k] * inv[k][j] for k in range(2)) for j in range(2)] for i in range(2)]\nrec("I02", det == -2, ["2x2 formula ad-bc", "Leibniz/cofactor by hand in Fractions"])\nrec("I03", prod == [[1, 0], [0, 1]] and all(inv[i][j] * det == [[4, -2], [-3, 1]][i][j] * 1 for i in range(2) for j in range(2)) is False or prod == [[1, 0], [0, 1]],\n    ["A*A^-1 == I (Fractions)"])\nR["I03"]["ok"] = prod == [[1, 0], [0, 1]] and [[inv[i][j] for j in range(2)] for i in range(2)] == [[Fr(-2), Fr(1)], [Fr(3, 2), Fr(-1, 2)]]\n\ndef rref_fr(m):\n    m = [row[:] for row in m]\n    r = 0\n    for c in range(len(m[0])):\n        p = next((i for i in range(r, len(m)) if m[i][c] != 0), None)\n        if p is None:\n            continue\n        m[r], m[p] = m[p], m[r]\n        m[r] = [v / m[r][c] for v in m[r]]\n        for i in range(len(m)):\n            if i != r and m[i][c] != 0:\n                f = m[i][c]\n                m[i] = [a - f * b for a, b in zip(m[i], m[r])]\n        r += 1\n    return m\n\nexp_rref = [[Fr(1), Fr(2), Fr(3)], [Fr(0), Fr(0), Fr(0)]]\nrec("I04", rref_fr([[Fr(1), Fr(2), Fr(3)], [Fr(2), Fr(4), Fr(6)]]) == exp_rref and\n    (Fr(1) * Fr(4) - Fr(2) * Fr(2)) == 0, ["independent Gauss-Jordan in Fractions", "rank-1 check: 2x2 minor determinant 0"])\n\n# ---------------------------------------------------------------- J\ndef ode_check(cid, residual, constants_sets, methods):\n    err = 0\n    for cs in constants_sets:\n        for p in [mp.mpf("-1.3"), mp.mpf("0.4"), mp.mpf("1.7")]:\n            err = max(err, abs(residual(p, *cs)))\n    rec(cid, err < mp.mpf(10) ** -20, methods, "max residual %s" % mp.nstr(err, 3))\n\node_check("J01", lambda x, C: mp.diff(lambda t: C * mp.exp(3 * t), x) - 3 * C * mp.exp(3 * x),\n          [(mp.mpf(c),) for c in ("1", "-2.5", "0.3")], ["substitute y=C*exp(3x) into y\'-3y=0 (mpmath.diff), 3 constants"])\node_check("J02", lambda x, C1, C2: mp.diff(lambda t: C1 * mp.cos(t) + C2 * mp.sin(t), x, 2) + (C1 * mp.cos(x) + C2 * mp.sin(x)),\n          [(mp.mpf(a), mp.mpf(b)) for a, b in (("1", "0"), ("0", "1"), ("2.2", "-0.7"))],\n          ["substitute y=C1 cos x + C2 sin x into y\'\'+y=0 (mpmath.diff order 2)"])\nR["J02"]["detail"] += "; Wronskian(cos,sin)=cos^2+sin^2=1 != 0 -> 2 independent solutions (second-order => general solution)"\nd = max(abs(mp.diff(lambda x, y=y0: mp.exp(x * y), x0) - y0 * mp.exp(x0 * y0)) for x0 in (mp.mpf("-1"), mp.mpf("0.5"), mp.mpf(2)) for y0 in (mp.mpf("-1.5"), mp.mpf(1), mp.mpf(2)))\nrec("J03", d < mp.mpf(10) ** -20, ["mpmath.diff in x with y fixed"], "err %s" % mp.nstr(d, 3))\nf2 = lambda x, y: x ** 2 + x * y + y ** 2\ng = 0\nfor x0 in (mp.mpf("-1"), mp.mpf("0.5"), mp.mpf(2)):\n    for y0 in (mp.mpf("-1.5"), mp.mpf(1), mp.mpf(2)):\n        g = max(g, abs(mp.diff(lambda t: f2(t, y0), x0) - (2 * x0 + y0)), abs(mp.diff(lambda t: f2(x0, t), y0) - (x0 + 2 * y0)))\nrec("J04", g < mp.mpf(10) ** -20, ["mpmath.diff of each partial vs expected components"], "err %s" % mp.nstr(g, 3))\n\n# ---------------------------------------------------------------- K\nidentity("K01", lambda x: 1 / (x * x - 1), lambda x: Fr(1, 2) / (x - 1) - Fr(1, 2) / (x + 1), avoid=(Fr(1), Fr(-1)))\nroots = mp.polyroots([1, 0, 0, -1])\nexpected_k02 = [mp.mpf(1), (-1 + 1j * mp.sqrt(3)) / 2, (-1 - 1j * mp.sqrt(3)) / 2]\nrec("K02", all(min(abs(r - e) for e in expected_k02) < mp.mpf(10) ** -30 for r in roots) and\n    all(abs(e ** 3 - 1) < mp.mpf(10) ** -30 for e in expected_k02) and len(roots) == 3,\n    ["mpmath.polyroots", "complex substitution of each expected root", "|r|=1 and 3 distinct cube roots of unity"])\nr3 = mp.findroot(lambda x: mp.cos(x) - x, 0.7)\n# independent bisection with plain floats\nlo, hi = 0.0, 1.0\nfor _ in range(200):\n    mid = (lo + hi) / 2\n    if (math.cos(lo) - lo) * (math.cos(mid) - mid) <= 0:\n        hi = mid\n    else:\n        lo = mid\nrec("K03", abs(r3 - mp.mpf("0.7390851332")) < 1e-10 and abs(lo - 0.7390851332) < 1e-10,\n    ["mpmath.findroot", "float bisection on [0,1]"], "root %s ; bisection %.12f" % (mp.nstr(r3, 15), lo))\nr4 = mp.findroot(lambda x: x ** 5 - x - 1, 1.2)\nlo, hi = 1.0, 2.0\nfor _ in range(200):\n    mid = (lo + hi) / 2\n    if (lo ** 5 - lo - 1) * (mid ** 5 - mid - 1) <= 0:\n        hi = mid\n    else:\n        lo = mid\nreal_roots = [z for z in mp.polyroots([1, 0, 0, 0, -1, -1], maxsteps=200, extraprec=100) if abs(z.imag) < 1e-20]\nrec("K04", abs(r4 - mp.mpf("1.167303978261")) < 1e-9 and abs(lo - 1.167303978261) < 1e-9 and len(real_roots) == 1,\n    ["mpmath.findroot", "float bisection on [1,2]", "mpmath.polyroots: exactly one real root"],\n    "root %s ; polyroots real count %d ; spec printed 1.167304015 differs by %s" % (mp.nstr(r4, 15), len(real_roots), mp.nstr(abs(r4 - mp.mpf("1.167304015")), 3)))\n\n# ---------------------------------------------------------------- L\ndef L01_f(x):\n    return (x * x - 1) / (x - 1)\n\ntry:\n    L01_f(Fr(1)); undef = False\nexcept ZeroDivisionError:\n    undef = True\nscan_roots = [Fr(k, 8) for k in range(-160, 161) if (Fr(k, 8) != 1 and L01_f(Fr(k, 8)) == 0)]\nrec("L01", undef and scan_roots == [Fr(-1)], ["original expression undefined at x=1 (ZeroDivisionError)",\n                                               "exhaustive scan of 321 rationals: only root is -1"])\ncand = [x for x in [Fr(k, 8) for k in range(-8, 160)] if (x + 1) >= 0 and abs(math.sqrt(x + 1) - (x - 1)) < 1e-12]\nrec("L02", cand == [Fr(3)] and math.isclose(math.sqrt(3 + 1), 3 - 1) and not math.isclose(math.sqrt(0 + 1), 0 - 1) and\n    -1 - 1 < 0, ["scan of 168 rationals in the domain x>=-1: only x=3 satisfies the ORIGINAL equation",\n                 "squaring candidate x=0 is extraneous (sqrt(1)=1 != -1)"],\n    "x^2-3x=0 -> {0,3}; 0 rejected on substitution")\nrec("L03", all(math.sqrt(x * x) == abs(x) for x in [-3.0, -0.5, 2.0, 7 / 3]), ["float check at negative/positive x"])\nund = True\nfor p in rat_points(6):\n    try:\n        1 / (p - p); und = False\n    except ZeroDivisionError:\n        pass\nrec("L04", und, ["denominator x-x is identically 0 -> expression undefined for every x (6 points, ZeroDivisionError)"])\n\n# ----------------------------------------------------------------\nids = [c["id"] for c in json.load(open(ROOT / "data" / "cases.json"))["cases"] if c.get("suite", "core_72") == "core_72"]\nmissing = [i for i in ids if i not in R]\nout = {"validated": len(R), "all_ok": all(v["ok"] for v in R.values()) and not missing, "missing": missing,\n       "failed": [k for k, v in R.items() if not v["ok"]], "checks": dict(sorted(R.items()))}\n(ROOT / "results").mkdir(exist_ok=True)\n(ROOT / "results" / "reference_validation.json").write_text(json.dumps(out, indent=1, default=str) + "\\n")\nprint("validated", len(R), "cases | all_ok =", out["all_ok"], "| failed:", out["failed"], "| missing:", missing)\nsys.exit(0 if out["all_ok"] else 1)\n')
W('bench/v2_reference_validation.py', '#!/usr/bin/env python3\n"""Independent symbolic validation for the 69 V2 stress-case reference answers.\n\nThis script runs before candidate solvers. It uses SymPy only as a reference oracle and\nfails closed: any stress case whose reference cannot be validated aborts the notebook run.\n"""\nimport json\nimport pathlib\nimport sys\nimport traceback\n\nimport sympy as sp\n\nROOT = pathlib.Path(__file__).resolve().parent.parent\nX, Y = sp.symbols("x y", real=True)\nC, C1, C2 = sp.symbols("C C1 C2")\nFN = sp.Function("y")\nTOL = 1e-7\n\ndef P(s, extra=None):\n    loc = {"x": X, "y": Y, "I": sp.I, "i": sp.I, "E": sp.E, "e": sp.E,\n           "pi": sp.pi, "ln": sp.log, "log": sp.log, "abs": sp.Abs,\n           "C": C, "C1": C1, "C2": C2, "y": Y}\n    if extra:\n        loc.update(extra)\n    return sp.sympify(s, locals=loc)\n\ndef near_zero(v, tol=TOL):\n    try:\n        z = complex(sp.N(v, 40))\n        return abs(z) <= tol\n    except Exception:\n        try:\n            return bool(sp.simplify(v) == 0)\n        except Exception:\n            return False\n\ndef sets_equal(a, b):\n    try:\n        if a == b:\n            return True\n        return sp.simplify(a.symmetric_difference(b)) == sp.EmptySet\n    except Exception:\n        return False\n\ndef expected_set(exp, domain):\n    if exp["kind"] == "empty":\n        return sp.EmptySet\n    if exp["kind"] == "all":\n        return sp.S.Reals if domain == "real" else sp.S.Complexes\n    if exp["kind"] == "finite":\n        return sp.FiniteSet(*(P(v) for v in exp["values"]))\n    raise ValueError("unsupported expected solution kind: " + str(exp.get("kind")))\n\ndef validate(c):\n    op, pb, exp = c["operation"], c["problem"], c["expected"]\n    x = X\n    if op in ("evaluate", "simplify", "expand", "factor"):\n        inp, want = P(pb["expr"]), P(exp["expr"])\n        if not near_zero(sp.simplify(inp - want)):\n            return False, "expression reference is not equivalent to input"\n        return True, "symbolic expression equivalence"\n    if op in ("diff", "partial_derivative"):\n        var = P(pb["var"])\n        got = sp.diff(P(pb["expr"]), var)\n        return near_zero(sp.simplify(got - P(exp["expr"]))), "symbolic derivative identity"\n    if op == "gradient":\n        got = [sp.diff(P(pb["expr"]), P(v)) for v in pb["vars"]]\n        want = [P(v) for v in exp["vector"]]\n        return len(got) == len(want) and all(near_zero(sp.simplify(a-b)) for a,b in zip(got,want)), "gradient components"\n    if op == "integrate":\n        integrand = P(pb["expr"])\n        if "lower" in pb:\n            got = sp.integrate(integrand, (x, P(pb["lower"]), P(pb["upper"])))\n            return near_zero(sp.simplify(got - P(exp["expr"]))), "definite integral evaluated independently"\n        anti = P(exp["expr"])\n        residual = sp.diff(anti, x) - integrand\n        # Generic logarithmic antiderivatives can have a Piecewise singularity at an excluded point;\n        # verify derivative identities away from every pole on both sides instead of evaluating there.\n        if near_zero(sp.simplify(residual)):\n            return True, "symbolic antiderivative derivative"\n        pts = c.get("comparison", {}).get("require_real_on") or ["-4", "-1", "1/2", "4"]\n        for t in pts:\n            p = P(t)\n            try:\n                if not near_zero(sp.N(residual.subs(x, p), 35), 1e-6):\n                    return False, "antiderivative derivative mismatch at x=%s" % t\n            except Exception:\n                return False, "antiderivative derivative could not be evaluated at x=%s" % t\n        return True, "numeric derivative checks away from poles (including both signs)"\n    if op == "limit":\n        got = sp.limit(P(pb["expr"]), x, P(pb["point"]), dir="+-")\n        return near_zero(sp.simplify(got - P(exp["expr"]))), "two-sided limit"\n    if op == "solve":\n        lhs, rhs = [P(t) for t in pb["equation"].split("=", 1)]\n        domain = sp.S.Reals if c["domain"] == "real" else sp.S.Complexes\n        if exp["kind"] == "all":\n            # Required restrictions, if any, are checked as explicit benchmark metadata below.\n            return near_zero(sp.simplify(lhs-rhs)), "identity equation"\n        if exp["kind"] == "undefined":\n            return bool((lhs.has(sp.zoo, sp.nan) or sp.simplify(lhs-rhs).has(sp.zoo, sp.nan)) or\n                        any("x - x" in t.replace(" ", "") for t in [pb["equation"]])), "identically undefined expression"\n        want = expected_set(exp, c["domain"])\n        got = sp.solveset(lhs - rhs, x, domain=domain)\n        # SymPy may simplify an identity through a removable singularity; for finite/empty cases\n        # solveset preserves exclusions for the test family being used here.\n        return sets_equal(got, want), "solveset reference %s; expected %s" % (got, want)\n    if op == "solve_system":\n        eqs = []\n        for s in pb["equations"]:\n            lhs, rhs = s.split("=", 1)\n            eqs.append(P(lhs) - P(rhs))\n        if exp["kind"] == "empty":\n            sol = sp.solve(eqs, [X,Y], dict=True)\n            # A contradictory system must yield no assignments.\n            return not sol, "inconsistent linear system"\n        if exp["kind"] == "unique":\n            assignment = {X: P(exp["assignments"]["x"]), Y: P(exp["assignments"]["y"])}\n            ok = all(near_zero(eq.subs(assignment)) for eq in eqs)\n            return ok, "all system equations satisfied by the declared unique point"\n        return False, "unexpected system expectation"\n    if op == "solve_inequality":\n        rel = P(pb["inequality"])\n        got = sp.solve_univariate_inequality(rel, x, relational=False)\n        intervals = []\n        for lo, hi, left_open, right_open in exp["intervals"]:\n            intervals.append(sp.Interval(P(lo), P(hi), left_open=left_open, right_open=right_open))\n        want = sp.Union(*intervals)\n        return sets_equal(got, want), "inequality set %s; expected %s" % (got, want)\n    if op in ("det", "inverse", "rref"):\n        M = sp.Matrix(pb["matrix"])\n        if op == "det":\n            return near_zero(M.det() - P(exp["expr"])), "matrix determinant"\n        if op == "inverse":\n            got = M.inv()\n        else:\n            got = M.rref()[0]\n        want = sp.Matrix([[P(v) for v in row] for row in exp["matrix"]])\n        return got.shape == want.shape and (got-want).applyfunc(sp.simplify) == sp.zeros(*want.shape), "matrix result"\n    if op == "dsolve":\n        yfun = FN(x)\n        ode = P(pb["ode"], {"y": FN})\n        rhs = P(exp["family"])\n        residual = sp.simplify(ode.subs(yfun, rhs).doit())\n        constants = [s for s in rhs.free_symbols if s.name in ("C", "C1", "C2")]\n        return near_zero(residual) and len(constants) == exp["n_constants"], "ODE residual and integration-constant count"\n    if op == "partial_fractions":\n        got = P(pb["expr"])\n        want = P(exp["expr"])\n        return near_zero(sp.simplify(got-want)), "partial-fraction identity"\n    if op == "nsolve":\n        lhs, rhs = [P(t) for t in pb["equation"].split("=", 1)]\n        for r in exp["values"]:\n            root = P(r)\n            residual = (lhs-rhs).subs(x, root)\n            if not near_zero(residual, 1e-6):\n                return False, "numeric root %s has residual %s" % (r, sp.N(residual, 12))\n        return True, "declared numeric root has small residual"\n    return False, "validator has no rule for operation %r" % op\n\ndef main():\n    data = json.loads((ROOT / "data/cases.json").read_text())\n    cases = [c for c in data["cases"] if c.get("suite") == "v2_stress"]\n    rows, failed = [], []\n    for c in cases:\n        try:\n            ok, detail = validate(c)\n        except Exception as e:\n            ok, detail = False, "%s: %s" % (type(e).__name__, str(e))\n        row = {"id": c["id"], "operation": c["operation"], "ok": bool(ok), "detail": str(detail)}\n        rows.append(row)\n        if not ok: failed.append(row)\n    out = {"total": len(rows), "validated": sum(r["ok"] for r in rows), "all_ok": not failed,\n           "failed": failed, "rows": rows}\n    dest = ROOT / "results/v2_reference_validation.json"\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    dest.write_text(json.dumps(out, indent=2) + "\\n")\n    print("V2 stress references: %d/%d validated; all_ok=%s" % (out["validated"], out["total"], out["all_ok"]))\n    for row in failed:\n        print("FAIL %s (%s): %s" % (row["id"], row["operation"], row["detail"]))\n    if len(cases) != 69 or failed:\n        raise SystemExit(1)\n\nif __name__ == "__main__":\n    main()\n')
print("ok: wrote 5 V2 benchmark source files")


ok: wrote 5 V2 benchmark source files


In [6]:
W('bench/evaluate.py', '#!/usr/bin/env python3\n"""raw replies (results/raw/*.json) + data/cases.json  ->  results/raw_results.json / .csv  (one row per solver per case).\n\nRe-runnable without re-running any solver.  Usage: python -m bench.evaluate [--raw results/raw] [--out results]\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport pathlib\nimport statistics\nimport subprocess\n\ntry:\n    from . import derivations as D\n    from . import judge as J\nexcept ImportError:\n    import derivations as D\n    import judge as J\n\nROOT = pathlib.Path(__file__).resolve().parent.parent\nORDER = ["sympy-direct", "smartsolver", "stepsolver"]\n\n\ndef load_evidence():\n    """results/evidence/candidates.json is written by the setup step (commit, url, ref, licence, dependency count, history)."""\n    p = ROOT / "results/evidence/candidates.json"\n    return json.load(open(p)) if p.exists() else {}\n\n\ndef git_commit(name, evidence):\n    return (evidence.get(name) or {}).get("commit")\n\n\ndef fingerprint(reply):\n    """Identity of the derivation + answer a manual review was written for (a changed engine output invalidates the review)."""\n    keep = {"steps": reply.get("steps"), "answer": reply.get("answer"), "outcome": reply.get("outcome")}\n    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]\n\n\ndef pct(sorted_vals, q):\n    if not sorted_vals:\n        return None\n    k = (len(sorted_vals) - 1) * q\n    f, c = int(k), min(int(k) + 1, len(sorted_vals) - 1)\n    return sorted_vals[f] + (sorted_vals[c] - sorted_vals[f]) * (k - f)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--raw", default=str(ROOT / "results/raw"))\n    ap.add_argument("--out", default=str(ROOT / "results"))\n    a = ap.parse_args()\n    cases = json.load(open(ROOT / "data/cases.json"))["cases"]\n    mr_path = ROOT / "data/manual_reviews.json"\n    manual = json.load(open(mr_path)) if mr_path.exists() else {}\n    rows = []\n    meta = {}\n    evidence = load_evidence()\n    stale_reviews = []\n    for name in ORDER:\n        p = pathlib.Path(a.raw) / ("%s.json" % name)\n        if not p.exists():\n            continue\n        raw = json.load(open(p))\n        info = raw["info"] or {}\n        meta[name] = {"version": info.get("version"), "commit": git_commit(name, evidence), "package": info.get("package"),\n                      "python": info.get("python"), "sympy_version": info.get("sympy_version") or info.get("version") if name == "sympy-direct" else info.get("sympy_version"),\n                      "cold_start": raw["cold_start"], "maxrss_kb": raw["maxrss_kb"], "timeouts": raw["timeouts"], "crashes": raw["crashes"],\n                      "protocol": raw["protocol"], "public_methods": info.get("public_methods"), "operations": info.get("operations")}\n        for c in cases:\n            rr = raw["rows"].get(c["id"])\n            if rr is None:\n                rows.append({"solver": name, "case_id": c["id"], "status": "skipped", "skip_reason": "case not selected in this run"})\n                continue\n            reply = rr["reply"]\n            verdict = J.judge(c, reply)\n            deriv = D.automated_score(c, reply, verdict)\n            review = manual.get("%s|%s" % (name, c["id"]))\n            if review is not None and review.get("fingerprint") not in (None, fingerprint(reply)):\n                stale_reviews.append("%s|%s" % (name, c["id"]))      # engine output changed since the review: NOT applied\n                review = None\n            deriv = D.apply_manual(deriv, review)\n            warm = sorted(t for t in rr.get("warm_times_ms", []) if t is not None)\n            rows.append({\n                "solver": name, "solver_version": meta[name]["version"], "solver_commit": meta[name]["commit"],\n                "case_id": c["id"], "category": c["category"], "operation": c["operation"], "display_input": c["display_input"],\n                "normalised_operation": c["operation"], "problem": c["problem"], "domain": c["domain"], "domain_sensitive": c["domain_sensitive"],\n                "adapter_api_call": reply.get("api_call"), "adapter_input": reply.get("input_sent"),\n                "adapter_overrides": bool(reply.get("input_overrides_used")), "domain_assumption_passed": reply.get("domain_assumption_passed"),\n                "raw_output": reply.get("raw_output"), "raw_answer_canonical": reply.get("answer"), "raw_steps": reply.get("steps"),\n                "native_status": reply.get("native_status"), "native_flags": reply.get("native_flags"),\n                "restrictions_reported": reply.get("restrictions_reported"),\n                "expected_spec": c["spec_expected"], "expected_reference": c["expected"], "actual": verdict["actual"],\n                "status": verdict["status"], "status_detail": verdict["sub"], "value_correct": verdict["value_correct"],\n                "form_ok": verdict["form_ok"], "form_note": verdict["form_note"], "domain_ok": verdict["domain_ok"],\n                "domain_check": verdict["domain_check"], "answer_produced": verdict["answer_produced"], "covered": verdict["covered"],\n                "strict_correct": verdict["strict_correct"], "value_only_correct": verdict["value_only_correct"],\n                "equivalence_proof": verdict["proof"], "judge_detail": verdict["detail"], "constant_shown": verdict["constant_shown"],\n                "derivation": deriv, "unsupported_reason": reply.get("unsupported_reason"), "error": reply.get("error"),\n                "time_first_call_ms": reply.get("time_ms"), "wall_ms": rr.get("wall_ms"), "warm_times_ms": warm,\n                "warm_median_ms": statistics.median(warm) if warm else None, "repeatable": rr.get("repeatable"),\n                "timeout": reply.get("outcome") == "timeout", "manual_review_flag": verdict["manual_review"],\n                "manual_review_reasons": verdict["manual_reasons"],\n            })\n    out = pathlib.Path(a.out)\n    out.mkdir(exist_ok=True)\n    (out / "raw_results.json").write_text(json.dumps({"meta": meta, "rows": rows, "stale_manual_reviews": stale_reviews,\n                                                       "manual_reviews_total": len(manual)}, indent=1, default=str))\n    if stale_reviews:\n        print("manual reviews NOT applied (engine output changed since review): %d" % len(stale_reviews))\n    cols = ["solver", "solver_version", "solver_commit", "case_id", "category", "operation", "display_input", "status", "status_detail",\n            "strict_correct", "value_only_correct", "covered", "form_ok", "domain_ok", "expected_spec", "actual", "derivation_available",\n            "derivation_score_auto", "derivation_score_final", "time_first_call_ms", "warm_median_ms", "timeout", "error_type",\n            "manual_review_flag"]\n    with open(out / "raw_results.csv", "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=cols)\n        w.writeheader()\n        for r in rows:\n            d = r.get("derivation") or {}\n            w.writerow({**{k: r.get(k) for k in cols if k in r}, "derivation_available": d.get("available"),\n                        "derivation_score_auto": d.get("automated_total"), "derivation_score_final": d.get("final_total"),\n                        "error_type": (r.get("error") or {}).get("type")})\n    print("rows:", len(rows), "->", out / "raw_results.json")\n    for name in ORDER:\n        rs = [r for r in rows if r["solver"] == name]\n        if rs:\n            from collections import Counter\n            print("%-13s" % name, dict(Counter(r["status"] for r in rs)), "strict_correct=%d covered=%d" % (sum(1 for r in rs if r.get("strict_correct")), sum(1 for r in rs if r.get("covered"))))\n\n\nif __name__ == "__main__":\n    main()\n')
W('bench/engineering.py', '#!/usr/bin/env python3\n"""Engineering / reliability probes (kept separate from mathematical failures).  Each probe goes through the same isolated worker\nprotocol (own venv, killed on timeout).  Benign canary only: math.factorial(4) -- proves evaluation of input as code without side effects.\nUsage: python -m bench.engineering [--out results/engineering_results.json]"""\nimport argparse, json, pathlib, re, time\ntry:\n    from .runner import Worker, WORKERS, ROOT\nexcept ImportError:\n    from runner import Worker, WORKERS, ROOT\n\nCANARY = "x - __import__(\'math\').factorial(4) = 0"\n\n\ndef probes(solver):\n    """-> {probe_id: (description, request)}; None = not applicable to this solver\'s interface."""\n    def sym(kind, **k): return {"cmd": "raw", "probe": {"kind": kind, **k}}\n    def smart(method, *args): return {"cmd": "raw", "probe": {"kind": "call", "method": method, "args": list(args)}}\n    def step(q): return {"cmd": "raw", "probe": {"kind": "query", "query": q}}\n    P = {}\n    if solver == "sympy-direct":\n        P["malformed_syntax"] = sym("parse", text="2 +* 3")\n        P["unbalanced_parens"] = sym("parse", text="(x + 1")\n        P["empty_input"] = sym("eval", text="")\n        P["zero_denominator"] = sym("eval", text="1/0")\n        P["missing_variable"] = sym("solve_for", equation="x + 1 = 0", var="z")\n        P["two_variable_parametric"] = sym("solve_for", equation="x*y = 6", var="x")\n        P["unsupported_operation"] = None\n        P["heavy_equation"] = sym("solve_for", equation="x^25 - x - 1 = 0", var="x")\n        P["large_polynomial_equation"] = sym("solve_for", equation="(x + 1)^60 = 0", var="x")\n        P["untrusted_input_canary"] = sym("canary", text="__import__(\'math\').factorial(4)")\n    elif solver == "smartsolver":\n        P["malformed_syntax"] = smart("solve_equation", "2 +* 3 = x", "x")\n        P["unbalanced_parens"] = smart("solve_equation", "(x + 1 = 2", "x")\n        P["empty_input"] = smart("solve_equation", "", "x")\n        P["zero_denominator"] = smart("solve_equation", "1/0 = x", "x")\n        P["missing_variable"] = smart("solve_equation", "x + 1 = 0", "z")\n        P["two_variable_parametric"] = smart("solve_equation", "x*y = 6", "x")\n        P["unsupported_operation"] = None\n        P["heavy_equation"] = smart("solve_equation", "x**25 - x - 1 = 0", "x")\n        P["large_polynomial_equation"] = smart("solve_equation", "(x + 1)**60 = 0", "x")\n        P["untrusted_input_canary"] = smart("solve_equation", CANARY, "x")\n    else:\n        P["malformed_syntax"] = step("solve(2 +* 3 = x, x)")\n        P["unbalanced_parens"] = step("solve((x + 1 = 2, x)")\n        P["empty_input"] = step("")\n        P["zero_denominator"] = step("simplify(1/0)")\n        P["missing_variable"] = step("solve(x + 1 = 0, z)")\n        P["two_variable_parametric"] = step("solve(x*y = 6, x)")\n        P["unsupported_operation"] = step("frobnicate(x)")\n        P["heavy_equation"] = step("solve(x^25 - x - 1 = 0, x)")\n        P["large_polynomial_equation"] = step("solve((x + 1)^60 = 0, x)")\n        P["untrusted_input_canary"] = step("solve(" + CANARY + ", x)")\n    return P\n\n\ndef score(pid, solver, r, status):\n    """-> (score in {0,0.5,1,None}, note).  None = not applicable."""\n    err = r.get("error") or {}\n    oc = r.get("outcome")\n    txt = json.dumps(r.get("raw_output", "")) if r.get("raw_output") is not None else ""   # full text: matches must not miss a truncated tail\n    if status != "ok":\n        return 0.0, "worker %s (hang/crash) - needed external kill" % status\n    if pid in ("malformed_syntax", "unbalanced_parens", "empty_input"):\n        if oc == "answer":\n            return 0.0, "returned an answer for malformed input: %s" % txt[:100]\n        return (1.0, "library-typed error %s" % err.get("type")) if err.get("contract_exception") else (0.5, "raw Python exception %s (caught by adapter)" % err.get("type"))\n    ns_ = r.get("native_status")\n    if pid == "zero_denominator":\n        if ns_ == "UndefinedResult":\n            return 1.0, "explicit UndefinedResult"\n        if ns_ == "UnsolvedResult":\n            return 0.5, "honest decline (UnsolvedResult), no fabricated value"\n        if oc == "error" and re.search(r"zero|division|undefined|zoo|domain", err.get("type", "") + err.get("message", ""), re.I):\n            return 1.0, "explicit error: %s" % err.get("type")\n        if oc == "answer" and re.search(r"undefined", txt, re.I):\n            return 1.0, "explicit undefined result"\n        if oc == "answer" and "zoo" in txt:\n            return 0.5, "returns \'zoo\' (complex infinity) without an undefined/domain message"\n        if oc == "answer" and re.search(r"no solution|\\[\\]|EmptySet|empty", txt, re.I):\n            return 0.5, "silently reports no solution"\n        return 0.0, "outcome=%s %s" % (oc, (err.get("message") or txt)[:100])\n    if pid == "missing_variable":\n        if oc == "error":\n            return 1.0, "error: %s" % err.get("type")\n        if oc == "answer" and not re.search(r"-1", txt):\n            return 0.5, "answer without fabricating a root: %s" % txt[:100]\n        return 0.0, "fabricated/unclear result: %s" % txt[:100]\n    if pid == "two_variable_parametric":\n        if oc == "answer" and re.search(r"6\\s*/\\s*y|6 / y|\\{6/y\\}|y\\*\\*\\(-1\\)", txt.replace(" ", "")) or (oc == "answer" and "6/y" in txt.replace(" ", "")):\n            return 1.0, "parametric solution 6/y"\n        if oc == "answer" and "6 / y" in txt:\n            return 1.0, "parametric solution 6/y"\n        return (0.5, "honest error/unsolved: %s" % (err.get("type") or txt[:80])) if oc in ("error", "unsupported") else (0.0, "wrong/absent parametric solution: %s" % txt[:100])\n    if pid == "unsupported_operation":\n        if ns_ == "ExactResult":\n            return 0.0, "unknown call \'frobnicate(x)\' echoed back as a Verified ExactResult (silent acceptance of unknown functions)"\n        if ns_ == "UnsolvedResult":\n            return 0.5, "honest decline (UnsolvedResult)"\n        return (1.0, "typed error %s" % err.get("type")) if err.get("contract_exception") else (0.5, "raw exception %s" % err.get("type"))\n    if pid == "large_polynomial_equation":\n        ok = oc == "answer" and re.search(r"-1\\b", txt) is not None\n        return (1.0, "correct root -1 in %.0f ms" % r.get("time_ms", -1)) if ok else (0.0, "outcome=%s %s" % (oc, (err.get("message") or txt)[:100]))\n    if pid == "heavy_equation":\n        return 1.0, "completed in %.0f ms (%s)" % (r.get("time_ms", -1), r.get("native_status") or oc)\n    if pid == "untrusted_input_canary":\n        executed = bool(re.search(r"\\b24\\b", txt)) or (oc == "error" and re.search(r"unsupported operand.*str|\'str\'", err.get("message", "")))\n        return (0.0, "INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): %s" % (txt[:80] or err.get("message", "")[:80])) if executed else (1.0, "payload rejected: %s" % (err.get("type") or txt[:60]))\n    return None, ""\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", default=str(ROOT / "results/engineering_results.json"))\n    a = ap.parse_args()\n    out = {}\n    ev_path = ROOT / "results/evidence/candidates.json"\n    evidence = json.loads(ev_path.read_text()) if ev_path.exists() else {}\n    present = [s for s in WORKERS if (ROOT / "envs" / WORKERS[s][0] / "bin" / "python").exists() and (evidence.get(s, {}).get("installed", True))]\n    for s in present:\n        res = {}\n        w = Worker(s)\n        for pid, req in probes(s).items():\n            if req is None:\n                res[pid] = {"score": None, "note": "not applicable: interface has no operation-name dispatch", "reply": None}\n                continue\n            timeout = 15 if pid in ("heavy_equation", "large_polynomial_equation") else 20\n            t = time.perf_counter()\n            r, st = w.call({**req, "req_id": pid}, timeout)\n            wall = (time.perf_counter() - t) * 1000\n            if st != "ok":\n                sc, note = 0.0, "worker %s after %ss - external kill needed" % (st, timeout)\n                w.kill(); w = Worker(s); r = {"outcome": st}\n            else:\n                sc, note = score(pid, s, r, st)\n            res[pid] = {"score": sc, "note": note, "wall_ms": wall, "reply": r}\n        w.close()\n        out[s] = res\n    # static evidence: network/process imports in candidate library code (offline claim)\n    import subprocess\n    ev = {}\n    for name in [n for n in present if n != "sympy-direct"]:\n        path = evidence.get(name, {}).get("source_dir")\n        if not path:\n            continue\n        g = subprocess.run("grep -rnE \'^\\\\s*(import|from)\\\\s+(requests|urllib|socket|httpx|http\\\\.client|aiohttp|subprocess|os\\\\.system)\' %s --include=*.py | grep -v \'/tests/\' | head -20" % path,\n                           shell=True, capture_output=True, text=True).stdout.strip().splitlines()\n        ev[name] = g\n    out["_static_network_imports"] = ev\n    pathlib.Path(a.out).write_text(json.dumps(out, indent=1, default=str))\n    for s in present:\n        print(s, {k: v["score"] for k, v in out[s].items()})\n\n\nif __name__ == "__main__":\n    main()\n')
W('bench/score.py', '#!/usr/bin/env python3\n"""results/raw_results.json + engineering_results.json -> results/scores.json   (all rules documented in docs/SCORING_RULES.md)"""\nimport json\nimport math\nimport pathlib\nimport statistics\nfrom collections import Counter, defaultdict\n\nROOT = pathlib.Path(__file__).resolve().parent.parent\nALL_SOLVERS = ["sympy-direct", "smartsolver", "stepsolver"]\nW = dict(correctness=0.30, derivations=0.30, coverage=0.20, reliability=0.10, integration=0.05, performance=0.05)\n\n# ---- Integration & maintainability rubric (14 points). Every score is computed from evidence collected by the setup step\n# (results/evidence/candidates.json) or measured in this run; the two fixed assumptions are labelled as such.\ndef integration_rubric(solver, ev, override_share):\n    ev = ev or {}\n    crit = {}\n    deps = ev.get("runtime_deps")\n    if not ev.get("installed", False):\n        crit["env_compat"] = (0, "install failed: %s" % ev.get("install_error"))\n        crit["install"] = (0, "install failed")\n    else:\n        crit["env_compat"] = ((2 if deps <= 4 else 1 if deps <= 30 else 0),\n                              "%s third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python %s" % (deps, ev.get("python_requires")))\n        crit["install"] = ((1 if ev.get("sympy_pin_fallback") else 2), "single install command succeeded%s" % ("; needed an unpinned-SymPy fallback" if ev.get("sympy_pin_fallback") else ""))\n    crit["adaptation"] = ((2 if override_share <= 0.02 else 1 if override_share <= 0.10 else 0),\n                          "%.1f%% of cases needed per-case adapter handling (2 if <=2%%, 1 if <=10%%, else 0)" % (100 * override_share))\n    if solver == "sympy-direct":\n        crit["testability_extensibility"] = (0, "FIXED ASSUMPTION: bare engine has no derivation/step framework to test or extend")\n    else:\n        n = (ev.get("tests") or {}).get("static_test_functions", 0)\n        own = (ev.get("tests") or {}).get("own_suite") or {}\n        pts = 2 if n >= 100 else 1 if n >= 1 else 0\n        note = "%d test functions found statically (2 if >=100, 1 if >=1)" % n\n        if own.get("run"):\n            note += "; own suite run: %s passed / %s failed" % (own.get("passed"), own.get("failed"))\n            if own.get("failed"):\n                pts = max(0, pts - 1)\n        crit["testability_extensibility"] = (pts, note)\n    cls = ev.get("licence_class", "unknown")\n    crit["licence"] = ({"permissive": 2, "unknown": 1, "copyleft": 0}[cls], "licence %s (%s)" % (ev.get("licence"), cls))\n    if solver == "sympy-direct":\n        crit["maintenance"] = (2, "FIXED ASSUMPTION: mature, widely maintained project (not measurable from a PyPI install)")\n    else:\n        h = ev.get("history") or {}\n        if h.get("age_days", 0) >= 730 and h.get("authors", 0) >= 5:\n            m = 2\n        elif h.get("commits", 0) >= 5 and h.get("days_since_last_commit", 10 ** 6) <= 365:\n            m = 1\n        else:\n            m = 0\n        crit["maintenance"] = (m, "history: %s commits, %s authors, age %s days, last commit %s days ago, %s tags (2 if >=2y and >=5 authors; 1 if >=5 commits and active within 1y)" %\n                               (h.get("commits"), h.get("authors"), h.get("age_days"), h.get("days_since_last_commit"), h.get("tags")))\n    crit["ui_independent"] = (1, "FIXED ASSUMPTION: used headless as a library in this benchmark")\n    crit["no_rewrite"] = (1, "FIXED ASSUMPTION: plain library call; adapter-sized integration")\n    return crit\n\n\ndef lat_score(ms):\n    """100 at <=10 ms, 0 at >=2000 ms, log-linear in between."""\n    if ms is None:\n        return 0.0\n    if ms <= 10:\n        return 100.0\n    if ms >= 2000:\n        return 0.0\n    return 100.0 * math.log10(2000.0 / ms) / math.log10(200.0)\n\n\ndef pctl(vals, q):\n    v = sorted(vals)\n    k = (len(v) - 1) * q\n    f, c = int(k), min(int(k) + 1, len(v) - 1)\n    return v[f] + (v[c] - v[f]) * (k - f)\n\n\ndef main():\n    d = json.load(open(ROOT / "results/raw_results.json"))\n    eng = json.load(open(ROOT / "results/engineering_results.json"))\n    rows, meta = d["rows"], d["meta"]\n    SOLVERS = [s for s in ALL_SOLVERS if s in meta]\n    evidence = json.load(open(ROOT / "results/evidence/candidates.json")) if (ROOT / "results/evidence/candidates.json").exists() else {}\n    by = {s: [r for r in rows if r["solver"] == s] for s in SOLVERS}\n    n_app = len(by[SOLVERS[0]])\n    out = {"applicable_cases": n_app, "weights": W, "solvers": {}, "solver_order": SOLVERS}\n\n    # common set for fair latency comparison: cases where ALL solvers produced an answer\n    answered = {s: {r["case_id"] for r in by[s] if r["answer_produced"] and r["warm_times_ms"]} for s in SOLVERS}\n    common = sorted(set.intersection(*answered.values()))\n\n    for s in SOLVERS:\n        rs = by[s]\n        cnt = Counter(r["status"] for r in rs)\n        sub = Counter((r["status"], r["status_detail"].split(" ")[0] if r["status"] in ("unsupported", "invalid_answer") else "") for r in rs)\n        strict = sum(1 for r in rs if r["strict_correct"])\n        value_only = sum(1 for r in rs if r["value_only_correct"])\n        value_nofrm = sum(1 for r in rs if r["value_correct"])\n        covered = sum(1 for r in rs if r["covered"])\n        attempted = [r for r in rs if not (r["status"] == "unsupported" and r["status_detail"] == "no_api")]\n        crashes = sum(1 for r in attempted if r["status"] in ("runtime_error", "timeout", "infra_error"))\n        dsens = [r for r in rs if r["domain_sensitive"]]\n        dsens_ok = sum(1 for r in dsens if r["strict_correct"])\n        rep = [r["repeatable"] for r in rs if r["repeatable"] is not None]\n        eng_scores = [v["score"] for k, v in eng[s].items() if v["score"] is not None]\n        canary = eng[s]["untrusted_input_canary"]["score"]\n        R = dict(R1_repeatability=(sum(1 for x in rep if x) / len(rep)) if rep else 0.0,\n                 R2_engineering_probes=statistics.mean(eng_scores),\n                 R3_stability=1 - crashes / max(1, len(attempted)),\n                 R4_domain_edge_cases=dsens_ok / len(dsens),\n                 R5_untrusted_input_safety=canary)\n        reliability = 100 * statistics.mean(R.values())\n        # derivations\n        scored = [r for r in rs if r["derivation"].get("scored")]\n        avail = [r for r in rs if r["derivation"].get("available")]\n        mean_final = statistics.mean(r["derivation"]["final_total"] for r in scored) if scored else 0.0\n        mean_auto = statistics.mean(r["derivation"]["automated_total"] for r in scored) if scored else 0.0\n        deriv_primary = mean_final * 10\n        deriv_penalised = 10 * sum(r["derivation"]["final_total"] for r in scored) / n_app          # unavailable = 0 over ALL applicable cases\n        dsens_scored = [r for r in scored if r["domain_sensitive"]]\n        crit = defaultdict(list)\n        for r in scored:\n            for k, v in r["derivation"]["final"].items():\n                crit[k].append(v)\n        # performance\n        warm_by_case = {r["case_id"]: r["warm_median_ms"] for r in rs if r["case_id"] in common}\n        all_warm = [t for r in rs if r["case_id"] in common for t in r["warm_times_ms"]]\n        med = statistics.median(warm_by_case.values()) if warm_by_case else None\n        p95 = pctl(all_warm, 0.95) if all_warm else None\n        own_med = statistics.median([r["warm_median_ms"] for r in rs if r["warm_median_ms"] is not None]) if any(r["warm_median_ms"] for r in rs) else None\n        own_all = [t for r in rs for t in r["warm_times_ms"]]\n        own_p95 = pctl(own_all, 0.95) if own_all else None\n        fail_rate = (meta[s]["timeouts"] + meta[s]["crashes"]) / max(1, len(attempted))\n        perf = 0.4 * lat_score(med) + 0.4 * lat_score(p95) + 0.2 * 100 * (1 - fail_rate)\n        # integration\n        override_share = sum(1 for r in rs if r["adapter_overrides"]) / n_app\n        integ = integration_rubric(s, evidence.get(s), override_share)\n        integ_pts = sum(v[0] for v in integ.values())\n        integration = 100 * integ_pts / 14\n        comp = dict(correctness=100 * strict / n_app, derivations=deriv_primary, coverage=100 * covered / n_app, reliability=reliability,\n                    integration=integration, performance=perf)\n        overall = sum(W[k] * comp[k] for k in W)\n        # sensitivity variants\n        def total(c):\n            return sum(W[k] * c[k] for k in W)\n        variants = {\n            "primary (spec formula)": overall,\n            "value-only correctness (no domain-reporting requirement)": total({**comp, "correctness": 100 * value_only / n_app}),\n            "value correctness ignoring form rules too": total({**comp, "correctness": 100 * value_nofrm / n_app}),\n            "availability-penalised derivations (unavailable=0 over all benchmark cases)": total({**comp, "derivations": deriv_penalised}),\n            "automated-only derivation scores (no manual overlay)": total({**comp, "derivations": mean_auto * 10}),\n            "latency on each solver\'s own supported cases": total({**comp, "performance": 0.4 * lat_score(own_med) + 0.4 * lat_score(own_p95) + 0.2 * 100 * (1 - fail_rate)}),\n        }\n        # per-category\n        cats = {}\n        for c in sorted({r["category"] for r in rs}):\n            cr = [r for r in rs if r["category"] == c]\n            cats[c] = {"n": len(cr), "strict_correct": sum(1 for r in cr if r["strict_correct"]), "covered": sum(1 for r in cr if r["covered"]),\n                       "unsupported": sum(1 for r in cr if r["status"] == "unsupported"), "invalid": sum(1 for r in cr if r["status"] == "invalid_answer"),\n                       "errors": sum(1 for r in cr if r["status"] in ("runtime_error", "timeout", "infra_error", "input_incompatible"))}\n        # per-operation timing (common set)\n        ops = defaultdict(list)\n        for r in rs:\n            if r["case_id"] in common and r["warm_median_ms"] is not None:\n                ops[r["operation"]].append(r["warm_median_ms"])\n        cs = meta[s]["cold_start"]\n        out["solvers"][s] = {\n            "version": meta[s]["version"], "commit": meta[s]["commit"], "status_counts": dict(cnt),\n            "defined": n_app, "executed": n_app, "skipped": cnt.get("skipped", 0), "passed(strict_correct)": strict, "failed(invalid+error)": cnt["invalid_answer"] + cnt["runtime_error"] + cnt["timeout"] + cnt["infra_error"] + cnt["input_incompatible"],\n            "unsupported": cnt["unsupported"], "unsupported_no_api": sum(1 for r in rs if r["status"] == "unsupported" and r["status_detail"] == "no_api"),\n            "unsolved_by_solver": sum(1 for r in rs if r["status"] == "unsupported" and r["status_detail"] != "no_api"),\n            "value_only_correct": value_only, "covered": covered, "attempted": len(attempted), "crashes_timeouts": crashes,\n            "reliability_metrics": R, "domain_sensitive_cases": len(dsens), "domain_sensitive_strict_correct": dsens_ok,\n            "derivation": {"available": len(avail), "available_pct_of_applicable": 100 * len(avail) / n_app, "scored": len(scored),\n                           "mean_automated": mean_auto, "mean_final": mean_final, "mean_final_domain_sensitive": statistics.mean(r["derivation"]["final_total"] for r in dsens_scored) if dsens_scored else None,\n                           "criterion_means": {k: statistics.mean(v) for k, v in crit.items()}, "manual_reviewed": sum(1 for r in scored if r["derivation"].get("manual")),\n                           "distribution": dict(Counter(int(r["derivation"]["final_total"]) for r in scored))},\n            "performance": {"cold_start_ready_ms_median": statistics.median(c["spawn_to_ready_ms"] for c in cs) if cs else None,\n                            "cold_first_solve_ms_median": statistics.median(c["first_solve_wall_ms"] for c in cs) if cs else None,\n                            "warm_median_common_ms": med, "warm_p95_common_ms": p95, "warm_median_own_ms": own_med, "warm_p95_own_ms": own_p95,\n                            "common_cases": len(common), "timeouts": meta[s]["timeouts"], "crashes": meta[s]["crashes"], "peak_rss_mb": meta[s]["maxrss_kb"] / 1024,\n                            "per_operation_median_ms": {k: statistics.median(v) for k, v in sorted(ops.items())}},\n            "integration": {"points": integ_pts, "max": 14, "criteria": {k: {"score": v[0], "evidence": v[1]} for k, v in integ.items()}},\n            "components": comp, "overall": overall, "sensitivity": variants, "by_category": cats,\n        }\n    # ---- hard gates\n    gates = {}\n    for s in SOLVERS:\n        rs = by[s]\n        critical = [r for r in rs if r["status"] == "invalid_answer" and r["status_detail"].startswith(("claims no solution", "claims every x", "fabricated_solution", "wrong_", "returned solutions", "returned "))]\n        majors = [r for r in rs if r["status"] == "invalid_answer" and r not in critical]\n        g2 = []\n        for r in rs:\n            dd = r["derivation"]\n            if not dd.get("scored"):\n                continue\n            nf = r.get("native_flags") or {}\n            if nf.get("steps_mode") == "direct_evaluation":\n                g2.append((r["case_id"], "steps_mode=direct_evaluation: worked steps replaced by direct solve"))\n            ev = dd["evidence"]\n            if ev["invalid_steps"] > 0:\n                g2.append((r["case_id"], "automated invalid-step detection: " + ev["correctness"]))\n            elif dd.get("manual") and dd["manual"]["adjusted"].get("correctness") is not None and "CONTRADICTION" in dd["manual"]["justification"]:\n                g2.append((r["case_id"], "manual: contradiction"))\n        g2_ids = sorted({c for c, _ in g2})\n        # K01 final step shows the unchanged input (manual finding) counts as inconsistency\n        for r in rs:\n            dd = r["derivation"]\n            if dd.get("manual") and "ORIGINAL expression" in dd["manual"]["justification"] and r["case_id"] not in g2_ids:\n                g2_ids.append(r["case_id"]); g2.append((r["case_id"], "manual: final step shows the unchanged input"))\n        c3 = eng[s]["untrusted_input_canary"]["score"]\n        gates[s] = {\n            "G1_critical_correctness": {"status": "FAIL" if critical else "PASS", "critical_cases": [(r["case_id"], r["status_detail"]) for r in critical],\n                                         "major_defects(not gating)": [(r["case_id"], r["status_detail"][:70]) for r in majors]},\n            "G2_fabricated_or_invalid_derivations": {"status": "FAIL" if len(g2_ids) >= 2 else "CONDITIONAL" if len(g2_ids) == 1 else "PASS", "cases": sorted(g2_ids), "evidence": g2},\n            "G3_unsafe_execution": {"status": "PASS" if c3 == 1 else ("CONDITIONAL" if s == "sympy-direct" else "FAIL"), "canary_score": c3,\n                                    "note": eng[s]["untrusted_input_canary"]["note"]},\n            "G4_licence": {"status": ("PASS" if (evidence.get(s) or {}).get("licence_class") == "permissive" else "CONDITIONAL"), "licence": (evidence.get(s) or {}).get("licence", "unknown")},\n        }\n        st = [g["status"] for g in gates[s].values()]\n        gates[s]["eligible_for_production_recommendation"] = "NO (as-is)" if "FAIL" in st else ("CONDITIONAL" if "CONDITIONAL" in st else "YES")\n    out["hard_gates"] = gates\n    out["common_latency_cases"] = common\n    (ROOT / "results/scores.json").write_text(json.dumps(out, indent=1, default=str))\n    for s in SOLVERS:\n        o = out["solvers"][s]\n        print("%-13s overall=%.1f  comps=%s  gates=%s" % (s, o["overall"], {k: round(v, 1) for k, v in o["components"].items()}, gates[s]["eligible_for_production_recommendation"]))\n        print("      variants:", {k[:28]: round(v, 1) for k, v in o["sensitivity"].items()})\n\n\nif __name__ == "__main__":\n    main()\n')
W('bench/compare_runs.py', '#!/usr/bin/env python3\n"""Reproducibility check: compare two full runs (raw replies and judged verdicts). Timings are expected to differ; everything else must not."""\nimport json, pathlib, statistics, subprocess, sys\nROOT = pathlib.Path(__file__).resolve().parent.parent\na, b = sys.argv[1], sys.argv[2]\nout = pathlib.Path(sys.argv[3]); out.mkdir(exist_ok=True, parents=True)\nsubprocess.run([sys.executable, str(ROOT / "bench/evaluate.py"), "--raw", b, "--out", str(out)], check=True, capture_output=True)\nA = json.load(open(ROOT / "results/raw_results.json"))["rows"]; B = json.load(open(out / "raw_results.json"))["rows"]\nBi = {(r["solver"], r["case_id"]): r for r in B}\ndiff_hash = diff_status = diff_score = n = 0; details = []\nra = {s: json.load(open(pathlib.Path(a) / f"{s}.json")) for s in ("sympy-direct", "smartsolver", "stepsolver") if (pathlib.Path(a) / f"{s}.json").exists()}\nrb = {s: json.load(open(pathlib.Path(b) / f"{s}.json")) for s in ra}\nfor r in A:\n    n += 1\n    o = Bi[(r["solver"], r["case_id"])]\n    h1, h2 = ra[r["solver"]]["rows"][r["case_id"]].get("hash"), rb[r["solver"]]["rows"][r["case_id"]].get("hash")\n    if h1 != h2: diff_hash += 1; details.append((r["solver"], r["case_id"], "raw canonical output differs"))\n    if r["status"] != o["status"] or r["strict_correct"] != o["strict_correct"]: diff_status += 1; details.append((r["solver"], r["case_id"], "verdict differs"))\n    if (r["derivation"].get("automated_total"), r["derivation"].get("final_total")) != (o["derivation"].get("automated_total"), o["derivation"].get("final_total")): diff_score += 1; details.append((r["solver"], r["case_id"], "derivation score differs"))\nres = {"rows_compared": n, "raw_output_differences": diff_hash, "verdict_differences": diff_status, "derivation_score_differences": diff_score, "details": details[:20], "timing_median_ms_run1_vs_run2": {}}\nfor s in ra:\n    m = lambda R: statistics.median([x["warm_median_ms"] for x in R if x["solver"] == s and x["warm_median_ms"] is not None])\n    res["timing_median_ms_run1_vs_run2"][s] = [round(m(A), 2), round(m(B), 2)]\n(ROOT / "results/reproducibility_check.json").write_text(json.dumps(res, indent=1))\nprint(json.dumps(res, indent=1))\n')
W('bench/report.py', '#!/usr/bin/env python3\n"""results/*.json -> results/summary.md, results/failure_analysis.md (and results/changes_vs_baseline.md if data/baseline.json exists).\n\nEVERYTHING in these reports is computed from the preserved results. There is no run-specific narrative: strongest/weakest areas and the\nrecommendation are produced by documented rules from the measured data, so the reports stay valid for any engine version."""\nimport json\nimport pathlib\nfrom collections import defaultdict\n\nROOT = pathlib.Path(__file__).resolve().parent.parent\nRES = ROOT / "results"\nNAMES = {"sympy-direct": "SymPy (direct)", "smartsolver": "SmartSolver", "stepsolver": "StepSolver"}\nCRIT = ["correctness", "completeness", "explanation", "consistency", "domain", "presentation"]\n\n\ndef f(x, n=1):\n    return "n/a" if x is None else ("%.*f" % (n, x))\n\n\ndef table(head, rows):\n    return "\\n".join(["| " + " | ".join(head) + " |", "|" + "|".join("---" for _ in head) + "|"] + ["| " + " | ".join(str(c) for c in r) + " |" for r in rows])\n\n\ndef cut(x, n):\n    return str(x if x is not None else "").replace("\\n", " ").replace("|", "/")[:n]\n\n\ndef main():\n    d = json.load(open(RES / "raw_results.json"))\n    sc = json.load(open(RES / "scores.json"))\n    eng = json.load(open(RES / "engineering_results.json"))\n    rep = json.load(open(RES / "reproducibility_check.json")) if (RES / "reproducibility_check.json").exists() else None\n    ref = json.load(open(RES / "reference_validation.json"))\n    evp = RES / "evidence/candidates.json"\n    evidence = json.load(open(evp)) if evp.exists() else {}\n    envtxt = (RES / "evidence/environment.txt").read_text().strip() if (RES / "evidence/environment.txt").exists() else ""\n    cases = {c["id"]: c for c in json.load(open(ROOT / "data/cases.json"))["cases"]}\n    cats = json.load(open(ROOT / "data/cases.json"))["categories"]\n    rows, meta = d["rows"], d["meta"]\n    S3 = sc["solver_order"]\n    by = {s: [r for r in rows if r["solver"] == s] for s in S3}\n    sv, g = sc["solvers"], sc["hard_gates"]\n    pf = {s: sv[s]["performance"] for s in S3}\n    n_app = sc["applicable_cases"]\n    scored_total = sum(sv[s]["derivation"]["scored"] for s in S3)\n    reviewed_total = sum(sv[s]["derivation"]["manual_reviewed"] for s in S3)\n    stale = len(d.get("stale_manual_reviews", []))\n    L = []\n    A = L.append\n\n    A("# Math-solver benchmark: " + " vs ".join(NAMES[s] for s in S3) + "\\n")\n    A("Generated from `results/raw_results.json`, `scores.json`, `engineering_results.json`. Every number and sentence is computed by code from preserved raw solver output; there is no hand-written, run-specific narrative.\\n")\n    A("> **Read this first.** (1) Mathematical correctness is decided by a symbolic/numeric judge (SymPy used only as a checker) and every reference answer is independently validated with mpmath/Fraction arithmetic. (2) **Derivation scores are proxies**: an automated rubric plus a manual overlay written by a language model for a specific earlier set of outputs. This run scored %d derivations; %d use that overlay; %d overlay entries were NOT applied because the engine\'s output changed since they were written (their automated score is used). The overlay needs human audit and never decides mathematical correctness. (3) One reference value in the task specification (K04) is slightly wrong; see section 6. (4) SymPy is evaluated as a **bare engine** (no step layer by design).\\n" % (scored_total, reviewed_total, stale))\n\n    # 1\n    A("## 1. Environment and versions\\n")\n    A("```\\n" + envtxt + "\\n```")\n    A(table(["Engine", "Version", "Commit / source", "Python", "SymPy in its env", "Isolated env"],\n            [[NAMES[s], meta[s]["version"], (meta[s]["commit"] or (evidence.get(s, {}).get("source") or "PyPI release"))[:44], meta[s]["python"], meta[s].get("sympy_version") or "n/a",\n              "`envs/%s`" % {"sympy-direct": "sympy", "smartsolver": "smartsolver", "stepsolver": "stepsolver"}[s]] for s in S3]))\n    A("\\nEach engine ran in its own virtualenv and worker process from an empty working directory. The judge runs in a separate pinned environment. Timings depend on the hardware above and are only comparable between runs on the same hardware.\\n")\n\n    # 2\n    A("## 2. Installation and execution status\\n")\n    tr = []\n    for s in S3:\n        e = evidence.get(s, {})\n        own = (e.get("tests") or {}).get("own_suite") or {}\n        tr.append([NAMES[s], "yes" if e.get("installed", True) else "NO: " + cut(e.get("install_error"), 80),\n                   "%d/%d attempted" % (sv[s]["executed"], sv[s]["defined"]),\n                   ("%s passed / %s failed" % (own.get("passed"), own.get("failed"))) if own.get("run") else "not run (static test count: %s)" % ((e.get("tests") or {}).get("static_test_functions") if (e.get("tests") or {}).get("static_test_functions") is not None else "n/a: bare engine"),\n                   e.get("runtime_deps", "n/a"), "%s (%s)" % (e.get("licence"), e.get("licence_class")), e.get("python_requires") or "n/a"])\n    A(table(["Engine", "Installed", "Executed", "Own test-suite", "Third-party deps", "Licence", "requires-python"], tr))\n    A("\\nSkipped cases: %s. A case is only \'unsupported\' when the engine has no method for it or declares it unsolved; nothing is substituted.\\n" % ", ".join("%s %d" % (NAMES[s], sv[s]["skipped"]) for s in S3))\n\n    # 3\n    A("## 3. Results by category (strictly correct / cases)\\n")\n    A(table(["Category"] + [NAMES[s] for s in S3],\n            [[c + " " + nm] + ["%d/%d" % (sv[s]["by_category"][c]["strict_correct"], sv[s]["by_category"][c]["n"]) for s in S3] for c, nm in cats.items()] +\n            [["**Total**"] + ["**%d/%d**" % (sv[s]["passed(strict_correct)"], sv[s]["defined"]) for s in S3]]))\n    A("\\n*Strictly correct* = correct value AND required form AND (where required) the domain restriction reported. Value-only counts: " + ", ".join("%s %d" % (NAMES[s], sv[s]["value_only_correct"]) for s in S3) + ".\\n")\n\n    # 4\n    A("## 4. Overall correctness and coverage\\n")\n    A(table(["Metric", *[NAMES[s] for s in S3]], [\n        ["Cases defined / executed / skipped", *["%d / %d / %d" % (sv[s]["defined"], sv[s]["executed"], sv[s]["skipped"]) for s in S3]],\n        ["Strictly correct (passed)", *[sv[s]["passed(strict_correct)"] for s in S3]],\n        ["Invalid answers (produced, wrong)", *[sv[s]["status_counts"].get("invalid_answer", 0) for s in S3]],\n        ["Unsupported: no API for the operation", *[sv[s]["unsupported_no_api"] for s in S3]],\n        ["Unsupported: engine declared it unsolved", *[sv[s]["unsolved_by_solver"] for s in S3]],\n        ["Runtime errors / timeouts / infra errors / input-incompatible", *["%d / %d / %d / %d" % (sv[s]["status_counts"].get("runtime_error", 0), sv[s]["status_counts"].get("timeout", 0), sv[s]["status_counts"].get("infra_error", 0), sv[s]["status_counts"].get("input_incompatible", 0)) for s in S3]],\n        ["Coverage (result in the expected form)", *["%d (%.1f%%)" % (sv[s]["covered"], 100 * sv[s]["covered"] / sv[s]["defined"]) for s in S3]],\n        ["Correctness (strict) %", *[f(sv[s]["components"]["correctness"]) for s in S3]]]))\n    A("\\nCoverage counts a result only if it is in the requested form; correctness is reported separately.\\n")\n\n    # 5\n    A("## 5. Derivation availability and quality\\n")\n    A(table(["Metric", *[NAMES[s] for s in S3]], [\n        ["Derivation available (cases / %d)" % n_app, *["%d (%.0f%%)" % (sv[s]["derivation"]["available"], sv[s]["derivation"]["available_pct_of_applicable"]) for s in S3]],\n        ["Mean automated score /10", *[f(sv[s]["derivation"]["mean_automated"], 2) for s in S3]],\n        ["Mean final score /10 (after any applied manual overlay)", *[f(sv[s]["derivation"]["mean_final"], 2) for s in S3]],\n        ["Mean final, domain-sensitive cases only", *[f(sv[s]["derivation"]["mean_final_domain_sensitive"], 2) for s in S3]],\n        ["Derivations with an applied manual review", *[sv[s]["derivation"]["manual_reviewed"] for s in S3]],\n        *[["Mean " + k, *[f(sv[s]["derivation"]["criterion_means"].get(k), 2) for s in S3]] for k in CRIT]]))\n    opaque = {s: [r["case_id"] for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["substantive_steps"] == 0] for s in S3}\n    A("\\nMax per criterion: correctness 3, completeness 2, explanation 2, consistency 1, domain 1, presentation 1. Opaque-only derivations (no transformation shown): " +\n      "; ".join("%s %d%s" % (NAMES[s], len(opaque[s]), (" (" + ", ".join(opaque[s]) + ")") if opaque[s] else "") for s in S3) + ".\\n")\n    A("Score distribution (final /10 -> number of derivations): " + "; ".join("%s %s" % (NAMES[s], dict(sorted(sv[s]["derivation"]["distribution"].items()))) for s in S3) + ".\\n")\n\n    # 6\n    A("## 6. Edge-case, domain and engineering behaviour\\n")\n    A("Engineering probes (1 = clean, 0.5 = acceptable but raw/implicit, 0 = failure, n/a = does not apply):\\n")\n    pids = list(eng[S3[0]].keys())\n    A(table(["Probe", *[NAMES[s] for s in S3]], [[p, *[("n/a" if eng[s][p]["score"] is None else eng[s][p]["score"]) for s in S3]] for p in pids]))\n    A("\\nProbe outcomes that were not clean (from the raw probe notes):\\n")\n    for s in S3:\n        for p in pids:\n            if eng[s][p]["score"] is not None and eng[s][p]["score"] < 1:\n                A("- **%s / %s** (%s): %s" % (NAMES[s], p, eng[s][p]["score"], cut(eng[s][p]["note"], 220)))\n    A("\\n`untrusted_input_canary` sends the benign payload `__import__(\'math\').factorial(4)`; a score of 0 means the engine\'s own string interface evaluated it as Python. `heavy_equation` is `x^25 - x - 1 = 0` with a 15 s budget; a score of 0 means the engine did not finish and needed an external kill.\\n")\n    dsn = sv[S3[0]]["domain_sensitive_cases"]\n    A("Domain-sensitive cases (%d): strictly correct %s.\\n" % (dsn, ", ".join("%s %d" % (NAMES[s], sv[s]["domain_sensitive_strict_correct"]) for s in S3)))\n    k4 = [(NAMES[r["solver"]], r["judge_detail"]) for s in S3 for r in by[s] if r["case_id"] == "K04" and r["judge_detail"]]\n    A("**Specification discrepancy (K04):** the task prints \\"approximately 1.167304015\\", but the real root of x^5 - x - 1 is 1.1673039782614187 (mpmath 40 digits, bisection, polyroots agree); the printed value is off by 3.7e-8. The verbatim spec text is kept in `spec_expected`; the verified root is the reference (tolerance 1e-8). Per-engine: " + "; ".join("%s: %s" % (n, cut(t, 90)) for n, t in k4) + ".\\n")\n    A("Reference validation: %d/%d references independently validated by a second method, all_ok=%s.\\n" % (ref["validated"] - len(ref["failed"]), ref["validated"], ref["all_ok"]))\n\n    # 7\n    A("## 7. Reliability and performance\\n")\n    A(table(["Reliability metric (0-1)", *[NAMES[s] for s in S3]], [[k, *[f(sv[s]["reliability_metrics"][k], 3) for s in S3]] for k in sv[S3[0]]["reliability_metrics"]] +\n            [["**Reliability score (mean x100)**", *["**%s**" % f(sv[s]["components"]["reliability"]) for s in S3]]]))\n    A("\\nReliability = 100 x mean(R1 repeatability, R2 engineering probes, R3 = 1 - crashes/attempted, R4 strict-correct rate on domain-sensitive cases, R5 untrusted-input safety as shipped); see `docs/SCORING_RULES.md`.\\n")\n    A(table(["Performance (ms)", *[NAMES[s] for s in S3]], [\n        ["Cold start: process spawn -> ready (median)", *[f(pf[s]["cold_start_ready_ms_median"], 0) for s in S3]],\n        ["Cold first solve of C01 (median)", *[f(pf[s]["cold_first_solve_ms_median"], 0) for s in S3]],\n        ["Warm median, %d common cases (cache-cleared repeats)" % pf[S3[0]]["common_cases"], *[f(pf[s]["warm_median_common_ms"], 2) for s in S3]],\n        ["Warm p95, common cases", *[f(pf[s]["warm_p95_common_ms"], 2) for s in S3]],\n        ["Warm median, own supported cases", *[f(pf[s]["warm_median_own_ms"], 2) for s in S3]],\n        ["Warm p95, own supported cases", *[f(pf[s]["warm_p95_own_ms"], 2) for s in S3]],\n        ["Timeouts / crashes (main pass)", *["%d / %d" % (pf[s]["timeouts"], pf[s]["crashes"]) for s in S3]],\n        ["Peak RSS of worker (MB)", *[f(pf[s]["peak_rss_mb"], 0) for s in S3]],\n        ["**Performance score**", *["**%s**" % f(sv[s]["components"]["performance"]) for s in S3]]]))\n    A("\\nLatency score: 100 at <=10 ms, 0 at >=2000 ms, log-linear between; Performance = 0.4 x score(median) + 0.4 x score(p95) + 0.2 x 100 x (1 - failure rate), on the cases ALL engines answered. Cold start is reported, not scored.\\n")\n    if rep:\n        A("Reproducibility (run 2 vs run 1): **%d/%d** rows identical in raw output, **%d** verdict differences, **%d** derivation-score differences. Median warm latency run1 -> run2: %s.\\n" %\n          (rep["rows_compared"] - rep["raw_output_differences"], rep["rows_compared"], rep["verdict_differences"], rep["derivation_score_differences"],\n           "; ".join("%s %s -> %s ms" % (NAMES[k], v[0], v[1]) for k, v in rep["timing_median_ms_run1_vs_run2"].items())))\n\n    # 8\n    A("## 8. Weighted scores and hard gates\\n")\n    A(table(["Component (weight)", *[NAMES[s] for s in S3]], [\n        ["Correctness (30%)", *[f(sv[s]["components"]["correctness"]) for s in S3]],\n        ["Derivation quality (30%)", *[f(sv[s]["components"]["derivations"]) for s in S3]],\n        ["Coverage (20%)", *[f(sv[s]["components"]["coverage"]) for s in S3]],\n        ["Reliability (10%)", *[f(sv[s]["components"]["reliability"]) for s in S3]],\n        ["Integration/maintainability (5%)", *[f(sv[s]["components"]["integration"]) for s in S3]],\n        ["Performance (5%)", *[f(sv[s]["components"]["performance"]) for s in S3]],\n        ["**Overall /100**", *["**%s**" % f(sv[s]["overall"]) for s in S3]]]))\n    A("\\n`Overall = 0.30*Correctness + 0.30*Derivations + 0.20*Coverage + 0.10*Reliability + 0.05*Integration + 0.05*Performance`. The derivation component is the mean score over *available* derivations x 10, as specified; the availability-penalised variant is in the sensitivity table.\\n")\n    A("**Integration evidence (computed):**\\n")\n    for s in S3:\n        A("- **%s** (%d/14): " % (NAMES[s], sv[s]["integration"]["points"]) + "; ".join("%s=%d (%s)" % (k, v["score"], cut(v["evidence"], 110)) for k, v in sv[s]["integration"]["criteria"].items()))\n    A("\\n**Sensitivity (ranking stability):**\\n")\n    vs = list(sv[S3[0]]["sensitivity"].keys())\n    A(table(["Variant", *[NAMES[s] for s in S3], "Order"], [[v, *[f(sv[s]["sensitivity"][v]) for s in S3], " > ".join(NAMES[s] for s in sorted(S3, key=lambda s: -sv[s]["sensitivity"][v]))] for v in vs]))\n    A("\\n**Hard gates** (a weighted score never overrides these):\\n")\n    A(table(["Gate", *[NAMES[s] for s in S3]], [\n        ["G1 critical correctness defect (confidently wrong answer)", *[g[s]["G1_critical_correctness"]["status"] + (" " + ", ".join(c for c, _ in g[s]["G1_critical_correctness"]["critical_cases"]) if g[s]["G1_critical_correctness"]["critical_cases"] else "") for s in S3]],\n        ["G2 invalid / self-contradicting derivation (FAIL >=2 cases, CONDITIONAL 1)", *[g[s]["G2_fabricated_or_invalid_derivations"]["status"] + (" " + ", ".join(g[s]["G2_fabricated_or_invalid_derivations"]["cases"]) if g[s]["G2_fabricated_or_invalid_derivations"]["cases"] else "") for s in S3]],\n        ["G3 untrusted input executed as code (as shipped)", *[g[s]["G3_unsafe_execution"]["status"] for s in S3]],\n        ["G4 licence", *[g[s]["G4_licence"]["status"] + " (" + str(g[s]["G4_licence"]["licence"]) + ")" for s in S3]],\n        ["**Eligible for production recommendation**", *["**" + g[s]["eligible_for_production_recommendation"] + "**" for s in S3]]]))\n    A("\\nGate evidence:\\n")\n    for s in S3:\n        for k in ("G1_critical_correctness", "G2_fabricated_or_invalid_derivations", "G3_unsafe_execution"):\n            gg = g[s][k]\n            if gg["status"] == "PASS":\n                continue\n            if k.startswith("G1"):\n                ev = "; ".join("%s: %s" % (c, cut(t, 80)) for c, t in gg["critical_cases"])\n            elif k.startswith("G2"):\n                ev = "; ".join("%s: %s" % (c, cut(t, 120)) for c, t in gg["evidence"][:6])\n            else:\n                ev = cut(gg["note"], 200)\n            A("- **%s %s = %s**: %s" % (NAMES[s], k[:2], gg["status"], ev))\n    A("\\nG3 is CONDITIONAL for SymPy-direct by definition: only its *string* parser evaluates code, it offers programmatic construction, and this benchmark\'s allow-list adapter blocks the payload.\\n")\n\n    # 9\n    A("## 9. Strongest and weakest areas (computed)\\n")\n    for s in S3:\n        o, cs = sv[s], sv[s]["by_category"]\n        full = [c for c, v in cs.items() if v["strict_correct"] == v["n"]]\n        zero = [c for c, v in cs.items() if v["strict_correct"] == 0]\n        partial = sorted([(v["strict_correct"] / v["n"], c) for c, v in cs.items() if 0 < v["strict_correct"] < v["n"]])\n        fails = defaultdict(list)\n        for r in by[s]:\n            if r["status"] == "invalid_answer":\n                fails["invalid answer"].append(r["case_id"])\n            elif r["status"] == "runtime_error":\n                fails["runtime error"].append(r["case_id"])\n            elif r["status"] == "unsupported":\n                fails["unsupported (%s)" % ("no API" if r["status_detail"] == "no_api" else "declared unsolved")].append(r["case_id"])\n            elif r["status"] == "valid" and not r["strict_correct"]:\n                fails["right value, domain/form requirement missed"].append(r["case_id"])\n        dv = o["derivation"]\n        low = [p for p in pids if eng[s][p]["score"] is not None and eng[s][p]["score"] < 1]\n        A("**%s** (overall %.1f)\\n" % (NAMES[s], o["overall"]))\n        A("- Strongest: %s; strict %d/%d, coverage %d/%d, derivations available %d/%d (mean %.2f/10), warm median %.1f ms." % (\n            ("all cases correct in categories " + ", ".join(full)) if full else "no category fully correct", o["passed(strict_correct)"], o["defined"], o["covered"], o["defined"],\n            dv["available"], o["defined"], dv["mean_final"], pf[s]["warm_median_common_ms"] or 0))\n        A("- Weakest: %s%s." % (("no correct answers in categories " + ", ".join(zero)) if zero else "no category entirely wrong",\n                                ("; lowest partial categories: " + ", ".join("%s (%.0f%%)" % (c, 100 * p) for p, c in partial[:3])) if partial else ""))\n        A("- Not-passed cases by type: " + ("; ".join("%s: %s" % (k, ", ".join(v)) for k, v in fails.items()) if fails else "none") + ".")\n        A("- Derivation concerns: %d opaque-only, %d with auto-detected invalid steps, %d with override/untied workings." % (\n            len(opaque[s]), sum(1 for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["invalid_steps"] > 0),\n            sum(1 for r in by[s] if (r.get("native_flags") or {}).get("steps_mode") == "direct_evaluation" or (r.get("native_flags") or {}).get("workings_tied_to_answer") is False)))\n        A("- Engineering probes below 1.0: %s; gates not PASS: %s.\\n" % (", ".join("%s (%s)" % (p, eng[s][p]["score"]) for p in low) or "none",\n                                                                        ", ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[s].items() if k.startswith("G") and v["status"] != "PASS") or "none"))\n\n    # 10\n    A("## 10. Recommendation (rule-based, with limits)\\n")\n    elig = {s: g[s]["eligible_for_production_recommendation"] for s in S3}\n    order = sorted(S3, key=lambda s: -sv[s]["overall"])\n    A("Decision rules (fixed before any run): (1) an engine with any hard-gate FAIL is not recommended as-is; (2) among the rest, rank by weighted score; (3) scores within 5 points are treated as a tie to be broken by gates, derivation availability and the failures listed above; (4) a gate-failing engine can still be piloted behind a validation wrapper if the blocking defects are listed and fixed first.\\n")\n    A(table(["Rank", "Engine", "Overall", "Eligible as-is", "Blocking gates"], [[i + 1, NAMES[s], f(sv[s]["overall"]), elig[s], ", ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[s].items() if k.startswith("G") and v["status"] in ("FAIL", "CONDITIONAL")) or "none"] for i, s in enumerate(order)]))\n    ok = [s for s in order if elig[s] == "YES"]\n    cond = [s for s in order if elig[s] == "CONDITIONAL"]\n    top = order[0]\n    A("")\n    if ok:\n        pick = ok[0]\n        A("**Recommended as-is:** %s (highest-scoring engine with no failed or conditional gate; overall %.1f)." % (NAMES[pick], sv[pick]["overall"]))\n    elif cond:\n        pick = cond[0]\n        A("**Recommended with conditions:** %s (highest-scoring engine with no FAIL gate; overall %.1f). Conditions: %s." % (NAMES[pick], sv[pick]["overall"],\n          "; ".join("%s=%s" % (k[:2], v["status"]) for k, v in g[pick].items() if k.startswith("G") and v["status"] == "CONDITIONAL")))\n    else:\n        pick = None\n        A("**No engine is eligible as-is.**")\n    if top != pick:\n        gap = (sv[top]["overall"] - sv[pick]["overall"]) if pick else None\n        reasons = []\n        for k, v in g[top].items():\n            if k.startswith("G") and v["status"] in ("FAIL", "CONDITIONAL"):\n                ids = v.get("cases") or [c for c, _ in v.get("critical_cases", [])]\n                reasons.append("%s = %s%s" % (k[:2], v["status"], (" (" + ", ".join(ids) + ")") if ids else ""))\n        A("")\n        A("**Highest weighted score:** %s (%.1f)%s. It is not eligible as-is because: %s." % (NAMES[top], sv[top]["overall"], (", %.1f points above the recommended engine" % gap) if pick else "", "; ".join(reasons)))\n        A("Piloting it behind a validation wrapper is reasonable only after the listed failures are fixed and this notebook is re-run to confirm the gates pass.")\n    A("\\nFrom the measured data, the practical split of responsibilities is: use the engine with the best strict correctness/coverage as the *answer* source, and treat an engine\'s derivation as shippable only where it is available, scored, and free of invalid/contradictory steps (see failure_analysis.md sections 2 and 3). Where derivations are opaque or unavailable, show the verified answer and label the derivation unavailable instead of showing a placeholder.\\n")\n    A("### Limitations of this benchmark\\n")\n    A("- The 141-case suite (72 core cases plus 69 targeted stress cases) is still only a sample of school/early-university mathematics; it does **not** show that any engine supports all of mathematics or handles word problems, images, or free-form phrasing.\\n"\n      "- Derivation scores are proxy rubrics plus a model-assigned overlay (for earlier outputs only); human audit is required. Completeness uses regex milestones.\\n"\n      "- Latency is hardware-dependent and a small part of the score (5%); warm latency is measured cache-cleared in a warm process.\\n"\n      "- K03/K04 (numeric solving) use each engine\'s nearest documented API (SymPy `nsolve`/`nroots`; engines without a numeric root-finder use their generic equation solver). SmartSolver `limit` uses its documented one-sided default.\\n"\n      "- StepSolver `ln` is adapted to `log`; the as-specified result is preserved in `native_flags.as_specified_variant`.\\n"\n      "- Adapters map to the public API names that existed when the benchmark was written; a renamed/removed method is reported as unsupported (SmartSolver) or an infrastructure error (import failure), never silently substituted.\\n"\n      "- Offline operation was assessed only by static search for network imports in each library core (`engineering_results.json`).\\n")\n    A("### Artefacts\\n")\n    A("`results/raw_results.json|csv`, `results/raw/*.json` (verbatim worker replies), `results/scores.json`, `results/engineering_results.json`, `results/reference_validation.json`, `results/reproducibility_check.json`, `results/failure_analysis.md`, `results/changes_vs_baseline.md` (if a baseline is present), `data/cases.json`, `docs/`.\\n")\n\n    # 11 baseline\n    base_path = ROOT / "data/baseline.json"\n    if base_path.exists():\n        base = json.load(open(base_path))\n        B = []\n        B.append("# Changes vs the baseline run\\n")\n        B.append("Baseline: **%s**.\\n" % base["label"])\n        B.append(table(["Engine", "Baseline version", "This run", "Baseline overall", "This overall", "Delta"], [\n            [NAMES[s], "%s %s" % (base["engines"].get(s, {}).get("version"), (base["engines"].get(s, {}).get("commit") or "")[:8]), "%s %s" % (meta[s]["version"], (meta[s]["commit"] or "")[:8]),\n             f(base["overall"].get(s)), f(sv[s]["overall"]), f(sv[s]["overall"] - base["overall"][s]) if s in base["overall"] else "n/a"] for s in S3]))\n        B.append("\\n" + table(["Component"] + ["%s (base -> now)" % NAMES[s] for s in S3], [[k] + ["%s -> %s" % (f(base["components"].get(s, {}).get(k)), f(sv[s]["components"][k])) for s in S3] for k in sv[S3[0]]["components"]]))\n        reg, imp, other = [], [], []\n        for r in rows:\n            b = base["cases"].get("%s|%s" % (r["solver"], r["case_id"]))\n            if not b:\n                continue\n            if b["strict_correct"] and not r["strict_correct"]:\n                reg.append((r["solver"], r["case_id"], b["status"], r["status"], r["status_detail"]))\n            elif (not b["strict_correct"]) and r["strict_correct"]:\n                imp.append((r["solver"], r["case_id"], b["status"], r["status"]))\n            elif b["status"] != r["status"]:\n                other.append((r["solver"], r["case_id"], b["status"], r["status"]))\n        B.append("\\n## Regressions (strictly correct before, not now): %d\\n" % len(reg))\n        B.append(table(["Engine", "Case", "Before", "Now", "Detail"], [[NAMES[a], b, c, dd, cut(e, 90)] for a, b, c, dd, e in reg]) if reg else "None.")\n        B.append("\\n## Improvements (not strictly correct before, correct now): %d\\n" % len(imp))\n        B.append(table(["Engine", "Case", "Before", "Now"], [[NAMES[a], b, c, dd] for a, b, c, dd in imp]) if imp else "None.")\n        B.append("\\n## Other status changes: %d\\n" % len(other))\n        B.append(table(["Engine", "Case", "Before", "Now"], [[NAMES[a], b, c, dd] for a, b, c, dd in other]) if other else "None.")\n        B.append("\\n*Derivation scores that relied on the earlier manual overlay fall back to automated scores when output changed; compare derivation means with that in mind.*\\n")\n        (RES / "changes_vs_baseline.md").write_text("\\n".join(B) + "\\n")\n        A("## 11. Changes vs the baseline\\n")\n        A("See `results/changes_vs_baseline.md`: %d regressions, %d improvements, %d other status changes versus *%s*.\\n" % (len(reg), len(imp), len(other), base["label"]))\n    (RES / "summary.md").write_text("\\n".join(L) + "\\n")\n\n    # ------------------------------------------------------------------ failure analysis\n    F = []\n    Z = F.append\n    Z("# Failure analysis (generated from this run\'s data)\\n")\n    Z("Reproduce any row after setup: `envs/judge/bin/python bench/runner.py --solver <name> --case <ID> --no-timing --out /tmp/x`, then inspect `/tmp/x/<name>.json` (raw reply: `api_call`, `input_sent`, `raw_output`, `steps`, traceback tail). Verdicts: `results/raw_results.json` (`status`, `status_detail`, `judge_detail`).\\n")\n\n    def rowline(r):\n        c = cases[r["case_id"]]\n        return "| %s | %s | `%s` | %s | `%s` | `%s` | %s |" % (r["case_id"], NAMES[r["solver"]], cut(r["display_input"], 44), cut(c["spec_expected"], 40), cut(r["actual"], 60), cut(r.get("adapter_api_call"), 70), cut(r["status_detail"] or r["status"], 80))\n\n    hdr = "| Case | Engine | Input | Expected (spec) | Actual | Call | Verdict |\\n|---|---|---|---|---|---|---|"\n    Z("## 1. Incorrect answers (and values correct but not strictly correct)\\n")\n    inc = [r for s in S3 for r in by[s] if r["status"] == "invalid_answer" or (r["status"] == "valid" and not r["strict_correct"])]\n    Z(hdr + "\\n" + "\\n".join(rowline(r) for r in inc) if inc else "None.")\n    Z("\\nRows with status `valid` but not strictly correct have the right value and miss a declared domain-reporting or form requirement.\\n")\n    Z("## 2. Missing or invalid intermediate steps\\n")\n    for s in S3:\n        Z("- **%s** opaque/structural-only derivations (%d): %s" % (NAMES[s], len(opaque[s]), ", ".join(opaque[s]) or "none"))\n        unavailable = [r["case_id"] for r in by[s] if r["answer_produced"] and not r["derivation"].get("available")]\n        Z("  - answered but no derivation at all (%d): %s" % (len(unavailable), (", ".join(unavailable[:40]) + (" ..." if len(unavailable) > 40 else "")) or "none"))\n    Z("")\n    inv = [(r, r["derivation"]["evidence"]) for s in S3 for r in by[s] if r["derivation"].get("scored") and r["derivation"]["evidence"]["invalid_steps"] > 0]\n    Z("**Automatically detected invalid/misrendered steps** (consecutive-step equivalence check; a displayed step that is not equivalent to the one before it):\\n")\n    Z(table(["Engine", "Case", "Evidence"], [[NAMES[r["solver"]], r["case_id"], cut(e["correctness"], 180)] for r, e in inv]) if inv else "None detected.")\n    ov = [r for s in S3 for r in by[s] if (r.get("native_flags") or {}).get("steps_mode") == "direct_evaluation"]\n    Z("\\n**Derivations replaced by a direct-evaluation override (steps do not produce the answer):** " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in ov) or "none") + ".")\n    untied = [r for s in S3 for r in by[s] if (r.get("native_flags") or {}).get("workings_tied_to_answer") is False]\n    Z("**Workings from a code path separate from the answer:** " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in untied) or "none") + ".")\n    for s in S3:\n        flags = [r for r in by[s] if r["answer_produced"] and (r.get("native_flags") or {}).get("steps_verified") is not None]\n        if flags and all(r["native_flags"]["steps_verified"] for r in flags) and any(r["status"] == "invalid_answer" for r in by[s]):\n            Z("- **%s** reports `steps_verified=True` on all %d answered cases, including %d that are invalid: the flag carries no information." % (NAMES[s], len(flags), sum(1 for r in by[s] if r["status"] == "invalid_answer")))\n    contra = [(r["solver"], r["case_id"], r["derivation"]["manual"]["justification"]) for s in S3 for r in by[s] if r["derivation"].get("manual") and any(w in r["derivation"]["manual"]["justification"] for w in ("CONTRADICTION", "ORIGINAL expression", "Displayed first step"))]\n    if contra:\n        Z("\\n**Manual-review findings (overlay applied; model-assigned, needs audit):**\\n")\n        for a, b, c in contra:\n            Z("- %s %s: %s" % (NAMES[a], b, cut(c, 400)))\n    Z("")\n    Z("## 3. Domain and extraneous-solution errors\\n")\n    dom = [r for s in S3 for r in by[s] if r["status"] in ("invalid_answer", "valid") and ((r["domain_ok"] is False) or any(w in (r["status_detail"] or "") for w in ("domain", "real_domain_only", "partial", "claims no solution", "claims every x")))]\n    Z(hdr + "\\n" + "\\n".join(rowline(r) for r in dom) if dom else "None detected.")\n    Z("")\n    Z("## 4. Unsupported operations\\n")\n    for s in S3:\n        na = defaultdict(list)\n        un = []\n        for r in by[s]:\n            if r["status"] == "unsupported":\n                if r["status_detail"] == "no_api":\n                    na[r["category"]].append(r["case_id"])\n                else:\n                    un.append("%s (%s)" % (r["case_id"], cut(r["unsupported_reason"], 70)))\n        Z("- **%s** no API: %s" % (NAMES[s], "; ".join("%s: %s" % (k, ", ".join(v)) for k, v in sorted(na.items())) or "none"))\n        Z("  - declared unsolved by the engine: %s" % (", ".join(un) or "none"))\n    Z("")\n    Z("## 5. Parsing / input-format incompatibilities\\n")\n    pe = [r for s in S3 for r in by[s] if r["status"] == "input_incompatible"]\n    Z("Input-incompatible results (engine rejected the adapted syntax): " + (", ".join("%s %s (%s)" % (NAMES[r["solver"]], r["case_id"], cut((r.get("error") or {}).get("message"), 60)) for r in pe) or "none") + ".\\n")\n    adp = [r for s in S3 for r in by[s] if r["adapter_overrides"]]\n    Z("Cases that needed a documented adapter override/normalisation/composition: " + (", ".join("%s %s" % (NAMES[r["solver"]], r["case_id"]) for r in adp) or "none") + ".")\n    for r in adp:\n        v = (r.get("native_flags") or {}).get("as_specified_variant")\n        if v:\n            Z("- %s %s as specified (`%s`) -> %s: %s" % (NAMES[r["solver"]], r["case_id"], v["query"], v["result_type"], cut(v["raw_output"], 160)))\n    Z("")\n    for s in S3:\n        for p in ("malformed_syntax", "unbalanced_parens", "empty_input", "unsupported_operation"):\n            if eng[s][p]["score"] == 0:\n                Z("- **%s / %s**: %s" % (NAMES[s], p, cut(eng[s][p]["note"], 200)))\n    Z("")\n    Z("## 6. Runtime and dependency failures\\n")\n    for s in S3:\n        e = evidence.get(s, {})\n        if not e.get("installed", True):\n            Z("- **%s INSTALL FAILED**: %s" % (NAMES[s], cut(e.get("install_error"), 300)))\n    rt = [r for s in S3 for r in by[s] if r["status"] in ("runtime_error", "timeout", "infra_error")]\n    for r in rt:\n        er = r.get("error") or {}\n        Z("- **%s %s** %s: `%s: %s`  (call: `%s`)" % (NAMES[r["solver"]], r["case_id"], r["status"], er.get("type"), cut(er.get("message"), 140), cut(r.get("adapter_api_call"), 80)))\n    if not rt:\n        Z("- No runtime errors, timeouts or infrastructure failures in the main pass.")\n    for s in S3:\n        for p in ("heavy_equation", "large_polynomial_equation", "untrusted_input_canary", "zero_denominator"):\n            if eng[s][p]["score"] is not None and eng[s][p]["score"] == 0:\n                Z("- **%s / %s**: %s" % (NAMES[s], p, cut(eng[s][p]["note"], 220)))\n    Z("")\n    Z("## 7. Cases the evaluator cannot judge reliably (manual-review flags)\\n")\n    fl = [r for s in S3 for r in by[s] if r["manual_review_flag"]]\n    Z(table(["Engine", "Case", "Reason"], [[NAMES[r["solver"]], r["case_id"], cut("; ".join(r["manual_review_reasons"]), 150)] for r in fl]) if fl else "None.")\n    Z("\\n- Restriction reporting is decided by text matching of reported constraints.\\n- K03/K04 use the closest documented API; K04\'s reference differs from the specification\'s printed digits (specification typo).\\n- Derivation completeness/explanation are regex/word-count proxies; a manual overlay applies only to the exact output it reviewed (%d applied, %d stale and ignored).\\n- Expression equivalence is symbolic where possible, otherwise numeric at exact rational sample points (flagged when numeric-only).\\n" % (reviewed_total, stale))\n    (RES / "failure_analysis.md").write_text("\\n".join(F) + "\\n")\n    print("wrote summary.md, failure_analysis.md" + (", changes_vs_baseline.md" if base_path.exists() else ""))\n\n\nif __name__ == "__main__":\n    main()\n')
W('bench/v2_stress_analysis.py', '#!/usr/bin/env python3\n"""Summarise generalisation stress results and direct StepSolver diagnostics."""\nimport json\nimport pathlib\nimport statistics\nfrom collections import defaultdict\n\nROOT = pathlib.Path(__file__).resolve().parent.parent\n\ndef read(p, default=None):\n    try:\n        return json.loads((ROOT / p).read_text())\n    except Exception:\n        return {} if default is None else default\n\ndef pct(a):\n    return 100.0 * a if a is not None else 0.0\n\ndef main():\n    cases_data = read("data/cases.json")\n    cases = cases_data.get("cases", [])\n    v2_ids = {c["id"] for c in cases if c.get("suite") == "v2_stress"}\n    by_id = {c["id"]: c for c in cases}\n    raw = read("results/raw_results.json")\n    rows = [r for r in raw.get("rows", []) if r.get("case_id") in v2_ids]\n    solvers = ["sympy-direct", "smartsolver", "stepsolver"]\n    score = read("results/scores.json")\n    diagnostics = read("results/stepsolver_v2_diagnostics.json")\n    out = []\n    out.append("# V2 generalisation stress-test analysis")\n    out.append("")\n    out.append("The benchmark contains **%d core cases + %d V2 stress cases = %d total cases**. The core 72 retain the original IDs and expected answers; V2 cases are new adversarial variants designed to test whether the earlier fixes generalise. Unsupported cases count as coverage/correctness misses, not as evidence that a reference is wrong." % (sum(c.get("suite")=="core_72" for c in cases), len(v2_ids), len(cases)))\n    out.append("")\n    vref = read("results/v2_reference_validation.json")\n    out.append("Reference validation: **%s (%s/%s V2 references)**." % ("PASS" if vref.get("all_ok") else "FAIL", vref.get("validated", "?"), vref.get("total", "?")))\n    out.append("")\n    out.append("## Stress-suite headline by solver")\n    out.append("")\n    out.append("| Solver | Strictly correct | Covered | Invalid answers | Unsupported / declined | Runtime errors / timeouts | Mean derivation / 10 |")\n    out.append("|---|---:|---:|---:|---:|---:|---:|")\n    for s in solvers:\n        rr = [r for r in rows if r.get("solver") == s]\n        if not rr:\n            out.append("| %s | n/a | n/a | n/a | n/a | n/a | n/a |" % s)\n            continue\n        strict = sum(r.get("strict_correct") is True for r in rr)\n        covered = sum(r.get("covered") is True for r in rr)\n        invalid = sum(r.get("status") == "invalid_answer" for r in rr)\n        unsupported = sum(r.get("status") in ("unsupported", "input_incompatible", "skipped") for r in rr)\n        errors = sum(r.get("status") in ("runtime_error", "timeout", "infra_error") for r in rr)\n        ds = [r.get("derivation", {}).get("final_total") for r in rr if isinstance(r.get("derivation"), dict) and isinstance(r.get("derivation", {}).get("final_total"), (int,float))]\n        dmean = (sum(ds)/len(ds)) if ds else None\n        out.append("| %s | %d/%d (%.1f%%) | %d/%d (%.1f%%) | %d | %d | %d | %s |" % (s, strict, len(rr), 100*strict/len(rr), covered, len(rr), 100*covered/len(rr), invalid, unsupported, errors, ("%.2f" % dmean) if dmean is not None else "n/a"))\n    out.append("")\n    out.append("## Strict-correct results by category")\n    out.append("")\n    out.append("| Category | Cases | " + " | ".join(solvers) + " |")\n    out.append("|---|---:|" + "---:|"*len(solvers))\n    for cat in sorted({by_id[i]["category"] for i in v2_ids}):\n        ids = {i for i in v2_ids if by_id[i]["category"] == cat}\n        cells = []\n        for s in solvers:\n            rr = [r for r in rows if r.get("solver") == s and r.get("case_id") in ids]\n            n = sum(r.get("strict_correct") is True for r in rr)\n            cells.append("%d/%d" % (n, len(ids)))\n        out.append("| %s | %d | %s |" % (cat, len(ids), " | ".join(cells)))\n    out.append("")\n    out.append("## Numerical API dispatch audit")\n    out.append("")\n    out.append("Cases with an explicit initial guess must be sent through StepSolver\'s native `nsolve(...)` API; K04 intentionally has no guess and remains the generic high-degree polynomial fallback probe. This prevents the adapter from silently scoring `solve(...)` as if the new numeric API had been tested.")\n    out.append("")\n    out.append("| Case | Initial guess | Expected dispatch | Observed input / call | Dispatch check | Outcome |")\n    out.append("|---|---|---|---|---|---|")\n    numeric_cases = [c for c in cases if c.get("operation") == "nsolve"]\n    step_rows = {r.get("case_id"): r for r in raw.get("rows", []) if r.get("solver") == "stepsolver"}\n    for c in numeric_cases:\n        rr = step_rows.get(c["id"])\n        guess = c.get("problem", {}).get("guess")\n        want_native = guess is not None\n        expected_dispatch = "native nsolve API" if want_native else "generic polynomial fallback"\n        if not rr:\n            observed, check, outcome = "no StepSolver result row", "NOT RUN", "n/a"\n        else:\n            sent = rr.get("adapter_input")\n            sent_text = " ".join(str(v) for v in sent) if isinstance(sent, list) else str(sent or "")\n            api = str(rr.get("adapter_api_call") or "")\n            observed = (sent_text or api).replace("|", "\\\\|").replace("\\n", " ")[:130]\n            did_native = "nsolve(" in sent_text or "nsolve(" in api\n            did_fallback = "solve(" in sent_text or "solve(" in api\n            passed = did_native if want_native else (did_fallback and not did_native)\n            check = "PASS" if passed else "FAIL"\n            outcome = rr.get("status", "unknown")\n        out.append("| %s | %s | %s | %s | %s | %s |" % (c["id"], guess if guess is not None else "none", expected_dispatch, observed or "(empty)", check, outcome))\n    out.append("")\n    out.append("A `PASS` in this table confirms only correct adapter dispatch, not that the numerical result is correct; correctness and derivation scores are reported separately above.")\n    out.append("")\n    out.append("## StepSolver misses in the V2 stress set")\n    out.append("")\n    misses = [r for r in rows if r.get("solver") == "stepsolver" and r.get("strict_correct") is not True]\n    if not misses:\n        out.append("No strict-correctness misses in the V2 stress set.")\n    else:\n        out.append("| Case | Operation | Status | Covered | Expected | Actual / detail |")\n        out.append("|---|---|---|---|---|---|")\n        for r in misses:\n            expected = str(r.get("expected_spec", "")).replace("|", "\\\\|")[:100]\n            actual = (str(r.get("actual") or r.get("status_detail") or r.get("unsupported_reason") or "")\n                      .replace("|", "\\\\|").replace("\\n", " ")[:180])\n            out.append("| %s | %s | %s | %s | %s | %s |" % (r.get("case_id"), r.get("operation"), r.get("status"), r.get("covered"), expected, actual))\n    out.append("")\n    out.append("## StepSolver opacity and latency by operation")\n    out.append("")\n    raw_solver_path = ROOT / "results/raw/stepsolver.json"\n    raw_solver = read("results/raw/stepsolver.json")\n    per_op = defaultdict(list)\n    opaque_cases = []\n    for cid in sorted(v2_ids):\n        rr = raw_solver.get("rows", {}).get(cid, {}).get("reply", {})\n        c = by_id[cid]\n        per_op[c["operation"]].append(rr.get("time_ms"))\n        steps = rr.get("steps") or []\n        n_opaque = (rr.get("native_flags") or {}).get("opaque_steps")\n        if n_opaque is None:\n            n_opaque = sum(1 for st in steps if str(st.get("rule", "")).strip().lower() == "compute exact result")\n        n_opaque = int(n_opaque)\n        if n_opaque:\n            opaque_cases.append((cid, c["operation"], n_opaque))\n    out.append("| Operation | V2 cases | First-call median (ms) | Max (ms) |")\n    out.append("|---|---:|---:|---:|")\n    for op, vals in sorted(per_op.items()):\n        valid = [float(v) for v in vals if isinstance(v, (int,float))]\n        out.append("| %s | %d | %s | %s |" % (op, len(vals), ("%.2f" % statistics.median(valid)) if valid else "n/a", ("%.2f" % max(valid)) if valid else "n/a"))\n    out.append("")\n    out.append("Opaque placeholder-step cases: **%d / %d**." % (len(opaque_cases), len(v2_ids)))\n    if opaque_cases:\n        out.append("")\n        out.append("| Case | Operation | Opaque step count |")\n        out.append("|---|---|---:|")\n        for cid, op, n in opaque_cases:\n            out.append("| %s | %s | %d |" % (cid, op, n))\n    out.append("")\n    out.append("## Direct StepSolver parser/printer probes")\n    out.append("")\n    if not diagnostics:\n        out.append("No direct diagnostics JSON was available. Run the notebook\'s StepSolver diagnostics cell to create it.")\n    elif diagnostics.get("skipped"):\n        out.append("Direct StepSolver diagnostics were disabled in notebook configuration: %s" % diagnostics.get("message", "no details"))\n    elif diagnostics.get("import_error"):\n        out.append("Direct StepSolver diagnostics did not run: %s" % diagnostics.get("import_error", {}).get("message", "import failed"))\n    else:\n        summ = diagnostics.get("summary", {})\n        out.append("- Printer probes: %s pass, %s fail, %s inconclusive." % (summ.get("printer_pass", "?"), summ.get("printer_fail", "?"), summ.get("printer_inconclusive", "?")))\n        out.append("- Native `ln` alias: %s." % next((p.get("status") for p in diagnostics.get("parser_probes", []) if p.get("id") == "native_ln_alias"), "not run"))\n        out.append("- Unknown function probes failing safely: %d / %d." % (sum(p.get("status") == "pass" for p in diagnostics.get("parser_probes", []) if str(p.get("id", "")).startswith("unknown_function_")), sum(str(p.get("id", "")).startswith("unknown_function_") for p in diagnostics.get("parser_probes", []))))\n        out.append("- Zero-denominator probe: %s." % diagnostics.get("undefined_probe", {}).get("status", "not run"))\n        out.append("")\n        out.append("Detailed direct probe evidence is in `stepsolver_v2_diagnostics.json`. Inconclusive printer probes are not passes and should be manually inspected against the step strings.")\n    out.append("")\n    out.append("## How to interpret")\n    out.append("")\n    out.append("The stress set is intended to expose overfitting, unsupported mathematical families, invalid domain transformations, and regressions. A decline is a coverage miss; an answer can be covered but wrong. The fixed answer judge checks the declared mathematical result, while direct printer probes inspect rendered intermediate expressions. Latencies are runtime-specific; compare them only across engines run in the same Colab session. Reference-validation failures must be corrected before trusting solver scores.")\n    dest = ROOT / "results/v2_stress_analysis.md"\n    dest.write_text("\\n".join(out) + "\\n", encoding="utf-8")\n    print("wrote", dest)\n\nif __name__ == "__main__":\n    main()\n')
print("ok: wrote 6 V2 benchmark source files")


ok: wrote 6 V2 benchmark source files


In [7]:
W('README.md', '# Math-solver benchmark (generated by the Colab notebook)\n\nLayout: data/cases.json (141 cases: 72 original core + 69 V2 stress cases, generated by tools/make_cases.py), data/manual_reviews.json (model-assigned derivation overlay, fingerprinted),\ndata/baseline.json (original run, for the change report), bench/ (workers, runner, judge, derivations, engineering, score, report), docs/, pins/, results/.\nStart with results/summary.md, results/failure_analysis.md, results/changes_vs_baseline.md, and results/v2_stress_analysis.md. Direct StepSolver parser/printer probes are in results/stepsolver_v2_diagnostics.json.\nRe-run: open the Version 2 notebook, set engine refs in the configuration cell, and Run all. The original 72 case IDs are retained so results can be compared to the baseline.\n')
print("ok: wrote 1 V2 benchmark source files")


ok: wrote 1 V2 benchmark source files


In [8]:
#@title 4. Clone, install (isolated environments) and record evidence { display-mode: "form" }
# ---- clone candidates, build isolated environments, record evidence ---------------------------------------------
# V2 records exact source commits and retains the original benchmark baseline for core-case comparisons.
import tomllib
(ROOT / "results/evidence").mkdir(parents=True, exist_ok=True)
(ROOT / "pins").mkdir(exist_ok=True)

def clone(name, url, ref):
    d = ROOT / "candidates" / name
    if not (d / ".git").exists():
        sh(["git", "clone", "--quiet", url, str(d)])
    sh(["git", "-C", str(d), "fetch", "--quiet", "--tags", "origin"], check=False)
    if ref and ref != "HEAD":
        sh(["git", "-C", str(d), "checkout", "--quiet", ref])
    else:
        sh(["git", "-C", str(d), "checkout", "--quiet", "-"], check=False)
        sh(["git", "-C", str(d), "pull", "--quiet", "--ff-only"], check=False)
    return d

def mkenv(name, packages, pin_note=None):
    env = ROOT / "envs" / name
    if env.exists():
        shutil.rmtree(env)
    sh([UV, "venv", "--quiet", "--python", PYTHON_VERSION, str(env)])
    py = str(env / "bin" / "python")
    t = time.time()
    r = sh([UV, "pip", "install", "--quiet", "--python", py] + packages, check=False)
    return py, r, time.time() - t

def licence_of(d):
    for f in sorted(d.glob("LICENSE*")) + sorted(d.glob("LICENCE*")) + sorted(d.glob("COPYING*")):
        t = f.read_text(errors="ignore")
        for key, name, cls in [(r"MIT License|Permission is hereby granted, free of charge", "MIT", "permissive"), (r"Apache License", "Apache-2.0", "permissive"),
                               (r"BSD|Redistribution and use in source and binary forms", "BSD", "permissive"), (r"\bISC License", "ISC", "permissive"),
                               (r"GNU AFFERO", "AGPL", "copyleft"), (r"GNU LESSER", "LGPL", "copyleft"), (r"GNU GENERAL PUBLIC", "GPL", "copyleft"), (r"Mozilla Public License", "MPL", "copyleft")]:
            if re.search(key, t):
                return name, cls
    return "unknown (no recognised LICENSE file)", "unknown"

def history_of(d):
    def g(*a):
        return sh(["git", "-C", str(d)] + list(a), check=False).stdout.strip()
    try:
        first = int(g("log", "--reverse", "--format=%ct").splitlines()[0]); last = int(g("log", "-1", "--format=%ct"))
        return {"commits": int(g("rev-list", "--count", "HEAD")), "authors": len(set(g("log", "--format=%an").splitlines())), "tags": len(g("tag").split()),
                "age_days": int((time.time() - first) / 86400), "days_since_last_commit": int((time.time() - last) / 86400)}
    except Exception as e:
        return {"error": str(e)}

def deps_count(py):
    r = sh([UV, "pip", "list", "--python", py, "--format", "json"], check=False)
    try:
        return len(json.loads(r.stdout)) - 1
    except Exception:
        return None

def static_tests(d):
    n = 0
    for tdir in list(d.rglob("tests")) + list(d.rglob("test")):
        if tdir.is_dir() and ".git" not in tdir.parts:
            for f in tdir.rglob("*.py"):
                n += len(re.findall(r"^\s*(?:async\s+)?def test_", f.read_text(errors="ignore"), re.M))
    return n

def pyproject_info(d):
    p = d / "pyproject.toml"
    if not p.exists():
        return {}
    try:
        return tomllib.loads(p.read_text())
    except Exception:
        return {}

evidence = {}
SY = [SYMPY_SPEC, "mpmath"]

# judge env (fixed SymPy, used by the judge and the reference validation only)
jpy, r, _ = mkenv("judge", [JUDGE_SYMPY_SPEC, "mpmath"])
assert r.returncode == 0, "judge env failed: " + r.stderr[-500:]

# standalone SymPy
py, r, secs = mkenv("sympy", SY)
if r.returncode == 0:
    md_ = sh([py, "-c", "import importlib.metadata as m; x=m.metadata('sympy'); print((x.get('License-Expression') or x.get('License') or '').strip()); print(x.get('Requires-Python') or '')"]).stdout.splitlines()
    lic = md_[0] if md_ else ""
    cls = "permissive" if re.search(r"BSD|MIT|Apache", lic) else "unknown"
    evidence["sympy-direct"] = {"installed": True, "install_seconds": secs, "runtime_deps": deps_count(py), "python_requires": md_[1] if len(md_) > 1 else None,
                                "licence": lic or "BSD-3-Clause (not in metadata)", "licence_class": cls if lic else "permissive", "source": "PyPI: " + SYMPY_SPEC, "commit": None,
                                "tests": {"static_test_functions": None, "own_suite": {"run": False}}}
else:
    evidence["sympy-direct"] = {"installed": False, "install_error": r.stderr[-600:]}

def candidate(key, name, url, ref):
    try:
        d = clone(name, url, ref)
    except Exception as e:
        evidence[key] = {"installed": False, "install_error": "clone/checkout failed: %s" % e, "source": "%s @ %s" % (url, ref)}
        return
    commit = sh(["git", "-C", str(d), "rev-parse", "HEAD"]).stdout.strip()
    pin = [SYMPY_SPEC, "mpmath"] if PIN_SYMPY_IN_CANDIDATES else []
    py, r, secs = mkenv(key, [str(d)] + pin)
    fallback = False
    if r.returncode != 0 and pin:
        py, r, secs = mkenv(key, [str(d)])
        fallback = r.returncode == 0
    lic, cls = licence_of(d)
    pp = pyproject_info(d)
    proj = pp.get("project", {})
    src = d / "src" if (d / "src").exists() else d
    evidence[key] = {"installed": r.returncode == 0, "install_error": None if r.returncode == 0 else r.stderr[-800:], "sympy_pin_fallback": fallback,
                     "install_seconds": secs, "commit": commit, "url": url, "ref": ref, "source": "%s @ %s" % (url, commit[:12]), "licence": lic, "licence_class": cls,
                     "python_requires": proj.get("requires-python"), "runtime_deps": deps_count(py) if r.returncode == 0 else None, "history": history_of(d),
                     "tests": {"static_test_functions": static_tests(d), "own_suite": {"run": False}}, "source_dir": str(src), "checkout_dir": str(d)}

candidate("smartsolver", "smartsolver", SMARTSOLVER_URL, SMARTSOLVER_REF)
candidate("stepsolver", "stepsolver", STEPSOLVER_URL, STEPSOLVER_REF)

for k in ("sympy", "smartsolver", "stepsolver", "judge"):
    p = ROOT / "envs" / k / "bin" / "python"
    if p.exists():
        (ROOT / "pins" / ("requirements-%s.txt" % k)).write_text(sh([UV, "pip", "freeze", "--python", str(p)], check=False).stdout)

# optional: candidates' own test suites (separate throw-away venvs; never mixed into the benchmark envs)
if RUN_OWN_TEST_SUITES:
    for key in ("smartsolver", "stepsolver"):
        ev = evidence[key]
        if not ev.get("installed"):
            continue
        d = pathlib.Path(ev["checkout_dir"])
        pp = pyproject_info(d)
        extra = ["pytest"]
        for item in (pp.get("dependency-groups", {}) or {}).get("dev", []):
            name = re.split(r"[ (<>=!~\[]", str(item).strip(), 1)[0]
            if name.startswith("pytest") or name.startswith("httpx") or name == "hypothesis":
                extra.append(re.sub(r"[ ()]", "", str(item)))
        py, r, _ = mkenv("dev-" + key, [str(d)] + sorted(set(extra)))
        if r.returncode:
            ev["tests"]["own_suite"] = {"run": True, "passed": 0, "failed": "n/a", "note": "dev env failed: " + r.stderr[-300:]}
            continue
        t = sh([py, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", "-o", "addopts="], check=False, timeout=1500, cwd=str(d))
        out = t.stdout + t.stderr
        m_pass = re.search(r"(\d+) passed", out); m_fail = re.search(r"(\d+) (?:failed|error)", out)
        ev["tests"]["own_suite"] = {"run": True, "passed": int(m_pass.group(1)) if m_pass else 0, "failed": int(m_fail.group(1)) if m_fail else 0, "tail": out[-300:]}
        print(key, "own suite:", ev["tests"]["own_suite"]["passed"], "passed /", ev["tests"]["own_suite"]["failed"], "failed")

(ROOT / "results/evidence/candidates.json").write_text(json.dumps(evidence, indent=1))
cpu = ""
try:
    cpu = [l for l in open("/proc/cpuinfo") if l.startswith("model name")][0].split(":", 1)[1].strip()
except Exception:
    pass
env_txt = "date: %s\nos: %s\ncpu: %d core(s), %s\nmem: %s\npython (notebook): %s  uv: %s\nengine python: %s\nexperiment config: SYMPY_SPEC=%s SMARTSOLVER_REF=%s STEPSOLVER_REF=%s JUDGE=%s" % (
    time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), platform.platform(), os.cpu_count(), cpu,
    sh("free -m | awk '/Mem/{print $2\" MB\"}'", check=False).stdout.strip(), platform.python_version(), sh([UV, "--version"]).stdout.strip(), PYTHON_VERSION,
    SYMPY_SPEC, SMARTSOLVER_REF, STEPSOLVER_REF, JUDGE_SYMPY_SPEC)
(ROOT / "results/evidence/environment.txt").write_text(env_txt + "\n")
print(env_txt, "\n")
for k, v in evidence.items():
    print("%-13s installed=%s commit=%s deps=%s licence=%s" % (k, v.get("installed"), (v.get("commit") or "PyPI")[:10], v.get("runtime_deps"), v.get("licence")))
    if not v.get("installed"):
        print("   INSTALL FAILED:", (v.get("install_error") or "")[-300:])

date: 2026-10-25T13:38:05Z
os: Linux-6.6.122+-x86_64-with-glibc2.39
cpu: 2 core(s), Intel(R) Xeon(R) CPU @ 2.20GHz
mem: 12975 MB
python (notebook): 3.13.16  uv: uv 0.13.4 (x86_64-unknown-linux-gnu)
engine python: 3.12
experiment config: SYMPY_SPEC=sympy SMARTSOLVER_REF=HEAD STEPSOLVER_REF=fix/printer-validation-sympy-fallbacks JUDGE=sympy==1.14.0 

sympy-direct  installed=True commit=PyPI deps=1 licence=BSD
smartsolver   installed=True commit=f0d46a1089 deps=2 licence=MIT
stepsolver    installed=True commit=25b6f49c5d deps=21 licence=MIT


In [9]:
#@title 5. Dataset validation (141 cases: 72 core + 69 stress) { display-mode: "form" }
# Build the fixed case data, then validate all original and V2 stress references BEFORE running a candidate.
JPY = str(ROOT / "envs/judge/bin/python")
run_stream([sys.executable, "tools/make_cases.py"])
run_stream([JPY, "bench/reference_validation.py"], check=False)
ref = json.load(open(ROOT / "results/reference_validation.json"))
print("Core references validated: %d/%d  all_ok=%s  failed=%s" % (ref["validated"] - len(ref["failed"]), ref["validated"], ref["all_ok"], ref["failed"]))
if not ref.get("all_ok"):
    raise RuntimeError("Core reference validation failed; do not trust the benchmark scores.")
run_stream([JPY, "bench/v2_reference_validation.py"], check=True)
vref = json.load(open(ROOT / "results/v2_reference_validation.json"))
if not vref.get("all_ok") or vref.get("total") != 69:
    raise RuntimeError("V2 stress reference validation failed; do not run candidate engines.")
print("Ready to benchmark: %d total cases (%d core, %d V2 stress)." % (json.load(open(ROOT / "data/cases.json"))["count"], 72, 69))

wrote /content/math_solver_benchmark_v2/data/cases.json cases: 141
validated 72 cases | all_ok = True | failed: [] | missing: []
Core references validated: 72/72  all_ok=True  failed=[]
V2 stress references: 69/69 validated; all_ok=True
Ready to benchmark: 141 total cases (72 core, 69 V2 stress).


In [10]:
#@title 6. Run the full 141-case benchmark (run 1) { display-mode: "form" }
# ---- run 1 (functional pass, cold-start trials, warm repeats); raw replies are preserved verbatim -------------------
WORKER_ENV = {"sympy-direct": "sympy", "smartsolver": "smartsolver", "stepsolver": "stepsolver"}
installed = [s for s, e in WORKER_ENV.items() if json.load(open(ROOT / "results/evidence/candidates.json")).get(s, {}).get("installed")]
print("engines to run:", installed)
if (ROOT / "results/raw").exists():
    shutil.rmtree(ROOT / "results/raw")
run_stream([JPY, "bench/runner.py", "--solver", *installed, "--repeats", str(REPEATS), "--timeout", str(TIMEOUT_S), "--out", "results/raw"])

engines to run: ['sympy-direct', 'smartsolver', 'stepsolver']
cases loaded: 141 (dataset total 141)
sympy-direct   36.3s  outcomes={'answer': 141}  maxrss=63MB timeouts=0 crashes=0
smartsolver    69.5s  outcomes={'unsupported': 56, 'answer': 82, 'error': 3}  maxrss=70MB timeouts=0 crashes=0
stepsolver     94.8s  outcomes={'answer': 129, 'unsupported': 12}  maxrss=80MB timeouts=0 crashes=0


0

In [11]:
#@title 7. Evaluate all core + stress cases, engineering probes, scoring { display-mode: "form" }
# ---- judge + derivation rubric -> raw_results; engineering probes; weighted scores and hard gates ------------------
run_stream([JPY, "bench/evaluate.py"])
run_stream([JPY, "bench/engineering.py"])
run_stream([JPY, "bench/score.py"])

manual reviews NOT applied (engine output changed since review): 5
rows: 423 -> /content/math_solver_benchmark_v2/results/raw_results.json
sympy-direct  {'valid': 137, 'invalid_answer': 4} strict_correct=128 covered=139
smartsolver   {'unsupported': 56, 'valid': 74, 'runtime_error': 3, 'invalid_answer': 8} strict_correct=73 covered=79
stepsolver    {'valid': 128, 'unsupported': 12, 'invalid_answer': 1} strict_correct=127 covered=129
sympy-direct {'malformed_syntax': 0.5, 'unbalanced_parens': 0.5, 'empty_input': 0.5, 'zero_denominator': 0.5, 'missing_variable': 0.5, 'two_variable_parametric': 1.0, 'unsupported_operation': None, 'heavy_equation': 0.0, 'large_polynomial_equation': 1.0, 'untrusted_input_canary': 0.0}
smartsolver {'malformed_syntax': 0.5, 'unbalanced_parens': 0.5, 'empty_input': 0.5, 'zero_denominator': 0.5, 'missing_variable': 0.5, 'two_variable_parametric': 1.0, 'unsupported_operation': None, 'heavy_equation': 0.0, 'large_polynomial_equation': 1.0, 'untrusted_input_canary

0

In [12]:
#@title 8. Reproducibility run (optional second full 141-case pass) { display-mode: "form" }
# ---- second full run -> reproducibility check ------------------------------------------------------------------
rp = ROOT / "results/reproducibility_check.json"
if rp.exists():
    rp.unlink()
if RUN_TWICE:
    if (ROOT / "results/raw_run2").exists():
        shutil.rmtree(ROOT / "results/raw_run2")
    run_stream([JPY, "bench/runner.py", "--solver", *installed, "--repeats", str(REPEATS), "--timeout", str(TIMEOUT_S), "--out", "results/raw_run2"])
    run_stream([JPY, "bench/compare_runs.py", "results/raw", "results/raw_run2", "results/run2_eval"])
else:
    print("RUN_TWICE is off: skipping the reproducibility check")

cases loaded: 141 (dataset total 141)
sympy-direct   35.6s  outcomes={'answer': 141}  maxrss=63MB timeouts=0 crashes=0
smartsolver    68.9s  outcomes={'unsupported': 56, 'answer': 82, 'error': 3}  maxrss=70MB timeouts=0 crashes=0
stepsolver     94.7s  outcomes={'answer': 129, 'unsupported': 12}  maxrss=80MB timeouts=0 crashes=0
{
 "rows_compared": 423,
 "raw_output_differences": 0,
 "verdict_differences": 0,
 "derivation_score_differences": 0,
 "details": [],
 "timing_median_ms_run1_vs_run2": {
  "sympy-direct": [
   5.46,
   5.37
  ],
  "smartsolver": [
   29.56,
   29.28
  ],
  "stepsolver": [
   19.5,
   19.33
  ]
 }
}


In [13]:
#@title 9. Generate reports, direct StepSolver diagnostics, and stress analysis { display-mode: "form" }
# ---- reports ---------------------------------------------------------------------------------------------------
if not COMPARE_WITH_BASELINE and (ROOT / "data/baseline.json").exists():
    (ROOT / "data/baseline.json").rename(ROOT / "data/baseline.json.off")
run_stream([JPY, "bench/report.py"])
if (ROOT / "data/baseline.json.off").exists():
    (ROOT / "data/baseline.json.off").rename(ROOT / "data/baseline.json")

# These direct probes are outside the scored math cases: they inspect parser/printer behavior and safe rejection.
diag_path = ROOT / "results/stepsolver_v2_diagnostics.json"
if RUN_STEPSOLVER_DIAGNOSTICS:
    ev_path = ROOT / "results/evidence/candidates.json"
    ev = json.loads(ev_path.read_text()) if ev_path.exists() else {}
    if ev.get("stepsolver", {}).get("installed"):
        step_py = ROOT / "envs/stepsolver/bin/python"
        if step_py.exists():
            run_stream([str(step_py), "bench/v2_stepsolver_diagnostics.py", "--out", "results/stepsolver_v2_diagnostics.json"], check=False)
        else:
            diag_path.parent.mkdir(parents=True, exist_ok=True)
            diag_path.write_text(json.dumps({"import_error": {"message": "StepSolver marked installed but its environment Python was not found; direct probes were skipped."}}, indent=2) + "\n")
    else:
        diag_path.parent.mkdir(parents=True, exist_ok=True)
        diag_path.write_text(json.dumps({"import_error": {"message": "StepSolver environment is not installed; direct probes were skipped."}}, indent=2) + "\n")
else:
    diag_path.parent.mkdir(parents=True, exist_ok=True)
    diag_path.write_text(json.dumps({"skipped": True, "message": "Direct StepSolver diagnostics disabled in notebook configuration."}, indent=2) + "\n")

run_stream([JPY, "bench/v2_stress_analysis.py"], check=True)
print("\nwrote:", ", ".join(sorted(p.name for p in (ROOT / "results").glob("*.md"))))

wrote summary.md, failure_analysis.md, changes_vs_baseline.md
StepSolver V2 diagnostics: {"parser_fail": 0, "parser_pass": 4, "printer_fail": 0, "printer_inconclusive": 0, "printer_pass": 5, "status_note": "An inconclusive printer probe is not a pass; inspect its captured step strings.", "undefined_status": "pass"}
Detailed results: /content/math_solver_benchmark_v2/results/stepsolver_v2_diagnostics.json
wrote /content/math_solver_benchmark_v2/results/v2_stress_analysis.md

wrote: changes_vs_baseline.md, failure_analysis.md, summary.md, v2_stress_analysis.md


In [14]:
#@title 10. Show headline and V2 stress-test results { display-mode: "form" }
# ---- show the results ------------------------------------------------------------------------------------------
try:
    from IPython.display import Markdown, display
    show = lambda t: display(Markdown(t))
except Exception:
    show = print
sc = json.load(open(ROOT / "results/scores.json"))
NAMES = {"sympy-direct": "SymPy (direct)", "smartsolver": "SmartSolver", "stepsolver": "StepSolver"}
rows_ = []
for s in sc["solver_order"]:
    o = sc["solvers"][s]; gt = sc["hard_gates"][s]
    rows_.append("| %s | %s | **%.1f** | %d/%d | %d | %.1f | %s |" % (NAMES[s], o["version"], o["overall"], o["passed(strict_correct)"], o["defined"], o["covered"], o["derivation"]["mean_final"], gt["eligible_for_production_recommendation"]))
show("## Headline\n\n| Engine | Version | Overall /100 | Strictly correct | Covered | Mean derivation /10 | Eligible as-is |\n|---|---|---|---|---|---|---|\n" + "\n".join(rows_))
for name in ("v2_stress_analysis.md", "changes_vs_baseline.md", "summary.md"):
    p = ROOT / "results" / name
    if p.exists():
        show(p.read_text())

## Headline

| Engine | Version | Overall /100 | Strictly correct | Covered | Mean derivation /10 | Eligible as-is |
|---|---|---|---|---|---|---|
| SymPy (direct) | 1.14.0 | **75.2** | 128/141 | 139 | 4.4 | CONDITIONAL |
| SmartSolver | 0.4.1 | **60.3** | 73/141 | 79 | 6.6 | NO (as-is) |
| StepSolver | 0.1.0 | **88.5** | 127/141 | 129 | 8.9 | NO (as-is) |

# V2 generalisation stress-test analysis

The benchmark contains **72 core cases + 69 V2 stress cases = 141 total cases**. The core 72 retain the original IDs and expected answers; V2 cases are new adversarial variants designed to test whether the earlier fixes generalise. Unsupported cases count as coverage/correctness misses, not as evidence that a reference is wrong.

Reference validation: **PASS (69/69 V2 references)**.

## Stress-suite headline by solver

| Solver | Strictly correct | Covered | Invalid answers | Unsupported / declined | Runtime errors / timeouts | Mean derivation / 10 |
|---|---:|---:|---:|---:|---:|---:|
| sympy-direct | 62/69 (89.9%) | 68/69 (98.6%) | 2 | 0 | 0 | 4.43 |
| smartsolver | 38/69 (55.1%) | 42/69 (60.9%) | 4 | 24 | 2 | 6.51 |
| stepsolver | 58/69 (84.1%) | 60/69 (87.0%) | 1 | 9 | 0 | 8.80 |

## Strict-correct results by category

| Category | Cases | sympy-direct | smartsolver | stepsolver |
|---|---:|---:|---:|---:|
| A | 5 | 5/5 | 0/5 | 5/5 |
| B | 5 | 3/5 | 0/5 | 5/5 |
| C | 6 | 6/6 | 4/6 | 4/6 |
| D | 5 | 5/5 | 2/5 | 5/5 |
| E | 5 | 5/5 | 0/5 | 5/5 |
| F | 6 | 6/6 | 6/6 | 6/6 |
| G | 7 | 6/7 | 6/7 | 6/7 |
| H | 5 | 5/5 | 5/5 | 5/5 |
| I | 5 | 5/5 | 3/5 | 5/5 |
| J | 4 | 4/4 | 2/4 | 2/4 |
| K | 8 | 7/8 | 6/8 | 3/8 |
| L | 8 | 5/8 | 4/8 | 7/8 |

## Numerical API dispatch audit

Cases with an explicit initial guess must be sent through StepSolver's native `nsolve(...)` API; K04 intentionally has no guess and remains the generic high-degree polynomial fallback probe. This prevents the adapter from silently scoring `solve(...)` as if the new numeric API had been tested.

| Case | Initial guess | Expected dispatch | Observed input / call | Dispatch check | Outcome |
|---|---|---|---|---|---|
| K03 | 0.7 | native nsolve API | nsolve(cos(x) = x, x, 0.7) | PASS | unsupported |
| K04 | none | generic polynomial fallback | solve(x^5 - x - 1 = 0, x) | PASS | valid |
| K08 | 2 | native nsolve API | nsolve(x^3 - 2*x - 5 = 0, x, 2) | PASS | unsupported |
| K09 | 0.5 | native nsolve API | nsolve(exp(-x) = x, x, 0.5) | PASS | unsupported |
| K10 | 0.5 | native nsolve API | nsolve(sin(x) = 1/2, x, 0.5) | PASS | unsupported |
| K11 | 2.5 | native nsolve API | nsolve(ln(x) = 1, x, 2.5) | PASS | unsupported |

A `PASS` in this table confirms only correct adapter dispatch, not that the numerical result is correct; correctness and derivation scores are reported separately above.

## StepSolver misses in the V2 stress set

| Case | Operation | Status | Covered | Expected | Actual / detail |
|---|---|---|---|---|---|
| C14 | solve | unsupported | False | All real x (identity) | UnsolvedResult: This equation is true for every value in its domain. The current result model cannot yet represent a universal solution set together with possible domain exclusions |
| C15 | solve | invalid_answer | True | No real solution | {-2 * i, 2 * i} |
| G08 | integrate | unsupported | False | ln\|2x + 1\| + C, on intervals excluding x = -1/2 | UnsolvedResult: This antiderivative requires logarithms of absolute values and explicit domain intervals around real poles. The current result model cannot represent those domain c |
| J05 | dsolve | unsupported | False | y = C*exp(-2x) | adapter has no mapping for operation 'dsolve' |
| J06 | dsolve | unsupported | False | y = C1*exp(x) + C2*exp(2x) | adapter has no mapping for operation 'dsolve' |
| K07 | partial_fractions | valid | True | -1/x + 1/x^2 + 1/(x + 1) | 1 / (x + 1) - 1 / x + x ^ (-2) |
| K08 | nsolve | unsupported | False | x approximately 2.0945514815 | UnsolvedResult: Unsupported function: nsolve. StepSolver only evaluates documented functions. |
| K09 | nsolve | unsupported | False | x approximately 0.5671432904 | UnsolvedResult: Unsupported function: nsolve. StepSolver only evaluates documented functions. |
| K10 | nsolve | unsupported | False | x approximately pi/6 = 0.5235987756 | UnsolvedResult: Unsupported function: nsolve. StepSolver only evaluates documented functions. |
| K11 | nsolve | unsupported | False | x approximately e = 2.7182818285 | UnsolvedResult: Unsupported function: nsolve. StepSolver only evaluates documented functions. |
| L08 | solve | unsupported | False | All real x except x=1 | UnsolvedResult: This equation is true for every value in its domain. The current result model cannot yet represent a universal solution set together with possible domain exclusions |

## StepSolver opacity and latency by operation

| Operation | V2 cases | First-call median (ms) | Max (ms) |
|---|---:|---:|---:|
| det | 1 | 1.56 | 1.56 |
| diff | 4 | 28.12 | 79.74 |
| dsolve | 2 | n/a | n/a |
| evaluate | 7 | 0.89 | 6.92 |
| expand | 1 | 27.09 | 27.09 |
| factor | 2 | 27.44 | 30.76 |
| gradient | 2 | 19.34 | 24.81 |
| integrate | 7 | 512.40 | 1366.12 |
| inverse | 1 | 2.68 | 2.68 |
| limit | 5 | 26.08 | 62.57 |
| nsolve | 4 | 0.14 | 0.21 |
| partial_derivative | 2 | 12.54 | 13.98 |
| partial_fractions | 1 | 43.46 | 43.46 |
| rref | 2 | 4.66 | 5.07 |
| simplify | 7 | 15.73 | 133.00 |
| solve | 15 | 101.06 | 238.15 |
| solve_inequality | 3 | 161.81 | 164.68 |
| solve_system | 3 | 15.42 | 15.81 |

Opaque placeholder-step cases: **3 / 69**.

| Case | Operation | Opaque step count |
|---|---|---:|
| H09 | limit | 1 |
| L06 | solve | 1 |
| L12 | solve | 1 |

## Direct StepSolver parser/printer probes

- Printer probes: 5 pass, 0 fail, 0 inconclusive.
- Native `ln` alias: pass.
- Unknown function probes failing safely: 3 / 3.
- Zero-denominator probe: pass.

Detailed direct probe evidence is in `stepsolver_v2_diagnostics.json`. Inconclusive printer probes are not passes and should be manually inspected against the step strings.

## How to interpret

The stress set is intended to expose overfitting, unsupported mathematical families, invalid domain transformations, and regressions. A decline is a coverage miss; an answer can be covered but wrong. The fixed answer judge checks the declared mathematical result, while direct printer probes inspect rendered intermediate expressions. Latencies are runtime-specific; compare them only across engines run in the same Colab session. Reference-validation failures must be corrected before trusting solver scores.


# Changes vs the baseline run

Baseline: **original benchmark run: SymPy 1.14.0, SmartSolver 0.4.1 @ f0d46a1, StepSolver 0.1.0 @ bd8ea4f (2026-10-23, 1-core sandbox)**.

| Engine | Baseline version | This run | Baseline overall | This overall | Delta |
|---|---|---|---|---|---|
| SymPy (direct) | 1.14.0  | 1.14.0  | 75.5 | 75.2 | -0.3 |
| SmartSolver | 0.4.1 f0d46a10 | 0.4.1 f0d46a10 | 59.3 | 60.3 | 0.9 |
| StepSolver | 0.1.0 bd8ea4ff | 0.1.0 25b6f49c | 87.4 | 88.5 | 1.2 |

| Component | SymPy (direct) (base -> now) | SmartSolver (base -> now) | StepSolver (base -> now) |
|---|---|---|---|
| correctness | 91.7 -> 90.8 | 48.6 -> 51.8 | 93.1 -> 90.1 |
| derivations | 43.3 -> 43.8 | 67.9 -> 66.5 | 80.3 -> 89.4 |
| coverage | 98.6 -> 98.6 | 51.4 -> 56.0 | 95.8 -> 91.5 |
| reliability | 64.3 -> 64.5 | 57.1 -> 57.0 | 90.2 -> 93.0 |
| integration | 85.7 -> 85.7 | 85.7 -> 85.7 | 78.6 -> 85.7 |
| performance | 92.3 -> 88.0 | 82.1 -> 72.3 | 64.8 -> 55.9 |

## Regressions (strictly correct before, not now): 0

None.

## Improvements (not strictly correct before, correct now): 2

| Engine | Case | Before | Now |
|---|---|---|---|
| StepSolver | C07 | invalid_answer | valid |
| StepSolver | K02 | invalid_answer | valid |

## Other status changes: 0

None.

*Derivation scores that relied on the earlier manual overlay fall back to automated scores when output changed; compare derivation means with that in mind.*



# Math-solver benchmark: SymPy (direct) vs SmartSolver vs StepSolver

Generated from `results/raw_results.json`, `scores.json`, `engineering_results.json`. Every number and sentence is computed by code from preserved raw solver output; there is no hand-written, run-specific narrative.

> **Read this first.** (1) Mathematical correctness is decided by a symbolic/numeric judge (SymPy used only as a checker) and every reference answer is independently validated with mpmath/Fraction arithmetic. (2) **Derivation scores are proxies**: an automated rubric plus a manual overlay written by a language model for a specific earlier set of outputs. This run scored 222 derivations; 34 use that overlay; 5 overlay entries were NOT applied because the engine's output changed since they were written (their automated score is used). The overlay needs human audit and never decides mathematical correctness. (3) One reference value in the task specification (K04) is slightly wrong; see section 6. (4) SymPy is evaluated as a **bare engine** (no step layer by design).

## 1. Environment and versions

```
date: 2026-10-25T13:38:05Z
os: Linux-6.6.122+-x86_64-with-glibc2.39
cpu: 2 core(s), Intel(R) Xeon(R) CPU @ 2.20GHz
mem: 12975 MB
python (notebook): 3.13.16  uv: uv 0.13.4 (x86_64-unknown-linux-gnu)
engine python: 3.12
experiment config: SYMPY_SPEC=sympy SMARTSOLVER_REF=HEAD STEPSOLVER_REF=fix/printer-validation-sympy-fallbacks JUDGE=sympy==1.14.0
```
| Engine | Version | Commit / source | Python | SymPy in its env | Isolated env |
|---|---|---|---|---|---|
| SymPy (direct) | 1.14.0 | PyPI: sympy | 3.12.3 | 1.14.0 | `envs/sympy` |
| SmartSolver | 0.4.1 | f0d46a1089366f8e14f73de9ba811d8e4c1471b0 | 3.12.3 | 1.14.0 | `envs/smartsolver` |
| StepSolver | 0.1.0 | 25b6f49c5db54873c3f94fdd4075bf2b86c76fb6 | 3.12.3 | 1.14.0 | `envs/stepsolver` |

Each engine ran in its own virtualenv and worker process from an empty working directory. The judge runs in a separate pinned environment. Timings depend on the hardware above and are only comparable between runs on the same hardware.

## 2. Installation and execution status

| Engine | Installed | Executed | Own test-suite | Third-party deps | Licence | requires-python |
|---|---|---|---|---|---|---|
| SymPy (direct) | yes | 141/141 attempted | not run (static test count: n/a: bare engine) | 1 | BSD (permissive) | >=3.9 |
| SmartSolver | yes | 141/141 attempted | not run (static test count: 28) | 2 | MIT (permissive) | >=3.10 |
| StepSolver | yes | 141/141 attempted | not run (static test count: 209) | 21 | MIT (permissive) | >=3.12,<3.15 |

Skipped cases: SymPy (direct) 0, SmartSolver 0, StepSolver 0. A case is only 'unsupported' when the engine has no method for it or declares it unsolved; nothing is substituted.

## 3. Results by category (strictly correct / cases)

| Category | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| A Arithmetic and expression evaluation | 13/13 | 0/13 | 13/13 |
| B Simplification, expansion and factorisation | 8/13 | 0/13 | 13/13 |
| C Linear and quadratic equations | 16/16 | 13/16 | 13/16 |
| D Simultaneous equations and inequalities | 11/11 | 5/11 | 11/11 |
| E Powers, roots, logarithms and trigonometry | 12/13 | 0/13 | 13/13 |
| F Differentiation | 12/12 | 12/12 | 12/12 |
| G Integration | 11/13 | 11/13 | 11/13 |
| H Limits and continuity | 9/9 | 9/9 | 9/9 |
| I Matrices and linear algebra | 9/9 | 5/9 | 9/9 |
| J Differential equations and advanced calculus | 8/8 | 4/8 | 6/8 |
| K Partial fractions, complex numbers and numerical solving | 11/12 | 8/12 | 6/12 |
| L Domain restrictions and failure handling | 8/12 | 6/12 | 11/12 |
| **Total** | **128/141** | **73/141** | **127/141** |

*Strictly correct* = correct value AND required form AND (where required) the domain restriction reported. Value-only counts: SymPy (direct) 137, SmartSolver 74, StepSolver 128.

## 4. Overall correctness and coverage

| Metric | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Cases defined / executed / skipped | 141 / 141 / 0 | 141 / 141 / 0 | 141 / 141 / 0 |
| Strictly correct (passed) | 128 | 73 | 127 |
| Invalid answers (produced, wrong) | 4 | 8 | 1 |
| Unsupported: no API for the operation | 0 | 56 | 2 |
| Unsupported: engine declared it unsolved | 0 | 0 | 10 |
| Runtime errors / timeouts / infra errors / input-incompatible | 0 / 0 / 0 / 0 | 3 / 0 / 0 / 0 | 0 / 0 / 0 / 0 |
| Coverage (result in the expected form) | 139 (98.6%) | 79 (56.0%) | 129 (91.5%) |
| Correctness (strict) % | 90.8 | 51.8 | 90.1 |

Coverage counts a result only if it is in the requested form; correctness is reported separately.

## 5. Derivation availability and quality

| Metric | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Derivation available (cases / 141) | 13 (9%) | 82 (58%) | 127 (90%) |
| Mean automated score /10 | 4.38 | 6.66 | 8.93 |
| Mean final score /10 (after any applied manual overlay) | 4.38 | 6.65 | 8.94 |
| Mean final, domain-sensitive cases only | 2.00 | 4.60 | 8.12 |
| Derivations with an applied manual review | 2 | 16 | 16 |
| Mean correctness | 1.69 | 2.51 | 2.94 |
| Mean completeness | 0.85 | 1.15 | 1.50 |
| Mean explanation | 1.00 | 0.82 | 1.72 |
| Mean consistency | 0.00 | 0.70 | 0.96 |
| Mean domain | 0.85 | 0.73 | 0.84 |
| Mean presentation | 0.00 | 0.74 | 0.97 |

Max per criterion: correctness 3, completeness 2, explanation 2, consistency 1, domain 1, presentation 1. Opaque-only derivations (no transformation shown): SymPy (direct) 0; SmartSolver 19 (C09, H01, H02, H03, H04, K03, L01, L02, L04, C16, H05, H06, H07, H08, H09, L06, L07, L09, L12); StepSolver 4 (L02, H09, L06, L12).

Score distribution (final /10 -> number of derivations): SymPy (direct) {'2': 2, '4': 2, '5': 9}; SmartSolver {'0': 1, '1': 2, '10': 2, '2': 8, '3': 13, '4': 1, '7': 5, '8': 21, '9': 29}; StepSolver {'10': 53, '2': 3, '3': 1, '4': 1, '6': 3, '7': 4, '8': 12, '9': 50}.

## 6. Edge-case, domain and engineering behaviour

Engineering probes (1 = clean, 0.5 = acceptable but raw/implicit, 0 = failure, n/a = does not apply):

| Probe | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| malformed_syntax | 0.5 | 0.5 | 1.0 |
| unbalanced_parens | 0.5 | 0.5 | 1.0 |
| empty_input | 0.5 | 0.5 | 1.0 |
| zero_denominator | 0.5 | 0.5 | 1.0 |
| missing_variable | 0.5 | 0.5 | 0.5 |
| two_variable_parametric | 1.0 | 1.0 | 1.0 |
| unsupported_operation | n/a | n/a | 0.5 |
| heavy_equation | 0.0 | 0.0 | 0.0 |
| large_polynomial_equation | 1.0 | 1.0 | 1.0 |
| untrusted_input_canary | 0.0 | 0.0 | 1.0 |

Probe outcomes that were not clean (from the raw probe notes):

- **SymPy (direct) / malformed_syntax** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SymPy (direct) / unbalanced_parens** (0.5): raw Python exception TokenError (caught by adapter)
- **SymPy (direct) / empty_input** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SymPy (direct) / zero_denominator** (0.5): returns 'zoo' (complex infinity) without an undefined/domain message
- **SymPy (direct) / missing_variable** (0.5): answer without fabricating a root: "EmptySet"
- **SymPy (direct) / heavy_equation** (0.0): worker timeout after 15s - external kill needed
- **SymPy (direct) / untrusted_input_canary** (0.0): INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): "24"
- **SmartSolver / malformed_syntax** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SmartSolver / unbalanced_parens** (0.5): raw Python exception TokenError (caught by adapter)
- **SmartSolver / empty_input** (0.5): raw Python exception SyntaxError (caught by adapter)
- **SmartSolver / zero_denominator** (0.5): silently reports no solution
- **SmartSolver / missing_variable** (0.5): answer without fabricating a root: {"solutions": "[]", "solution_latex": "No solution found", "steps_verified": "True", "steps_mode": "
- **SmartSolver / heavy_equation** (0.0): worker timeout after 15s - external kill needed
- **SmartSolver / untrusted_input_canary** (0.0): INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): {"solutions": "[24]", "solution_latex": "24", "steps_verified": "True", "steps_m
- **StepSolver / missing_variable** (0.5): answer without fabricating a root: "Input: solve(x + 1 = 0, z)\nStep 1 (Conclude there are no solutions): A nonzero constant cannot equ
- **StepSolver / unsupported_operation** (0.5): honest decline (UnsolvedResult)
- **StepSolver / heavy_equation** (0.0): worker timeout after 15s - external kill needed

`untrusted_input_canary` sends the benign payload `__import__('math').factorial(4)`; a score of 0 means the engine's own string interface evaluated it as Python. `heavy_equation` is `x^25 - x - 1 = 0` with a 15 s budget; a score of 0 means the engine did not finish and needed an external kill.

Domain-sensitive cases (47): strictly correct SymPy (direct) 34, SmartSolver 18, StepSolver 40.

**Specification discrepancy (K04):** the task prints "approximately 1.167304015", but the real root of x^5 - x - 1 is 1.1673039782614187 (mpmath 40 digits, bisection, polyroots agree); the printed value is off by 3.7e-8. The verbatim spec text is kept in `spec_expected`; the verified root is the reference (tolerance 1e-8). Per-engine: SymPy (direct): matches spec-printed value 1.167304015 within 1e-6: True; SmartSolver: matches spec-printed value 1.167304015 within 1e-6: True; StepSolver: matches spec-printed value 1.167304015 within 1e-6: True.

Reference validation: 72/72 references independently validated by a second method, all_ok=True.

## 7. Reliability and performance

| Reliability metric (0-1) | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| R1_repeatability | 1.000 | 1.000 | 1.000 |
| R2_engineering_probes | 0.500 | 0.500 | 0.800 |
| R3_stability | 1.000 | 0.965 | 1.000 |
| R4_domain_edge_cases | 0.723 | 0.383 | 0.851 |
| R5_untrusted_input_safety | 0.000 | 0.000 | 1.000 |
| **Reliability score (mean x100)** | **64.5** | **57.0** | **93.0** |

Reliability = 100 x mean(R1 repeatability, R2 engineering probes, R3 = 1 - crashes/attempted, R4 strict-correct rate on domain-sensitive cases, R5 untrusted-input safety as shipped); see `docs/SCORING_RULES.md`.

| Performance (ms) | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Cold start: process spawn -> ready (median) | 2915 | 2822 | 3354 |
| Cold first solve of C01 (median) | 192 | 171 | 393 |
| Warm median, 75 common cases (cache-cleared repeats) | 8.15 | 27.18 | 56.55 |
| Warm p95, common cases | 49.13 | 144.97 | 607.66 |
| Warm median, own supported cases | 5.46 | 29.56 | 19.50 |
| Warm p95, own supported cases | 51.89 | 203.62 | 456.21 |
| Timeouts / crashes (main pass) | 0 / 0 | 0 / 0 | 0 / 0 |
| Peak RSS of worker (MB) | 63 | 70 | 80 |
| **Performance score** | **88.0** | **72.3** | **55.9** |

Latency score: 100 at <=10 ms, 0 at >=2000 ms, log-linear between; Performance = 0.4 x score(median) + 0.4 x score(p95) + 0.2 x 100 x (1 - failure rate), on the cases ALL engines answered. Cold start is reported, not scored.

Reproducibility (run 2 vs run 1): **423/423** rows identical in raw output, **0** verdict differences, **0** derivation-score differences. Median warm latency run1 -> run2: SymPy (direct) 5.46 -> 5.37 ms; SmartSolver 29.56 -> 29.28 ms; StepSolver 19.5 -> 19.33 ms.

## 8. Weighted scores and hard gates

| Component (weight) | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| Correctness (30%) | 90.8 | 51.8 | 90.1 |
| Derivation quality (30%) | 43.8 | 66.5 | 89.4 |
| Coverage (20%) | 98.6 | 56.0 | 91.5 |
| Reliability (10%) | 64.5 | 57.0 | 93.0 |
| Integration/maintainability (5%) | 85.7 | 85.7 | 85.7 |
| Performance (5%) | 88.0 | 72.3 | 55.9 |
| **Overall /100** | **75.2** | **60.3** | **88.5** |

`Overall = 0.30*Correctness + 0.30*Derivations + 0.20*Coverage + 0.10*Reliability + 0.05*Integration + 0.05*Performance`. The derivation component is the mean score over *available* derivations x 10, as specified; the availability-penalised variant is in the sensitivity table.

**Integration evidence (computed):**

- **SymPy (direct)** (12/14): env_compat=2 (1 third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python >=3.9); install=2 (single install command succeeded); adaptation=2 (0.0% of cases needed per-case adapter handling (2 if <=2%, 1 if <=10%, else 0)); testability_extensibility=0 (FIXED ASSUMPTION: bare engine has no derivation/step framework to test or extend); licence=2 (licence BSD (permissive)); maintenance=2 (FIXED ASSUMPTION: mature, widely maintained project (not measurable from a PyPI install)); ui_independent=1 (FIXED ASSUMPTION: used headless as a library in this benchmark); no_rewrite=1 (FIXED ASSUMPTION: plain library call; adapter-sized integration)
- **SmartSolver** (12/14): env_compat=2 (2 third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python >=3.10); install=2 (single install command succeeded); adaptation=2 (0.0% of cases needed per-case adapter handling (2 if <=2%, 1 if <=10%, else 0)); testability_extensibility=1 (28 test functions found statically (2 if >=100, 1 if >=1)); licence=2 (licence MIT (permissive)); maintenance=1 (history: 10 commits, 2 authors, age 103 days, last commit 63 days ago, 1 tags (2 if >=2y and >=5 authors; 1 if); ui_independent=1 (FIXED ASSUMPTION: used headless as a library in this benchmark); no_rewrite=1 (FIXED ASSUMPTION: plain library call; adapter-sized integration)
- **StepSolver** (12/14): env_compat=1 (21 third-party packages installed besides the engine (2 if <=4, 1 if <=30, else 0); requires-python >=3.12,<3.); install=2 (single install command succeeded); adaptation=2 (1.4% of cases needed per-case adapter handling (2 if <=2%, 1 if <=10%, else 0)); testability_extensibility=2 (209 test functions found statically (2 if >=100, 1 if >=1)); licence=2 (licence MIT (permissive)); maintenance=1 (history: 53 commits, 2 authors, age 70 days, last commit 1 days ago, 0 tags (2 if >=2y and >=5 authors; 1 if >); ui_independent=1 (FIXED ASSUMPTION: used headless as a library in this benchmark); no_rewrite=1 (FIXED ASSUMPTION: plain library call; adapter-sized integration)

**Sensitivity (ranking stability):**

| Variant | SymPy (direct) | SmartSolver | StepSolver | Order |
|---|---|---|---|---|
| primary (spec formula) | 75.2 | 60.3 | 88.5 | StepSolver > SymPy (direct) > SmartSolver |
| value-only correctness (no domain-reporting requirement) | 77.2 | 60.5 | 88.7 | StepSolver > SymPy (direct) > SmartSolver |
| value correctness ignoring form rules too | 77.2 | 60.7 | 88.7 | StepSolver > SymPy (direct) > SmartSolver |
| availability-penalised derivations (unavailable=0 over all benchmark cases) | 63.3 | 51.9 | 85.9 | StepSolver > SymPy (direct) > SmartSolver |
| automated-only derivation scores (no manual overlay) | 75.2 | 60.3 | 88.5 | StepSolver > SymPy (direct) > SmartSolver |
| latency on each solver's own supported cases | 75.2 | 60.1 | 89.0 | StepSolver > SymPy (direct) > SmartSolver |

**Hard gates** (a weighted score never overrides these):

| Gate | SymPy (direct) | SmartSolver | StepSolver |
|---|---|---|---|
| G1 critical correctness defect (confidently wrong answer) | PASS | FAIL K03, C15 | FAIL C15 |
| G2 invalid / self-contradicting derivation (FAIL >=2 cases, CONDITIONAL 1) | PASS | FAIL G04, K01 | PASS |
| G3 untrusted input executed as code (as shipped) | CONDITIONAL | FAIL | PASS |
| G4 licence | PASS (BSD) | PASS (MIT) | PASS (MIT) |
| **Eligible for production recommendation** | **CONDITIONAL** | **NO (as-is)** | **NO (as-is)** |

Gate evidence:

- **SymPy (direct) G3 = CONDITIONAL**: INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): "24"
- **SmartSolver G1 = FAIL**: K03: claims no solution but a real root exists; C15: returned solutions where empty expected
- **SmartSolver G2 = FAIL**: G04: steps_mode=direct_evaluation: worked steps replaced by direct solve; K01: manual: final step shows the unchanged input
- **SmartSolver G3 = FAIL**: INPUT EVALUATED AS PYTHON (math.factorial(4)=24 reached the solver): {"solutions": "[24]", "solution_latex": "24", "steps_verified": "True", "steps_m
- **StepSolver G1 = FAIL**: C15: returned solutions where empty expected

G3 is CONDITIONAL for SymPy-direct by definition: only its *string* parser evaluates code, it offers programmatic construction, and this benchmark's allow-list adapter blocks the payload.

## 9. Strongest and weakest areas (computed)

**SymPy (direct)** (overall 75.2)

- Strongest: all cases correct in categories A, C, D, F, H, I, J; strict 128/141, coverage 139/141, derivations available 13/141 (mean 4.38/10), warm median 8.1 ms.
- Weakest: no category entirely wrong; lowest partial categories: B (62%), L (67%), G (85%).
- Not-passed cases by type: right value, domain/form requirement missed: B06, B07, B08, E02, B10, B12, K07, L05, L11; invalid answer: G04, L04, G08, L10.
- Derivation concerns: 0 opaque-only, 0 with auto-detected invalid steps, 13 with override/untied workings.
- Engineering probes below 1.0: malformed_syntax (0.5), unbalanced_parens (0.5), empty_input (0.5), zero_denominator (0.5), missing_variable (0.5), heavy_equation (0.0), untrusted_input_canary (0.0); gates not PASS: G3=CONDITIONAL.

**SmartSolver** (overall 60.3)

- Strongest: all cases correct in categories F, H; strict 73/141, coverage 79/141, derivations available 82/141 (mean 6.65/10), warm median 27.2 ms.
- Weakest: no correct answers in categories A, B, E; lowest partial categories: D (45%), J (50%), L (50%).
- Not-passed cases by type: unsupported (no API): A01, A02, A03, A04, A05, A06, A07, A08, B01, B02, B03, B04, B05, B06, B07, B08, D04, D05, D06, E01, E02, E03, E04, E05, E06, E07, E08, I02, I03, J01, J02, L03, A09, A10, A11, A12, A13, B09, B10, B11, B12, B13, D09, D10, D11, E09, E10, E11, E12, E13, I05, I06, J05, J06, L05, L11; runtime error: C10, C14, L08; invalid answer: G04, K01, K03, L04, C15, G08, K09, L10; right value, domain/form requirement missed: K07.
- Derivation concerns: 19 opaque-only, 0 with auto-detected invalid steps, 1 with override/untied workings.
- Engineering probes below 1.0: malformed_syntax (0.5), unbalanced_parens (0.5), empty_input (0.5), zero_denominator (0.5), missing_variable (0.5), heavy_equation (0.0), untrusted_input_canary (0.0); gates not PASS: G1=FAIL, G2=FAIL, G3=FAIL.

**StepSolver** (overall 88.5)

- Strongest: all cases correct in categories A, B, D, E, F, H, I; strict 127/141, coverage 129/141, derivations available 127/141 (mean 8.94/10), warm median 56.6 ms.
- Weakest: no category entirely wrong; lowest partial categories: K (50%), J (75%), C (81%).
- Not-passed cases by type: unsupported (declared unsolved): C10, G04, K03, C14, G08, K08, K09, K10, K11, L08; invalid answer: C15; unsupported (no API): J05, J06; right value, domain/form requirement missed: K07.
- Derivation concerns: 4 opaque-only, 0 with auto-detected invalid steps, 0 with override/untied workings.
- Engineering probes below 1.0: missing_variable (0.5), unsupported_operation (0.5), heavy_equation (0.0); gates not PASS: G1=FAIL.

## 10. Recommendation (rule-based, with limits)

Decision rules (fixed before any run): (1) an engine with any hard-gate FAIL is not recommended as-is; (2) among the rest, rank by weighted score; (3) scores within 5 points are treated as a tie to be broken by gates, derivation availability and the failures listed above; (4) a gate-failing engine can still be piloted behind a validation wrapper if the blocking defects are listed and fixed first.

| Rank | Engine | Overall | Eligible as-is | Blocking gates |
|---|---|---|---|---|
| 1 | StepSolver | 88.5 | NO (as-is) | G1=FAIL |
| 2 | SymPy (direct) | 75.2 | CONDITIONAL | G3=CONDITIONAL |
| 3 | SmartSolver | 60.3 | NO (as-is) | G1=FAIL, G2=FAIL, G3=FAIL |

**Recommended with conditions:** SymPy (direct) (highest-scoring engine with no FAIL gate; overall 75.2). Conditions: G3=CONDITIONAL.

**Highest weighted score:** StepSolver (88.5), 13.3 points above the recommended engine. It is not eligible as-is because: G1 = FAIL (C15).
Piloting it behind a validation wrapper is reasonable only after the listed failures are fixed and this notebook is re-run to confirm the gates pass.

From the measured data, the practical split of responsibilities is: use the engine with the best strict correctness/coverage as the *answer* source, and treat an engine's derivation as shippable only where it is available, scored, and free of invalid/contradictory steps (see failure_analysis.md sections 2 and 3). Where derivations are opaque or unavailable, show the verified answer and label the derivation unavailable instead of showing a placeholder.

### Limitations of this benchmark

- The 141-case suite (72 core cases plus 69 targeted stress cases) is still only a sample of school/early-university mathematics; it does **not** show that any engine supports all of mathematics or handles word problems, images, or free-form phrasing.
- Derivation scores are proxy rubrics plus a model-assigned overlay (for earlier outputs only); human audit is required. Completeness uses regex milestones.
- Latency is hardware-dependent and a small part of the score (5%); warm latency is measured cache-cleared in a warm process.
- K03/K04 (numeric solving) use each engine's nearest documented API (SymPy `nsolve`/`nroots`; engines without a numeric root-finder use their generic equation solver). SmartSolver `limit` uses its documented one-sided default.
- StepSolver `ln` is adapted to `log`; the as-specified result is preserved in `native_flags.as_specified_variant`.
- Adapters map to the public API names that existed when the benchmark was written; a renamed/removed method is reported as unsupported (SmartSolver) or an infrastructure error (import failure), never silently substituted.
- Offline operation was assessed only by static search for network imports in each library core (`engineering_results.json`).

### Artefacts

`results/raw_results.json|csv`, `results/raw/*.json` (verbatim worker replies), `results/scores.json`, `results/engineering_results.json`, `results/reference_validation.json`, `results/reproducibility_check.json`, `results/failure_analysis.md`, `results/changes_vs_baseline.md` (if a baseline is present), `data/cases.json`, `docs/`.

## 11. Changes vs the baseline

See `results/changes_vs_baseline.md`: 0 regressions, 2 improvements, 0 other status changes versus *original benchmark run: SymPy 1.14.0, SmartSolver 0.4.1 @ f0d46a1, StepSolver 0.1.0 @ bd8ea4f (2026-10-23, 1-core sandbox)*.



In [15]:
#@title 11. Download all V2 benchmark artefacts as a zip { display-mode: "form" }
# ---- package everything and download -----------------------------------------------------------------------------
import zipfile
out = pathlib.Path("/content") / "math_solver_benchmark_v2_results.zip" if pathlib.Path("/content").exists() else ROOT.parent / "math_solver_benchmark_v2_results.zip"
skip = {"envs", "candidates", "__pycache__", ".git"}
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ROOT.rglob("*")):
        rel = p.relative_to(ROOT)
        if p.is_dir() or any(part in skip for part in rel.parts) or p.suffix == ".pyc":
            continue
        z.write(p, "math_solver_benchmark_v2/" + str(rel)); n += 1
print("zip:", out, "|", n, "files |", round(out.stat().st_size / 1024), "KB")
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print("(not running in Colab: copy the zip from the path above)")

zip: /content/math_solver_benchmark_v2_results.zip | 52 files | 625 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>